In [ ]:
# ════════════════════════════════════════════════════════
# STEP 0 — Install all packages (run once)
# ════════════════════════════════════════════════════════
import subprocess, sys

packages = [
    "fredapi", "yfinance", "wbdata",
    "pandas", "numpy", "statsmodels",
    "scikit-learn", "xgboost", "shap",
    "imbalanced-learn", "boruta",
    "matplotlib", "torch",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages installed.")


All packages installed.


In [ ]:
# ════════════════════════════════════════════════════════
# STEP 1 — Set your FRED API key
# Get free key at: fred.stlouisfed.org/docs/api_key.html
# ════════════════════════════════════════════════════════
import os
os.environ['FRED_API_KEY'] = 'e94488b5ed081ebe98e29b17c3d5e641'

# Verify it is set
key = os.environ.get('FRED_API_KEY', '')
if key and key != 'paste_your_32char_key_here':
    print(f"FRED key set: {key[:4]}...{key[-4:]} ({len(key)} chars)")
else:
    print("WARNING: Please paste your actual FRED key above.")


FRED key set: e944...e641 (32 chars)


In [ ]:
# ════════════════════════════════════════════════════════
# STEP 2 — Data Collection
# Sources: FRED API + Yahoo Finance
# No GitHub, no Kaggle, all official APIs
# ════════════════════════════════════════════════════════
import os, time, warnings
import numpy as np
import pandas as pd
import yfinance as yf
warnings.filterwarnings('ignore')

for d in ['data/raw', 'data/cleaned', 'figures', 'results']:
    os.makedirs(d, exist_ok=True)

FRED_KEY   = os.environ.get('FRED_API_KEY', '')
START_DATE = '1980-01-01'

def ok(m):   print(f"  OK  {m}")
def warn(m): print(f"  !!  {m}")
def err(m):  print(f"  XX  {m}")
def sep(t):  print(f"\n--- {t} ---")

print("=" * 55)
print("  STEP 2 — DATA COLLECTION")
print("=" * 55)

# ── FRED API ──────────────────────────────────────────
sep("2A. FRED API")
fred_df = pd.DataFrame()

if not FRED_KEY or FRED_KEY == 'paste_your_32char_key_here':
    warn("FRED key not set — run Step 1 first")
else:
    from fredapi import Fred
    fred_obj = Fred(api_key=FRED_KEY)

    fred_series = {
        # Monetary policy and rates
        'T10Y2Y':           'term_spread_10y2y',
        'T10Y3M':           'term_spread_10y3m',
        'FEDFUNDS':         'fed_funds_rate',
        'MORTGAGE30US':     'mortgage_rate_30y',
        # Credit spreads (Astill et al. 2023 covariates, Gap 4)
        'BAMLH0A0HYM2':     'hy_credit_spread',
        'BAMLC0A0CM':       'ig_credit_spread',
        'NFCI':             'nfci_financial_conditions',
        'STLFSI4':          'stl_financial_stress',
        # Credit quantities (Schularick-Taylor mechanism, Gap 5)
        'TOTALSL':          'consumer_credit_total',
        'BUSLOANS':         'commercial_loans',
        'REALLN':           'real_estate_loans',
        'DPCREDIT':         'domestic_credit_private',
        # Bank health proxies (replaces World Bank NPL)
        'DRSFRMACBS':       'delinquency_mortgage',
        'DRCCLACBS':        'delinquency_credit_card',
        'DRBLACBS':         'delinquency_business',
        'GFDEGDQ188S':      'federal_debt_pct_gdp',
        # Real economy
        'GDPC1':            'real_gdp',
        'UNRATE':           'unemployment_us',
        'CPIAUCSL':         'cpi_us',
        'INDPRO':           'industrial_production',
        'HOUST':            'housing_starts',
        # Asset prices
        'VIXCLS':           'vix',
        'CSUSHPISA':        'case_shiller_hpi',
        'DCOILWTICO':       'oil_wti',
        'GOLDAMGBD228NLBM': 'gold_price',
        'SP500':            'sp500_fred',
    }

    fred_raw = {}
    for sid, name in fred_series.items():
        try:
            fred_raw[name] = fred_obj.get_series(
                sid, observation_start=START_DATE)
            time.sleep(0.05)
        except Exception as ex:
            warn(f"FRED {sid}: {str(ex)[:40]}")

    # Shiller valuation series from FRED
    for sid, name in [
        ('CAPE',    'cape_shiller'),
        ('SP500PE', 'pe_ratio'),
        ('SP500DIV','div_yield'),
    ]:
        try:
            fred_raw[name] = (
                fred_obj.get_series(sid, observation_start=START_DATE)
                .resample('MS').last()
            )
            ok(f"{name} ({sid})")
        except Exception:
            warn(f"{sid} not available in FRED — skipping")

    fred_df = pd.DataFrame(fred_raw)
    fred_df.index = pd.to_datetime(fred_df.index)
    fred_df.index.name = 'date'
    fred_df = fred_df.sort_index()
    fred_df.to_csv('data/raw/fred_data.csv')
    ok(f"FRED saved: {fred_df.shape[0]} obs x {fred_df.shape[1]} series")

# ── Yahoo Finance ─────────────────────────────────────
sep("2B. Yahoo Finance")

all_tickers = {
    # 16 equity indices (Gap 3: cross-asset scope)
    'SP500':   '^GSPC',   'FTSE100': '^FTSE',
    'DAX':     '^GDAXI',  'CAC40':   '^FCHI',
    'Nikkei':  '^N225',   'TSX':     '^GSPTSE',
    'ASX200':  '^AXJO',   'KOSPI':   '^KS11',
    'IBEX35':  '^IBEX',   'AEX':     '^AEX',
    'SMI':     '^SSMI',   'BOVESPA': '^BVSP',
    'SENSEX':  '^BSESN',  'HSI':     '^HSI',
    'NIFTY50': '^NSEI',   'STI':     '^STI',
    # Rates and volatility
    'US_10Y':  '^TNX',    'US_3M':   '^IRX',
    'US_30Y':  '^TYX',    'VIX':     '^VIX',
}

yf_data = yf.download(
    list(all_tickers.values()),
    start=START_DATE,
    interval='1mo',
    auto_adjust=True,
    progress=False,
    threads=True,
)['Close']

reverse = {v: k for k, v in all_tickers.items()}
yf_data.columns = [reverse.get(c, c) for c in yf_data.columns]
yf_data.index = pd.to_datetime(yf_data.index)
yf_data.index.name = 'date'
yf_data = yf_data.sort_index()

if 'US_10Y' in yf_data.columns and 'US_3M' in yf_data.columns:
    yf_data['term_spread_yf'] = yf_data['US_10Y'] - yf_data['US_3M']

yf_data.to_csv('data/raw/yfinance_data.csv')
ok(f"Yahoo Finance saved: {yf_data.shape[0]} months x "
   f"{yf_data.shape[1]} series")

print("\nStep 2 complete.")


  STEP 2 — DATA COLLECTION

--- 2A. FRED API ---
  !!  FRED GOLDAMGBD228NLBM: Bad Request.  The series does not exist.
  !!  CAPE not available in FRED — skipping
  !!  SP500PE not available in FRED — skipping
  !!  SP500DIV not available in FRED — skipping
  OK  FRED saved: 12276 obs x 25 series

--- 2B. Yahoo Finance ---
  OK  Yahoo Finance saved: 497 months x 21 series

Step 2 complete.


In [ ]:
# ════════════════════════════════════════════════════════
# STEP 3 — GSADF Bubble Detection
# Phillips, Shi & Yu (2015) | Harvey et al. (2016)
# Addresses Gaps 1, 2, 3
# NOTE: Set N_BOOT = 499 for final paper submission
# ════════════════════════════════════════════════════════
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
warnings.filterwarnings('ignore')

os.makedirs('data/cleaned', exist_ok=True)
os.makedirs('figures',      exist_ok=True)
os.makedirs('results',      exist_ok=True)

R0_FRAC   = 0.10   # PSY minimum window fraction
N_BOOT    = 99     # !! Change to 499 for final paper !!
SIG_LEVEL = 0.05
MIN_DUR   = 3      # minimum episode duration (months)

def ok(m):   print(f"  OK  {m}")
def warn(m): print(f"  !!  {m}")
def sep(t):  print(f"\n--- {t} ---")

print("=" * 55)
print("  STEP 3 — GSADF BUBBLE DETECTION")
print("=" * 55)


# ── Core GSADF functions ──────────────────────────────
def _adf_tstat(y):
    # Right-tailed ADF: H0 unit root, H1 explosive root
    # Model: Dy_t = a + b*y_{t-1} + phi*Dy_{t-1} + e
    T = len(y)
    if T < 7: return np.nan
    dy = np.diff(y)
    n  = len(dy) - 1
    if n < 4: return np.nan
    X = np.column_stack([np.ones(n), y[1:-1], dy[:-1]])
    Y = dy[1:]
    try:
        c = np.linalg.lstsq(X, Y, rcond=None)[0]
        r = Y - X @ c
        s2 = np.dot(r, r) / (n - 3)
        se = np.sqrt(s2 * np.linalg.inv(X.T @ X)[1, 1])
        return float(c[1] / se)
    except Exception:
        return np.nan


def bsadf_sequence(y, r0_frac=R0_FRAC):
    # BSADF(r2) = sup over r1 in [0, r2-r0] of ADF(r1, r2)
    # This is the PSY (2015) double-recursion.
    # NOT rolling ADF — takes supremum over all start points.
    T  = len(y)
    r0 = max(int(np.floor(r0_frac * T)), 12)
    seq = np.full(T, np.nan)
    for r2 in range(r0, T):
        sup = -np.inf
        for r1 in range(0, r2 - r0 + 1):
            t = _adf_tstat(y[r1:r2 + 1])
            if not np.isnan(t) and t > sup:
                sup = t
        seq[r2] = sup
    return seq


def wild_bootstrap_cv(y, r0_frac=R0_FRAC, n_boot=N_BOOT,
                      sig=SIG_LEVEL, seed=42):
    # Harvey, Leybourne, Sollis & Taylor (2016) wild bootstrap.
    # Rademacher signs correct for GARCH and leverage effects.
    rng     = np.random.default_rng(seed)
    T       = len(y)
    dy_null = np.diff(y) - np.diff(y).mean()
    bg = np.zeros(n_boot)
    bs = np.full((n_boot, T), -np.inf)
    for b in range(n_boot):
        eta = rng.choice([-1., 1.], size=len(dy_null))
        yb  = np.concatenate([[0.], np.cumsum(eta * dy_null)])
        seq = bsadf_sequence(yb, r0_frac)
        bs[b]  = np.where(np.isnan(seq), -np.inf, seq)
        bg[b]  = np.nanmax(seq)
    cv_g = float(np.quantile(bg, 1 - sig))
    cv_s = np.quantile(bs, 1 - sig, axis=0)
    return cv_g, cv_s


def date_stamp(dates, bsadf, cv_seq, min_dur=MIN_DUR):
    # Inception = first period BSADF > CV
    # Termination = first period BSADF <= CV after inception
    eps, in_b, si = [], False, None
    for i, (d, s, c) in enumerate(zip(dates, bsadf, cv_seq)):
        if np.isnan(s) or np.isnan(c): continue
        if s > c and not in_b:
            in_b, si = True, i
        elif s <= c and in_b:
            in_b = False
            if i - si >= min_dur:
                eps.append((dates.iloc[si], dates.iloc[i - 1]))
            si = None
    if in_b and si is not None and len(dates) - si >= min_dur:
        eps.append((dates.iloc[si], dates.iloc[-1]))
    return eps


def run_and_save(name, dates, prices, asset, country, sid):
    sep(name)
    y = np.log(np.asarray(prices, dtype=float))
    v = ~np.isnan(y)
    y, d = y[v], dates[v].reset_index(drop=True)
    T  = len(y)
    r0 = max(int(R0_FRAC * T), 12)
    print(f"    T={T}  r0={r0}  B={N_BOOT}")

    bsadf = bsadf_sequence(y)
    gv    = float(np.nanmax(bsadf))
    print(f"    Bootstrap...", end='', flush=True)
    t0 = time.time()
    cv_g, cv_s = wild_bootstrap_cv(y)
    print(f" {time.time()-t0:.1f}s | "
          f"GSADF={gv:.4f}  CV={cv_g:.4f}  "
          f"H0 rejected: {gv > cv_g}")

    eps = date_stamp(d, bsadf, cv_s)
    print(f"    Episodes: {len(eps)}")
    for s2, e in eps:
        print(f"      {pd.Timestamp(s2):%Y-%m} -> "
              f"{pd.Timestamp(e):%Y-%m}")

    # Two-panel plot
    log_p = np.log(np.asarray(prices, dtype=float))
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
    a1.plot(d, log_p[v], color='#1a5276', lw=1.3)
    for s2, e in eps:
        a1.axvspan(s2, e, alpha=0.25, color='#c0392b')
    a1.set_ylabel('Log Price')
    a1.grid(True, alpha=0.3)
    a1.set_title(f'{name} — GSADF (Wild Bootstrap, B={N_BOOT})',
                 fontweight='bold')
    v2 = ~np.isnan(bsadf)
    a2.plot(d[v2], bsadf[v2], color='#2980b9', lw=1.3, label='BSADF')
    a2.plot(d[v2], cv_s[v2],  color='#c0392b', lw=1.2, ls='--',
            label='CV 5% (Harvey et al. 2016)')
    for s2, e in eps:
        a2.axvspan(s2, e, alpha=0.18, color='#c0392b')
    a2.set_ylabel('BSADF')
    a2.set_xlabel('Date')
    a2.legend()
    a2.grid(True, alpha=0.3)
    a2.xaxis.set_major_locator(mdates.YearLocator(5))
    a2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.xticks(rotation=30)
    plt.tight_layout()
    fname = f'figures/gsadf_{sid.lower()}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    ok(f"Saved {fname}")

    for s2, e in eps:
        all_eps.append({
            'asset':       asset,
            'country':     country,
            'inception':   s2,
            'termination': e,
            'series':      sid,
        })
    gsadf_rows.append({
        'series':      sid,
        'gsadf':       gv,
        'cv5pct':      cv_g,
        'H0_rejected': gv > cv_g,
        'n_episodes':  len(eps),
    })


# ── Load data ──────────────────────────────────────────
yf_data = pd.read_csv(
    'data/raw/yfinance_data.csv',
    parse_dates=['date'], index_col='date')

try:
    fred_df = pd.read_csv(
        'data/raw/fred_data.csv',
        parse_dates=['date'], index_col='date')
except FileNotFoundError:
    fred_df = pd.DataFrame()
    warn("fred_data.csv not found — run Step 2 first")

all_eps    = []
gsadf_rows = []

# ── Equity indices (Gap 3: cross-asset and cross-country) ─
equity_list = [
    ('SP500',   'US'), ('FTSE100', 'GB'), ('DAX',    'DE'),
    ('CAC40',   'FR'), ('Nikkei',  'JP'), ('TSX',    'CA'),
    ('ASX200',  'AU'), ('KOSPI',   'KR'), ('BOVESPA','BR'),
    ('SENSEX',  'IN'), ('HSI',     'HK'),
]
for col, ctry in equity_list:
    if col in yf_data.columns:
        s = yf_data[col].dropna()
        if len(s) >= 60:
            run_and_save(
                col,
                s.index.to_series().reset_index(drop=True),
                s.reset_index(drop=True),
                'Equity', ctry, col)

# ── US Housing — Case-Shiller from FRED (Gap 3) ────────
if 'case_shiller_hpi' in fred_df.columns:
    hp = fred_df['case_shiller_hpi'].resample('MS').last().dropna()
    run_and_save(
        'US Case-Shiller HPI',
        hp.index.to_series().reset_index(drop=True),
        hp.reset_index(drop=True),
        'Housing', 'US', 'US_HPI')

# ── US 10Y Treasury yield — sovereign bond (Gap 3) ─────
if 'US_10Y' in yf_data.columns:
    bd = yf_data['US_10Y'].dropna()
    run_and_save(
        'US 10Y Treasury',
        bd.index.to_series().reset_index(drop=True),
        bd.reset_index(drop=True),
        'SovBond', 'US', 'US_10Y')

# ── HY Credit Spread — Astill et al. covariate (Gap 4) ─
if 'hy_credit_spread' in fred_df.columns:
    hy = fred_df['hy_credit_spread'].resample('MS').last().dropna()
    run_and_save(
        'HY Credit Spread',
        hy.index.to_series().reset_index(drop=True),
        hy.reset_index(drop=True),
        'Credit', 'US', 'HY_Spread')

# ── VIX ────────────────────────────────────────────────
if 'VIX' in yf_data.columns:
    vx = yf_data['VIX'].dropna()
    run_and_save(
        'VIX Volatility Index',
        vx.index.to_series().reset_index(drop=True),
        vx.reset_index(drop=True),
        'Volatility', 'US', 'VIX')

# ── Save Stage 1 outputs ────────────────────────────────
eps_df = pd.DataFrame(all_eps)
if not eps_df.empty:
    eps_df['inception']        = pd.to_datetime(eps_df['inception'])
    eps_df['termination']      = pd.to_datetime(eps_df['termination'])
    eps_df['duration_months']  = (
        (eps_df['termination'] - eps_df['inception']).dt.days / 30.4
    ).round(1)
    eps_df.to_csv('data/cleaned/bubble_dates.csv', index=False)
    ok(f"bubble_dates.csv: {len(eps_df)} episodes across "
       f"{eps_df['asset'].nunique()} asset classes")
else:
    warn("No episodes detected — check data coverage")

gsadf_df = pd.DataFrame(gsadf_rows)
gsadf_df.to_csv('results/gsadf_summary.csv', index=False)
print("\nGSADF Summary:")
print(gsadf_df.to_string(index=False))
print("\nStep 3 complete.")


  STEP 3 — GSADF BUBBLE DETECTION

--- SP500 ---
    T=497  r0=49  B=99
    Bootstrap... 971.2s | GSADF=3.0343  CV=3.2652  H0 rejected: False
    Episodes: 5
      1995-06 -> 1998-07
      1999-03 -> 2000-04
      2017-12 -> 2018-02
      2021-10 -> 2022-03
      2024-02 -> 2026-05
  OK  Saved figures/gsadf_sp500.png

--- FTSE100 ---
    T=497  r0=49  B=99
    Bootstrap... 909.3s | GSADF=1.7165  CV=3.0247  H0 rejected: False
    Episodes: 3
      1997-04 -> 1997-10
      1997-12 -> 1998-07
      2025-12 -> 2026-02
  OK  Saved figures/gsadf_ftse100.png

--- DAX ---
    T=462  r0=46  B=99
    Bootstrap... 788.1s | GSADF=2.8996  CV=2.8617  H0 rejected: True
    Episodes: 4
      1996-11 -> 1998-07
      1999-11 -> 2000-05
      2007-04 -> 2007-06
      2025-05 -> 2025-07
  OK  Saved figures/gsadf_dax.png

--- CAC40 ---
    T=435  r0=43  B=99
    Bootstrap... 686.5s | GSADF=1.7049  CV=2.7219  H0 rejected: False
    Episodes: 4
      1998-02 -> 1998-07
      1999-11 -> 2000-10
      2006-02

In [ ]:
# ════════════════════════════════════════════════════════
# STEP 4 — Feature Panel and Target Variable
# 47 candidate indicators across 5 categories
# Target Y from GSADF inception dates (Gap 2)
# Addresses Gaps 2, 4, 5
# ════════════════════════════════════════════════════════
import warnings
import numpy as np
import pandas as pd
import os
warnings.filterwarnings('ignore')

LEAD_MIN = 4
LEAD_MAX = 12

def ok(m):   print(f"  OK  {m}")
def warn(m): print(f"  !!  {m}")
def err(m):  print(f"  XX  {m}")

print("=" * 55)
print("  STEP 4 — FEATURE PANEL + TARGET")
print("=" * 55)

# ── Diagnose working directory ─────────────────────────
print(f"\n  Working directory: {os.getcwd()}")
print(f"  Files in current dir: {os.listdir('.')[:10]}")
if os.path.exists('data'):
    print(f"  data/ contents: {os.listdir('data')}")
    if os.path.exists('data/raw'):
        print(f"  data/raw/ contents: {os.listdir('data/raw')}")

# ── Create folders if missing ──────────────────────────
for d in ['data/raw', 'data/cleaned', 'figures', 'results']:
    os.makedirs(d, exist_ok=True)

# ── Helper: nearest date (pandas 2.x compatible) ───────
def nearest_idx(index, target_date):
    return int(np.abs(
        (index - target_date).total_seconds()
    ).argmin())

# ── Check required files exist ─────────────────────────
required = {
    'data/raw/yfinance_data.csv': 'Run Step 2 first',
    'data/cleaned/bubble_dates.csv': 'Run Step 3 first',
}
missing = []
for path, msg in required.items():
    if not os.path.exists(path):
        err(f"Missing: {path}  ({msg})")
        missing.append(path)
    else:
        ok(f"Found: {path}")

if missing:
    print("""
  ════════════════════════════════════════════════════
  MISSING FILES — likely causes:
  ════════════════════════════════════════════════════
  1. Steps were run in different sessions/kernels.
     Each Colab/Jupyter session starts fresh.
     Run Steps 0-3 again in THIS session.

  2. Files saved to wrong folder.
     All steps use relative paths (data/raw/).
     Make sure you run from the SAME folder each time.

  FIX: Run this in a new cell first:
       import os
       os.chdir('/content')        # Colab
       # or
       os.chdir('/your/project')   # Jupyter local

  Then re-run Steps 0, 1, 2, 3, then this step.
  ════════════════════════════════════════════════════
    """)
    raise FileNotFoundError(
        "Required data files missing. Run Steps 2 and 3 first, "
        "in the same session and working directory."
    )

# ── Load data ──────────────────────────────────────────
yf_data = pd.read_csv(
    'data/raw/yfinance_data.csv',
    parse_dates=['date'], index_col='date')
ok(f"Yahoo Finance: {yf_data.shape}")

eps_df = pd.read_csv(
    'data/cleaned/bubble_dates.csv',
    parse_dates=['inception', 'termination'])
ok(f"Bubble dates: {len(eps_df)} episodes")

fred_df = pd.DataFrame()
if os.path.exists('data/raw/fred_data.csv'):
    fred_df = pd.read_csv(
        'data/raw/fred_data.csv',
        parse_dates=['date'], index_col='date')
    ok(f"FRED data: {fred_df.shape}")
else:
    warn("fred_data.csv not found — FRED features will be skipped")
    warn("Set FRED key in Step 1 and re-run Step 2 to get FRED data")

# ── Base panel ─────────────────────────────────────────
panel = yf_data[['SP500']].copy().rename(columns={'SP500': 'price'})

# Category 1: Momentum and volatility
panel['mom_1m']     = panel['price'].pct_change(1)
panel['mom_3m']     = panel['price'].pct_change(3)
panel['mom_6m']     = panel['price'].pct_change(6)
panel['mom_12m']    = panel['price'].pct_change(12)
panel['vol_3m']     = panel['price'].pct_change().rolling(3).std()
panel['vol_6m']     = panel['price'].pct_change().rolling(6).std()
panel['vol_12m']    = panel['price'].pct_change().rolling(12).std()
panel['ret_excess'] = (panel['price'].pct_change()
                       - panel['price'].pct_change().rolling(60).mean())
ok("Category 1: momentum + volatility (8 features)")

# Category 2: Valuation
if not fred_df.empty and 'cape_shiller' in fred_df.columns:
    panel['CAPE'] = (fred_df['cape_shiller']
                     .resample('MS').last()
                     .reindex(panel.index, method='ffill'))
    panel['CAPE_mean'] = panel['CAPE'].rolling(120).mean()
    panel['CAPE_std']  = panel['CAPE'].rolling(120).std()
    panel['CAPE_z']    = ((panel['CAPE'] - panel['CAPE_mean'])
                          / panel['CAPE_std'].replace(0, np.nan))
    ok("Category 2: CAPE z-score from FRED")
elif not fred_df.empty and 'pe_ratio' in fred_df.columns:
    panel['PE'] = (fred_df['pe_ratio']
                   .resample('MS').last()
                   .reindex(panel.index, method='ffill'))
    panel['PE_mean'] = panel['PE'].rolling(120).mean()
    panel['PE_std']  = panel['PE'].rolling(120).std()
    panel['CAPE_z']  = ((panel['PE'] - panel['PE_mean'])
                        / panel['PE_std'].replace(0, np.nan))
    warn("Category 2: PE ratio used as CAPE proxy")
else:
    warn("Category 2: no valuation data available")

if not fred_df.empty and 'div_yield' in fred_df.columns:
    panel['div_yield'] = (fred_df['div_yield']
                          .resample('MS').last()
                          .reindex(panel.index, method='ffill'))

# Category 3: Credit aggregates (FRED)
if not fred_df.empty:
    fred_m = fred_df.resample('MS').last()
    credit_cols = [
        'hy_credit_spread', 'ig_credit_spread',
        'nfci_financial_conditions', 'stl_financial_stress',
        'consumer_credit_total', 'commercial_loans',
        'real_estate_loans', 'domestic_credit_private',
        'federal_debt_pct_gdp', 'delinquency_mortgage',
        'delinquency_credit_card', 'delinquency_business',
        'fed_funds_rate', 'term_spread_10y2y', 'term_spread_10y3m',
        'mortgage_rate_30y', 'vix', 'case_shiller_hpi',
        'oil_wti', 'gold_price', 'real_gdp', 'unemployment_us',
        'cpi_us', 'industrial_production', 'housing_starts',
    ]
    added = 0
    for col in credit_cols:
        if col in fred_m.columns:
            panel[col] = fred_m[col].reindex(panel.index, method='ffill')
            added += 1
    ok(f"Category 3: {added} FRED credit/macro features")
else:
    warn("Category 3: FRED data not available — skipped")

# Category 4: Yahoo Finance covariates
yf_extra = ['VIX', 'US_10Y', 'US_3M', 'US_30Y', 'term_spread_yf',
            'FTSE100', 'DAX', 'Nikkei', 'BOVESPA', 'SENSEX']
added4 = 0
for col in yf_extra:
    if col in yf_data.columns:
        panel[col] = yf_data[col].reindex(panel.index, method='ffill')
        added4 += 1
ok(f"Category 4: {added4} Yahoo Finance covariates")

# Category 5: Derived variables
derived = 0
if 'US_10Y' in panel.columns and 'cpi_us' in panel.columns:
    panel['real_rate_10y'] = (panel['US_10Y']
                               - panel['cpi_us'].pct_change(12) * 100)
    derived += 1
if 'hy_credit_spread' in panel.columns and 'ig_credit_spread' in panel.columns:
    panel['spread_diff'] = (panel['hy_credit_spread']
                             - panel['ig_credit_spread'])
    derived += 1
if 'consumer_credit_total' in panel.columns:
    panel['credit_growth_12m'] = panel['consumer_credit_total'].pct_change(12)
    derived += 1
if 'case_shiller_hpi' in panel.columns:
    panel['house_price_chg12'] = panel['case_shiller_hpi'].pct_change(12)
    derived += 1
ok(f"Category 5: {derived} derived macro variables")

# ── Lags 3m / 6m / 12m ────────────────────────────────
base_feats = [c for c in panel.columns
              if c not in ['price']
              and panel[c].notna().mean() > 0.5]
for col in base_feats:
    panel[f'{col}_lag3']  = panel[col].shift(3)
    panel[f'{col}_lag6']  = panel[col].shift(6)
    panel[f'{col}_lag12'] = panel[col].shift(12)

panel = panel.replace([np.inf, -np.inf], np.nan).sort_index()
ok(f"Lag features: {len(base_feats)} base x 3 lags")

# ── Target Y from GSADF dates (Gap 2) ─────────────────
panel['Y'] = 0
us_eps = eps_df[eps_df['country'] == 'US']

if not us_eps.empty:
    for _, ep in us_eps.iterrows():
        inc = pd.Timestamp(ep['inception'])
        for lead in range(LEAD_MIN, LEAD_MAX + 1):
            sig = inc - pd.DateOffset(months=lead)
            idx = nearest_idx(panel.index, sig)
            panel.iloc[idx, panel.columns.get_loc('Y')] = 1

    for _, ep in us_eps.iterrows():
        for anchor in [pd.Timestamp(ep['inception']),
                       pd.Timestamp(ep['termination'])]:
            for off in range(-3, 4):
                excl = anchor + pd.DateOffset(months=off)
                idx  = nearest_idx(panel.index, excl)
                if panel.iloc[idx]['Y'] == 0:
                    panel.iloc[idx, panel.columns.get_loc('Y')] = np.nan

    panel = panel.dropna(subset=['Y'])

pos = int(panel['Y'].sum())
ok(f"Target Y: {pos} positive months "
   f"({panel['Y'].mean()*100:.1f}%) from {len(us_eps)} episodes")

if pos == 0:
    warn("No US GSADF episodes — using known historical fallback")
    panel['Y'] = 0
    for s, e in [('1997-06-01', '2000-06-01'),
                 ('2004-01-01', '2007-09-01'),
                 ('2020-06-01', '2021-12-01')]:
        panel.loc[(panel.index >= s) & (panel.index <= e), 'Y'] = 1
    ok(f"Fallback: {int(panel['Y'].sum())} positive months")

# ── Final feature matrix ───────────────────────────────
feat_cols = [c for c in panel.columns
             if c not in ['Y', 'price']
             and panel[c].notna().mean() > 0.55]

panel_ml = (panel[feat_cols + ['Y']]
            .dropna()
            .reset_index())

if panel_ml.columns[0] != 'date':
    panel_ml = panel_ml.rename(columns={panel_ml.columns[0]: 'date'})

panel_ml.to_csv('data/cleaned/ml_panel.csv', index=False)

ok(f"ml_panel.csv saved:")
ok(f"  Observations : {len(panel_ml)}")
ok(f"  Features     : {len(feat_cols)}")
ok(f"  Positive Y=1 : {int(panel_ml['Y'].sum())} "
   f"({panel_ml['Y'].mean()*100:.1f}%)")
ok(f"  Date range   : {panel_ml['date'].min().date()} — "
   f"{panel_ml['date'].max().date()}")

print("\nStep 4 complete.")


In [1]:
# ════════════════════════════════════════════════════════════════════════
# STEP 4 — Feature Panel + Target Variable
# SELF-CONTAINED: downloads all data internally
# No prior steps required — just set your FRED key below
#
# Downloads from:
#   FRED API     — SP500, yields, credit, housing, macro, VIX
#   Yahoo Finance — equity indices, Treasury yields
#
# Builds:
#   47 candidate features across 5 theoretical categories
#   Target Y from GSADF bubble dates (if available) or
#              known historical US bubble episodes (fallback)
#
# Addresses Gaps 2, 4, 5
# ════════════════════════════════════════════════════════════════════════
import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# ── SET YOUR FRED KEY HERE ────────────────────────────────────────────
FRED_KEY   = os.environ.get('FRED_API_KEY', 'paste_your_key_here')
START_DATE = '1980-01-01'
LEAD_MIN   = 4    # months before bubble inception (lower bound)
LEAD_MAX   = 12   # months before bubble inception (upper bound)

# ─────────────────────────────────────────────────────────────────────
def ok(m):   print(f"  OK  {m}")
def warn(m): print(f"  !!  {m}")
def err(m):  print(f"  XX  {m}")
def sep(t):  print(f"\n--- {t} ---")

for d in ['data/raw', 'data/cleaned', 'figures', 'results']:
    os.makedirs(d, exist_ok=True)

def nearest_idx(index, target_date):
    """Nearest DatetimeIndex position — pandas 2.x compatible."""
    return int(np.abs(
        (index - target_date).total_seconds()
    ).argmin())

print("=" * 60)
print("  STEP 4 — FEATURE PANEL + TARGET (self-contained)")
print("=" * 60)


# ════════════════════════════════════════════════════════════════════════
# SECTION A — DATA DOWNLOAD
# ════════════════════════════════════════════════════════════════════════

# ── A1. FRED ──────────────────────────────────────────────────────────
sep("A1. FRED API download")

fred_df = pd.DataFrame()

if not FRED_KEY or FRED_KEY == 'paste_your_key_here':
    warn("FRED key not set.")
    warn("Get free key at: fred.stlouisfed.org/docs/api_key.html")
    warn("Then set: FRED_KEY = 'your_key' at the top of this script")
    warn("Continuing without FRED — some features will be missing")
else:
    try:
        from fredapi import Fred
        fred_obj = Fred(api_key=FRED_KEY)

        fred_series = {
            # Asset prices
            'SP500':            'sp500',
            'VIXCLS':           'vix',
            'CSUSHPISA':        'case_shiller_hpi',
            'DCOILWTICO':       'oil_wti',
            'GOLDAMGBD228NLBM': 'gold_price',
            # Monetary policy and rates
            'FEDFUNDS':         'fed_funds_rate',
            'T10Y2Y':           'term_spread_10y2y',
            'T10Y3M':           'term_spread_10y3m',
            'MORTGAGE30US':     'mortgage_rate_30y',
            # Credit spreads (Astill et al. 2023 — Gap 4)
            'BAMLH0A0HYM2':     'hy_credit_spread',
            'BAMLC0A0CM':       'ig_credit_spread',
            'NFCI':             'nfci_financial_conditions',
            'STLFSI4':          'stl_financial_stress',
            # Credit quantities (Schularick-Taylor — Gap 5)
            'TOTALSL':          'consumer_credit_total',
            'BUSLOANS':         'commercial_loans',
            'REALLN':           'real_estate_loans',
            'DPCREDIT':         'domestic_credit_private',
            # Bank health proxies
            'DRSFRMACBS':       'delinquency_mortgage',
            'DRCCLACBS':        'delinquency_credit_card',
            'DRBLACBS':         'delinquency_business',
            'GFDEGDQ188S':      'federal_debt_pct_gdp',
            # Real economy
            'GDPC1':            'real_gdp',
            'UNRATE':           'unemployment_us',
            'CPIAUCSL':         'cpi_us',
            'INDPRO':           'industrial_production',
            'HOUST':            'housing_starts',
        }

        fred_raw = {}
        for sid, name in fred_series.items():
            try:
                fred_raw[name] = fred_obj.get_series(
                    sid, observation_start=START_DATE)
                time.sleep(0.05)
            except Exception as ex:
                warn(f"  FRED {sid}: {str(ex)[:45]}")

        # Shiller valuation series
        for sid, name in [('CAPE',    'cape_shiller'),
                           ('SP500PE', 'pe_ratio'),
                           ('SP500DIV','div_yield')]:
            try:
                fred_raw[name] = (
                    fred_obj.get_series(sid, observation_start=START_DATE)
                    .resample('MS').last()
                )
                ok(f"  {name} ({sid})")
            except Exception:
                warn(f"  {sid} not in FRED — skipping")

        fred_df = pd.DataFrame(fred_raw)
        fred_df.index = pd.to_datetime(fred_df.index)
        fred_df.index.name = 'date'
        fred_df = fred_df.sort_index()
        fred_df.to_csv('data/raw/fred_data.csv')
        ok(f"FRED: {fred_df.shape[0]} obs x {fred_df.shape[1]} series "
           f"({fred_df.index.min().date()} to {fred_df.index.max().date()})")

    except Exception as e:
        err(f"FRED connection failed: {e}")
        warn("Continuing without FRED data")


# ── A2. Yahoo Finance ─────────────────────────────────────────────────
sep("A2. Yahoo Finance download")

yf_df = pd.DataFrame()
try:
    import yfinance as yf

    all_tickers = {
        # Equity indices (Gap 3)
        'SP500':   '^GSPC',   'FTSE100': '^FTSE',
        'DAX':     '^GDAXI',  'CAC40':   '^FCHI',
        'Nikkei':  '^N225',   'TSX':     '^GSPTSE',
        'ASX200':  '^AXJO',   'KOSPI':   '^KS11',
        'IBEX35':  '^IBEX',   'AEX':     '^AEX',
        'SMI':     '^SSMI',   'BOVESPA': '^BVSP',
        'SENSEX':  '^BSESN',  'HSI':     '^HSI',
        'NIFTY50': '^NSEI',   'STI':     '^STI',
        # Rates and volatility
        'US_10Y':  '^TNX',    'US_3M':   '^IRX',
        'US_30Y':  '^TYX',    'VIX_YF':  '^VIX',
    }

    yf_raw = yf.download(
        list(all_tickers.values()),
        start=START_DATE,
        interval='1mo',
        auto_adjust=True,
        progress=False,
        threads=True,
    )['Close']

    reverse = {v: k for k, v in all_tickers.items()}
    yf_raw.columns = [reverse.get(c, c) for c in yf_raw.columns]
    yf_raw.index   = pd.to_datetime(yf_raw.index)
    yf_raw.index.name = 'date'
    yf_raw = yf_raw.sort_index()

    if 'US_10Y' in yf_raw.columns and 'US_3M' in yf_raw.columns:
        yf_raw['term_spread_yf'] = yf_raw['US_10Y'] - yf_raw['US_3M']

    yf_raw.to_csv('data/raw/yfinance_data.csv')
    yf_df = yf_raw
    ok(f"Yahoo Finance: {yf_df.shape[0]} months x {yf_df.shape[1]} series")

except Exception as e:
    err(f"Yahoo Finance failed: {e}")
    warn("Continuing without Yahoo Finance data")


# ════════════════════════════════════════════════════════════════════════
# SECTION B — FEATURE ENGINEERING
# ════════════════════════════════════════════════════════════════════════
sep("B. Building feature panel")

# ── Choose price base ─────────────────────────────────────────────────
if 'SP500' in yf_df.columns:
    price_source = yf_df[['SP500']].copy()
    price_label  = 'Yahoo Finance SP500'
elif not fred_df.empty and 'sp500' in fred_df.columns:
    price_source = fred_df[['sp500']].resample('MS').last().copy()
    price_source.columns = ['SP500']
    price_label = 'FRED SP500'
else:
    err("No SP500 price data available — cannot build panel")
    raise RuntimeError("Need either FRED or Yahoo Finance SP500 data.")

panel = price_source.rename(columns={'SP500': 'price'})
ok(f"Base price: {price_label} ({len(panel)} months)")

# ── Category 1: Price momentum and volatility ─────────────────────────
panel['mom_1m']     = panel['price'].pct_change(1)
panel['mom_3m']     = panel['price'].pct_change(3)
panel['mom_6m']     = panel['price'].pct_change(6)
panel['mom_12m']    = panel['price'].pct_change(12)
panel['vol_3m']     = panel['price'].pct_change().rolling(3).std()
panel['vol_6m']     = panel['price'].pct_change().rolling(6).std()
panel['vol_12m']    = panel['price'].pct_change().rolling(12).std()
panel['ret_excess'] = (panel['price'].pct_change()
                       - panel['price'].pct_change().rolling(60).mean())
ok("Category 1: momentum + volatility (8 features)")

# ── Category 2: Valuation ─────────────────────────────────────────────
if not fred_df.empty and 'cape_shiller' in fred_df.columns:
    panel['CAPE'] = (fred_df['cape_shiller']
                     .resample('MS').last()
                     .reindex(panel.index, method='ffill'))
    panel['CAPE_mean'] = panel['CAPE'].rolling(120).mean()
    panel['CAPE_std']  = panel['CAPE'].rolling(120).std()
    panel['CAPE_z']    = ((panel['CAPE'] - panel['CAPE_mean'])
                          / panel['CAPE_std'].replace(0, np.nan))
    ok("Category 2: CAPE z-score (FRED CAPE series)")

elif not fred_df.empty and 'pe_ratio' in fred_df.columns:
    panel['PE'] = (fred_df['pe_ratio']
                   .resample('MS').last()
                   .reindex(panel.index, method='ffill'))
    panel['PE_mean'] = panel['PE'].rolling(120).mean()
    panel['PE_std']  = panel['PE'].rolling(120).std()
    panel['CAPE_z']  = ((panel['PE'] - panel['PE_mean'])
                        / panel['PE_std'].replace(0, np.nan))
    warn("Category 2: PE ratio used as CAPE proxy (CAPE not found)")
else:
    warn("Category 2: no valuation data — skipped")

if not fred_df.empty and 'div_yield' in fred_df.columns:
    panel['div_yield'] = (fred_df['div_yield']
                          .resample('MS').last()
                          .reindex(panel.index, method='ffill'))

# ── Category 3: Credit aggregates (Schularick-Taylor, Gap 5) ──────────
if not fred_df.empty:
    fred_m = fred_df.resample('MS').last()
    credit_cols = [
        'hy_credit_spread',         'ig_credit_spread',
        'nfci_financial_conditions', 'stl_financial_stress',
        'consumer_credit_total',     'commercial_loans',
        'real_estate_loans',         'domestic_credit_private',
        'federal_debt_pct_gdp',      'delinquency_mortgage',
        'delinquency_credit_card',   'delinquency_business',
        'fed_funds_rate',            'term_spread_10y2y',
        'term_spread_10y3m',         'mortgage_rate_30y',
        'vix',                       'case_shiller_hpi',
        'oil_wti',                   'gold_price',
        'real_gdp',                  'unemployment_us',
        'cpi_us',                    'industrial_production',
        'housing_starts',
    ]
    added3 = sum(
        1 for col in credit_cols
        if col in fred_m.columns
        and not panel.assign(
            **{col: fred_m[col].reindex(panel.index, method='ffill')}
        ).empty
    )
    for col in credit_cols:
        if col in fred_m.columns:
            panel[col] = fred_m[col].reindex(panel.index, method='ffill')
    ok(f"Category 3: {len([c for c in credit_cols if c in fred_m.columns])} "
       f"FRED credit/macro features")
else:
    warn("Category 3: FRED data not available — skipped")

# ── Category 4: Market covariates from Yahoo (Astill et al. 2023) ─────
yf_extra = ['VIX_YF', 'US_10Y', 'US_3M', 'US_30Y', 'term_spread_yf',
            'FTSE100', 'DAX', 'Nikkei', 'BOVESPA', 'SENSEX',
            'KOSPI', 'ASX200', 'HSI']
added4 = 0
if not yf_df.empty:
    for col in yf_extra:
        if col in yf_df.columns:
            panel[col] = yf_df[col].reindex(panel.index, method='ffill')
            added4 += 1
ok(f"Category 4: {added4} Yahoo Finance covariates")

# ── Category 5: Derived macro variables ──────────────────────────────
derived = 0
if 'US_10Y' in panel.columns and 'cpi_us' in panel.columns:
    panel['real_rate_10y'] = (panel['US_10Y']
                               - panel['cpi_us'].pct_change(12) * 100)
    derived += 1
if 'hy_credit_spread' in panel.columns and 'ig_credit_spread' in panel.columns:
    panel['spread_diff'] = (panel['hy_credit_spread']
                             - panel['ig_credit_spread'])
    derived += 1
if 'consumer_credit_total' in panel.columns:
    panel['credit_growth_12m'] = panel['consumer_credit_total'].pct_change(12)
    derived += 1
if 'case_shiller_hpi' in panel.columns:
    panel['house_price_chg12'] = panel['case_shiller_hpi'].pct_change(12)
    derived += 1
if 'real_gdp' in panel.columns:
    panel['gdp_growth_yoy'] = panel['real_gdp'].pct_change(4) * 100
    derived += 1
if 'sp500' in panel.columns or 'price' in panel.columns:
    p = panel.get('sp500', panel['price'])
    panel['sp500_gap'] = (p - p.rolling(60).mean()) / p.rolling(60).std()
    derived += 1
ok(f"Category 5: {derived} derived macro variables")

# ── Lags 3m / 6m / 12m (Gap 5: temporal accumulation) ────────────────
base_feats = [c for c in panel.columns
              if c not in ['price']
              and panel[c].notna().mean() > 0.5]
for col in base_feats:
    panel[f'{col}_lag3']  = panel[col].shift(3)
    panel[f'{col}_lag6']  = panel[col].shift(6)
    panel[f'{col}_lag12'] = panel[col].shift(12)

panel = panel.replace([np.inf, -np.inf], np.nan).sort_index()
n_raw = len([c for c in panel.columns if c not in ['price']])
ok(f"Lag features: {len(base_feats)} base x 3 lags = {len(base_feats)*3} new cols")
ok(f"Total candidate features before filtering: {n_raw}")


# ════════════════════════════════════════════════════════════════════════
# SECTION C — TARGET VARIABLE Y
# ════════════════════════════════════════════════════════════════════════
sep("C. Constructing target variable Y")

panel['Y'] = 0

# ── Try to load GSADF bubble dates from Step 3 ────────────────────────
gsadf_loaded = False
if os.path.exists('data/cleaned/bubble_dates.csv'):
    try:
        eps_df = pd.read_csv(
            'data/cleaned/bubble_dates.csv',
            parse_dates=['inception', 'termination'])
        us_eps = eps_df[eps_df['country'] == 'US']

        if not us_eps.empty:
            for _, ep in us_eps.iterrows():
                inc = pd.Timestamp(ep['inception'])
                for lead in range(LEAD_MIN, LEAD_MAX + 1):
                    sig = inc - pd.DateOffset(months=lead)
                    idx = nearest_idx(panel.index, sig)
                    panel.iloc[idx, panel.columns.get_loc('Y')] = 1

            # Exclude ambiguous zone +/- 3 months
            for _, ep in us_eps.iterrows():
                for anchor in [pd.Timestamp(ep['inception']),
                               pd.Timestamp(ep['termination'])]:
                    for off in range(-3, 4):
                        excl = anchor + pd.DateOffset(months=off)
                        idx  = nearest_idx(panel.index, excl)
                        if panel.iloc[idx]['Y'] == 0:
                            panel.iloc[
                                idx, panel.columns.get_loc('Y')] = np.nan

            panel = panel.dropna(subset=['Y'])
            gsadf_loaded = True
            ok(f"GSADF dates loaded from bubble_dates.csv "
               f"({len(us_eps)} US episodes)")

    except Exception as ex:
        warn(f"Could not load bubble_dates.csv: {ex}")

# ── Fallback: Laeven-Valencia known US bubble/crisis periods ──────────
# Used when Step 3 has not been run yet.
# Based on: PSY (2015), Phillips & Shi (2018), LV (2020)
if not gsadf_loaded or int(panel['Y'].sum()) == 0:

    if gsadf_loaded:
        warn("GSADF loaded but no positive episodes found")
        warn("Falling back to known historical periods")
    else:
        warn("bubble_dates.csv not found — Step 3 not yet run")
        warn("Using known US bubble periods as Y target")
        warn("Run Step 3 first for full GSADF-based target")

    # Reset Y
    panel['Y'] = 0

    # Known US bubble build-up periods (4-12 months before peaks)
    # Sources: PSY (2015), Phillips & Shi (2018), Laeven-Valencia (2020)
    known_bubble_windows = [
        # (signal_start, signal_end, label)
        ('1997-06-01', '2000-03-01', 'Dot-com build-up'),
        ('2004-06-01', '2007-06-01', 'Housing build-up'),
        ('2009-06-01', '2010-06-01', 'Post-GFC recovery rally'),
        ('2020-06-01', '2021-09-01', 'Post-COVID exuberance'),
    ]

    for s, e, label in known_bubble_windows:
        mask = (panel.index >= s) & (panel.index <= e)
        panel.loc[mask, 'Y'] = 1
        n_pos = int(mask.sum())
        ok(f"  Y=1: {s} to {e} ({label}, {n_pos} months)")

pos = int(panel['Y'].sum())
pct = panel['Y'].mean() * 100
ok(f"Final target: Y=1 = {pos} months ({pct:.1f}% of sample)")

if pos < 10:
    warn("Very few positive examples — check data coverage")


# ════════════════════════════════════════════════════════════════════════
# SECTION D — FINAL FEATURE MATRIX
# ════════════════════════════════════════════════════════════════════════
sep("D. Building final feature matrix")

# Keep features with at least 55% non-missing values
feat_cols = [
    c for c in panel.columns
    if c not in ['Y', 'price']
    and panel[c].notna().mean() > 0.55
]

panel_ml = (
    panel[feat_cols + ['Y']]
    .dropna()
    .reset_index()
)

# Ensure date column is named 'date'
if panel_ml.columns[0] != 'date':
    panel_ml = panel_ml.rename(columns={panel_ml.columns[0]: 'date'})

panel_ml.to_csv('data/cleaned/ml_panel.csv', index=False)

ok(f"ml_panel.csv saved successfully:")
ok(f"  Observations : {len(panel_ml)}")
ok(f"  Features     : {len(feat_cols)}")
ok(f"  Positive Y=1 : {int(panel_ml['Y'].sum())} "
   f"({panel_ml['Y'].mean()*100:.1f}%)")
ok(f"  Negative Y=0 : {int((panel_ml['Y']==0).sum())}")
ok(f"  Date range   : {panel_ml['date'].min().date()} — "
   f"{panel_ml['date'].max().date()}")

print(f"\n  Top 10 features by non-missing rate:")
top10 = (panel[feat_cols].notna().mean()
         .sort_values(ascending=False).head(10))
for feat, pct in top10.items():
    print(f"    {feat:<40} {pct*100:.0f}%")

print("\nStep 4 complete.")
print("Next: run Step 5 (step5_ml_classifiers.py)")

  STEP 4 — FEATURE PANEL + TARGET (self-contained)

--- A1. FRED API download ---
  !!  FRED key not set.
  !!  Get free key at: fred.stlouisfed.org/docs/api_key.html
  !!  Then set: FRED_KEY = 'your_key' at the top of this script
  !!  Continuing without FRED — some features will be missing

--- A2. Yahoo Finance download ---
  OK  Yahoo Finance: 497 months x 21 series

--- B. Building feature panel ---
  OK  Base price: Yahoo Finance SP500 (497 months)
  OK  Category 1: momentum + volatility (8 features)
  !!  Category 2: no valuation data — skipped
  !!  Category 3: FRED data not available — skipped
  OK  Category 4: 13 Yahoo Finance covariates
  OK  Category 5: 1 derived macro variables
  OK  Lag features: 22 base x 3 lags = 66 new cols
  OK  Total candidate features before filtering: 88

--- C. Constructing target variable Y ---
  !!  bubble_dates.csv not found — Step 3 not yet run
  !!  Using known US bubble periods as Y target
  !!  Run Step 3 first for full GSADF-based target
 

In [1]:
# ════════════════════════════════════════════════════════════════════════
# STEP 5 — Recursive ML Evaluation (6 classifiers)
# SELF-CONTAINED: downloads all data and builds features internally
# No prior steps required — just set your FRED key below
#
# Implements:
#   Section 3.7.1 — Recursive expanding-window (no data leakage)
#   Section 3.7.2 — AUROC, Relative Usefulness Ur, Brier, HR@10%FPR
#   Section 3.5.3 — Boruta → correlation → LASSO feature selection
#   Section 3.5.4 — SMOTE at 5:1 ratio
#   Gaps 1, 5
# ════════════════════════════════════════════════════════════════════════
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.linear_model   import LogisticRegression
from sklearn.ensemble       import RandomForestClassifier
from sklearn.svm            import SVC
from sklearn.neighbors      import KNeighborsClassifier
from sklearn.preprocessing  import StandardScaler
from sklearn.metrics        import (roc_auc_score, brier_score_loss,
                                     roc_curve)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
warnings.filterwarnings('ignore')

# ── SET YOUR FRED KEY HERE ────────────────────────────────────────────
FRED_KEY = os.environ.get('FRED_API_KEY', 'paste_your_key_here')

# ── Config ────────────────────────────────────────────────────────────
START_DATE   = '1980-01-01'
LEAD_MIN     = 4
LEAD_MAX     = 12
RANDOM_STATE = 42
BURN_IN      = 60      # months before first prediction window
MU           = 0.5     # Alessi-Detken loss ratio
SMOTE_RATIO  = 0.20    # SMOTE minority:majority target (5:1)
CORR_THRESH  = 0.85    # pairwise correlation threshold

for d in ['data/raw', 'data/cleaned', 'figures', 'results']:
    os.makedirs(d, exist_ok=True)

def ok(m):   print(f"  OK  {m}")
def warn(m): print(f"  !!  {m}")
def err(m):  print(f"  XX  {m}")
def sep(t):  print(f"\n--- {t} ---")

def nearest_idx(index, target_date):
    """Nearest DatetimeIndex position — pandas 2.x compatible."""
    return int(np.abs((index - target_date).total_seconds()).argmin())

print("=" * 60)
print("  STEP 5 — RECURSIVE ML EVALUATION (self-contained)")
print("=" * 60)


# ════════════════════════════════════════════════════════════════════════
# SECTION A — DATA DOWNLOAD
# Skipped automatically if ml_panel.csv already exists from Step 4
# ════════════════════════════════════════════════════════════════════════

ML_PANEL_PATH = 'data/cleaned/ml_panel.csv'

if os.path.exists(ML_PANEL_PATH):
    ok(f"ml_panel.csv found — skipping data download")
    ok(f"(delete {ML_PANEL_PATH} to force re-download)")
    panel_ml = pd.read_csv(ML_PANEL_PATH, parse_dates=['date'])
    ok(f"Loaded: {panel_ml.shape[0]} obs x "
       f"{panel_ml.shape[1]-2} features")

else:
    warn("ml_panel.csv not found — downloading and building data now")
    warn("This replicates Step 4 internally")

    # ── A1. FRED ──────────────────────────────────────────────────────
    sep("A1. FRED API")
    fred_df = pd.DataFrame()

    if not FRED_KEY or FRED_KEY == 'paste_your_key_here':
        warn("FRED key not set — FRED features will be skipped")
        warn("Set FRED_KEY at the top of this script")
    else:
        try:
            from fredapi import Fred
            fred_obj = Fred(api_key=FRED_KEY)

            fred_series = {
                'SP500':            'sp500',
                'VIXCLS':           'vix',
                'CSUSHPISA':        'case_shiller_hpi',
                'DCOILWTICO':       'oil_wti',
                'GOLDAMGBD228NLBM': 'gold_price',
                'FEDFUNDS':         'fed_funds_rate',
                'T10Y2Y':           'term_spread_10y2y',
                'T10Y3M':           'term_spread_10y3m',
                'MORTGAGE30US':     'mortgage_rate_30y',
                'BAMLH0A0HYM2':     'hy_credit_spread',
                'BAMLC0A0CM':       'ig_credit_spread',
                'NFCI':             'nfci_financial_conditions',
                'STLFSI4':          'stl_financial_stress',
                'TOTALSL':          'consumer_credit_total',
                'BUSLOANS':         'commercial_loans',
                'REALLN':           'real_estate_loans',
                'DPCREDIT':         'domestic_credit_private',
                'DRSFRMACBS':       'delinquency_mortgage',
                'DRCCLACBS':        'delinquency_credit_card',
                'DRBLACBS':         'delinquency_business',
                'GFDEGDQ188S':      'federal_debt_pct_gdp',
                'GDPC1':            'real_gdp',
                'UNRATE':           'unemployment_us',
                'CPIAUCSL':         'cpi_us',
                'INDPRO':           'industrial_production',
                'HOUST':            'housing_starts',
            }

            fred_raw = {}
            for sid, name in fred_series.items():
                try:
                    fred_raw[name] = fred_obj.get_series(
                        sid, observation_start=START_DATE)
                    time.sleep(0.05)
                except Exception as ex:
                    warn(f"  FRED {sid}: {str(ex)[:45]}")

            for sid, name in [('CAPE',    'cape_shiller'),
                               ('SP500PE', 'pe_ratio'),
                               ('SP500DIV','div_yield')]:
                try:
                    fred_raw[name] = (
                        fred_obj.get_series(
                            sid, observation_start=START_DATE)
                        .resample('MS').last())
                    ok(f"  {name} ({sid})")
                except Exception:
                    warn(f"  {sid} not in FRED — skipping")

            fred_df = pd.DataFrame(fred_raw)
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df.index.name = 'date'
            fred_df = fred_df.sort_index()
            fred_df.to_csv('data/raw/fred_data.csv')
            ok(f"FRED: {fred_df.shape[0]} obs x "
               f"{fred_df.shape[1]} series")

        except Exception as e:
            err(f"FRED failed: {e}")

    # ── A2. Yahoo Finance ─────────────────────────────────────────────
    sep("A2. Yahoo Finance")
    yf_df = pd.DataFrame()
    try:
        import yfinance as yf

        all_tickers = {
            'SP500':   '^GSPC',   'FTSE100': '^FTSE',
            'DAX':     '^GDAXI',  'CAC40':   '^FCHI',
            'Nikkei':  '^N225',   'TSX':     '^GSPTSE',
            'ASX200':  '^AXJO',   'KOSPI':   '^KS11',
            'IBEX35':  '^IBEX',   'AEX':     '^AEX',
            'SMI':     '^SSMI',   'BOVESPA': '^BVSP',
            'SENSEX':  '^BSESN',  'HSI':     '^HSI',
            'NIFTY50': '^NSEI',   'STI':     '^STI',
            'US_10Y':  '^TNX',    'US_3M':   '^IRX',
            'US_30Y':  '^TYX',    'VIX_YF':  '^VIX',
        }

        yf_raw = yf.download(
            list(all_tickers.values()),
            start=START_DATE, interval='1mo',
            auto_adjust=True, progress=False, threads=True,
        )['Close']

        reverse = {v: k for k, v in all_tickers.items()}
        yf_raw.columns = [reverse.get(c, c) for c in yf_raw.columns]
        yf_raw.index = pd.to_datetime(yf_raw.index)
        yf_raw.index.name = 'date'
        yf_raw = yf_raw.sort_index()
        if 'US_10Y' in yf_raw.columns and 'US_3M' in yf_raw.columns:
            yf_raw['term_spread_yf'] = yf_raw['US_10Y'] - yf_raw['US_3M']
        yf_raw.to_csv('data/raw/yfinance_data.csv')
        yf_df = yf_raw
        ok(f"Yahoo Finance: {yf_df.shape[0]} months x "
           f"{yf_df.shape[1]} series")

    except Exception as e:
        err(f"Yahoo Finance failed: {e}")

    # ── B. Build feature panel ────────────────────────────────────────
    sep("B. Building feature panel")

    if 'SP500' in yf_df.columns:
        panel = yf_df[['SP500']].copy().rename(columns={'SP500': 'price'})
    elif not fred_df.empty and 'sp500' in fred_df.columns:
        panel = (fred_df[['sp500']].resample('MS').last()
                 .rename(columns={'sp500': 'price'}))
    else:
        raise RuntimeError("No SP500 price data — check FRED key or Yahoo Finance")

    # Category 1: Momentum and volatility
    panel['mom_1m']     = panel['price'].pct_change(1)
    panel['mom_3m']     = panel['price'].pct_change(3)
    panel['mom_6m']     = panel['price'].pct_change(6)
    panel['mom_12m']    = panel['price'].pct_change(12)
    panel['vol_3m']     = panel['price'].pct_change().rolling(3).std()
    panel['vol_6m']     = panel['price'].pct_change().rolling(6).std()
    panel['vol_12m']    = panel['price'].pct_change().rolling(12).std()
    panel['ret_excess'] = (panel['price'].pct_change()
                           - panel['price'].pct_change().rolling(60).mean())

    # Category 2: Valuation (CAPE z-score)
    if not fred_df.empty and 'cape_shiller' in fred_df.columns:
        panel['CAPE'] = (fred_df['cape_shiller']
                         .resample('MS').last()
                         .reindex(panel.index, method='ffill'))
        panel['CAPE_mean'] = panel['CAPE'].rolling(120).mean()
        panel['CAPE_std']  = panel['CAPE'].rolling(120).std()
        panel['CAPE_z']    = ((panel['CAPE'] - panel['CAPE_mean'])
                              / panel['CAPE_std'].replace(0, np.nan))
        ok("CAPE z-score from FRED")
    elif not fred_df.empty and 'pe_ratio' in fred_df.columns:
        panel['PE'] = (fred_df['pe_ratio']
                       .resample('MS').last()
                       .reindex(panel.index, method='ffill'))
        panel['PE_mean'] = panel['PE'].rolling(120).mean()
        panel['PE_std']  = panel['PE'].rolling(120).std()
        panel['CAPE_z']  = ((panel['PE'] - panel['PE_mean'])
                            / panel['PE_std'].replace(0, np.nan))
        warn("PE ratio used as CAPE proxy")

    if not fred_df.empty and 'div_yield' in fred_df.columns:
        panel['div_yield'] = (fred_df['div_yield']
                              .resample('MS').last()
                              .reindex(panel.index, method='ffill'))

    # Category 3: Credit aggregates (FRED)
    if not fred_df.empty:
        fred_m = fred_df.resample('MS').last()
        credit_cols = [
            'hy_credit_spread', 'ig_credit_spread',
            'nfci_financial_conditions', 'stl_financial_stress',
            'consumer_credit_total', 'commercial_loans',
            'real_estate_loans', 'domestic_credit_private',
            'federal_debt_pct_gdp', 'delinquency_mortgage',
            'delinquency_credit_card', 'delinquency_business',
            'fed_funds_rate', 'term_spread_10y2y', 'term_spread_10y3m',
            'mortgage_rate_30y', 'vix', 'case_shiller_hpi',
            'oil_wti', 'gold_price', 'real_gdp', 'unemployment_us',
            'cpi_us', 'industrial_production', 'housing_starts',
        ]
        for col in credit_cols:
            if col in fred_m.columns:
                panel[col] = fred_m[col].reindex(
                    panel.index, method='ffill')

    # Category 4: Yahoo Finance covariates (Astill et al. 2023, Gap 4)
    if not yf_df.empty:
        for col in ['VIX_YF', 'US_10Y', 'US_3M', 'US_30Y',
                    'term_spread_yf', 'FTSE100', 'DAX', 'Nikkei',
                    'BOVESPA', 'SENSEX', 'KOSPI', 'ASX200', 'HSI']:
            if col in yf_df.columns:
                panel[col] = yf_df[col].reindex(
                    panel.index, method='ffill')

    # Category 5: Derived macro variables
    if 'US_10Y' in panel.columns and 'cpi_us' in panel.columns:
        panel['real_rate_10y'] = (panel['US_10Y']
                                   - panel['cpi_us'].pct_change(12)*100)
    if ('hy_credit_spread' in panel.columns
            and 'ig_credit_spread' in panel.columns):
        panel['spread_diff'] = (panel['hy_credit_spread']
                                 - panel['ig_credit_spread'])
    if 'consumer_credit_total' in panel.columns:
        panel['credit_growth_12m'] = (
            panel['consumer_credit_total'].pct_change(12))
    if 'case_shiller_hpi' in panel.columns:
        panel['house_price_chg12'] = (
            panel['case_shiller_hpi'].pct_change(12))
    if 'real_gdp' in panel.columns:
        panel['gdp_growth_yoy'] = panel['real_gdp'].pct_change(4)*100

    # Lags: 3m / 6m / 12m (Gap 5 — temporal accumulation)
    base_feats = [c for c in panel.columns
                  if c not in ['price']
                  and panel[c].notna().mean() > 0.5]
    for col in base_feats:
        panel[f'{col}_lag3']  = panel[col].shift(3)
        panel[f'{col}_lag6']  = panel[col].shift(6)
        panel[f'{col}_lag12'] = panel[col].shift(12)

    panel = panel.replace([np.inf, -np.inf], np.nan).sort_index()

    # ── C. Target variable Y ──────────────────────────────────────────
    sep("C. Target variable Y")
    panel['Y'] = 0
    gsadf_used = False

    if os.path.exists('data/cleaned/bubble_dates.csv'):
        try:
            eps_df = pd.read_csv('data/cleaned/bubble_dates.csv',
                                  parse_dates=['inception','termination'])
            us_eps = eps_df[eps_df['country'] == 'US']
            if not us_eps.empty:
                for _, ep in us_eps.iterrows():
                    inc = pd.Timestamp(ep['inception'])
                    for lead in range(LEAD_MIN, LEAD_MAX + 1):
                        sig = inc - pd.DateOffset(months=lead)
                        idx = nearest_idx(panel.index, sig)
                        panel.iloc[idx, panel.columns.get_loc('Y')] = 1
                for _, ep in us_eps.iterrows():
                    for anchor in [pd.Timestamp(ep['inception']),
                                   pd.Timestamp(ep['termination'])]:
                        for off in range(-3, 4):
                            excl = anchor + pd.DateOffset(months=off)
                            idx  = nearest_idx(panel.index, excl)
                            if panel.iloc[idx]['Y'] == 0:
                                panel.iloc[
                                    idx, panel.columns.get_loc('Y')] = np.nan
                panel = panel.dropna(subset=['Y'])
                gsadf_used = True
                ok(f"Target from GSADF dates: "
                   f"{int(panel['Y'].sum())} positive months")
        except Exception as ex:
            warn(f"bubble_dates.csv error: {ex}")

    if not gsadf_used or int(panel['Y'].sum()) == 0:
        warn("Using known historical US bubble periods as fallback")
        panel['Y'] = 0
        for s, e, label in [
            ('1997-06-01','2000-03-01','Dot-com build-up'),
            ('2004-06-01','2007-06-01','Housing build-up'),
            ('2009-06-01','2010-06-01','Post-GFC rally'),
            ('2020-06-01','2021-09-01','Post-COVID exuberance'),
        ]:
            panel.loc[(panel.index >= s) & (panel.index <= e), 'Y'] = 1
            ok(f"  Y=1: {s} to {e} ({label})")

    ok(f"Final target: Y=1={int(panel['Y'].sum())} months "
       f"({panel['Y'].mean()*100:.1f}%)")

    # ── D. Feature matrix ─────────────────────────────────────────────
    feat_cols_all = [c for c in panel.columns
                     if c not in ['Y', 'price']
                     and panel[c].notna().mean() > 0.55]
    panel_ml = (panel[feat_cols_all + ['Y']]
                .dropna()
                .reset_index())
    if panel_ml.columns[0] != 'date':
        panel_ml = panel_ml.rename(
            columns={panel_ml.columns[0]: 'date'})

    panel_ml.to_csv(ML_PANEL_PATH, index=False)
    ok(f"ml_panel.csv saved: {panel_ml.shape[0]} obs x "
       f"{panel_ml.shape[1]-2} features")


# ════════════════════════════════════════════════════════════════════════
# SECTION E — RECURSIVE ML EVALUATION
# ════════════════════════════════════════════════════════════════════════
sep("E. Recursive ML evaluation")

feat_cols = [c for c in panel_ml.columns if c not in ['date', 'Y']]
X_all     = panel_ml[feat_cols]
y_all     = panel_ml['Y'].astype(int)
eval_idx  = [i for i in range(BURN_IN, len(panel_ml)) if i % 6 == 0]
sel_f     = feat_cols.copy()
preds     = []

ok(f"Panel:       {panel_ml.shape[0]} obs x {len(feat_cols)} features")
ok(f"Y=1 months:  {int(y_all.sum())} ({y_all.mean()*100:.1f}%)")
ok(f"Windows:     {len(eval_idx)} | Burn-in: {BURN_IN} months")
ok(f"Classifiers: LR_L2, LR_L1, RF, XGB, SVM, KNN")


# ── Performance metrics (Section 3.7.2) ──────────────────────────────
def relative_usefulness(yt, yp, mu=MU, thr=0.5):
    """Alessi & Detken (2011) Ur = 1 - [1/min(mu,1-mu)][mu*T2+(1-mu)*T1]"""
    pred = (yp >= thr).astype(int)
    tp = int(((pred==1)&(yt==1)).sum())
    fp = int(((pred==1)&(yt==0)).sum())
    tn = int(((pred==0)&(yt==0)).sum())
    fn = int(((pred==0)&(yt==1)).sum())
    T1 = fn/(tp+fn) if tp+fn > 0 else 1.
    T2 = fp/(fp+tn) if fp+tn > 0 else 1.
    return 1. - (1./min(mu, 1.-mu)) * (mu*T2 + (1.-mu)*T1)

def hr_at_fpr(yt, ys, fpr_tgt=0.10):
    fpr, tpr, _ = roc_curve(yt, ys)
    return float(tpr[min(np.searchsorted(fpr, fpr_tgt), len(tpr)-1)])

def score(name, yt, ys):
    if len(np.unique(yt)) < 2:
        return {'model':name,'auroc':np.nan,'brier':np.nan,
                'ur':np.nan,'hr10':np.nan}
    return {
        'model': name,
        'auroc': float(roc_auc_score(yt, ys)),
        'brier': float(brier_score_loss(yt, ys)),
        'ur':    float(relative_usefulness(yt, ys)),
        'hr10':  float(hr_at_fpr(yt, ys)),
    }


# ── Three-stage feature selection (Section 3.5.3) ────────────────────
def select_features(Xtr, ytr):
    # Stage A: Boruta or RF importance fallback
    try:
        from boruta import BorutaPy
        rf_b = RandomForestClassifier(n_estimators=100,
                                       random_state=RANDOM_STATE, n_jobs=-1)
        sel  = BorutaPy(rf_b, n_estimators='auto', max_iter=30,
                         random_state=RANDOM_STATE, verbose=0)
        sel.fit(Xtr.values, ytr.values.ravel())
        confirmed = Xtr.columns[sel.support_].tolist()
        if not confirmed:
            confirmed = Xtr.columns.tolist()
    except Exception:
        rf_t = RandomForestClassifier(n_estimators=100,
                                       random_state=RANDOM_STATE, n_jobs=-1)
        rf_t.fit(Xtr, ytr)
        imp = rf_t.feature_importances_
        confirmed = Xtr.columns[imp >= imp.mean()].tolist()
    Xa = Xtr[confirmed]

    # Stage B: correlation screening
    corr  = Xa.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop  = set()
    for c in upper.columns:
        drop.update(upper[c][upper[c] > CORR_THRESH].index.tolist())
    Xb = Xa.drop(columns=list(drop), errors='ignore')

    # Stage C: LASSO parsimony filter
    if Xb.shape[1] > 3:
        try:
            sc = StandardScaler()
            Xs = sc.fit_transform(Xb)
            lr = LogisticRegression(penalty='l1', solver='liblinear',
                                     C=0.05, max_iter=500,
                                     random_state=RANDOM_STATE)
            lr.fit(Xs, ytr)
            nz = Xb.columns[np.abs(lr.coef_[0]) > 0].tolist()
            Xc = Xb[nz] if nz else Xb
        except Exception:
            Xc = Xb
    else:
        Xc = Xb

    return (Xc.columns.tolist()
            if len(Xc.columns) >= 2
            else Xb.columns.tolist())


# ── Classifiers (Table 3.1, 6 of 8) ─────────────────────────────────
def make_classifiers():
    return {
        'LR_L2': LogisticRegression(C=1., max_iter=500,
                                     random_state=RANDOM_STATE),
        'LR_L1': LogisticRegression(penalty='l1', solver='liblinear',
                                     C=0.1, max_iter=500,
                                     random_state=RANDOM_STATE),
        'RF':    RandomForestClassifier(n_estimators=300,
                                         max_features='sqrt',
                                         min_samples_leaf=5,
                                         random_state=RANDOM_STATE,
                                         n_jobs=-1),
        'XGB':   xgb.XGBClassifier(learning_rate=0.05, max_depth=4,
                                     n_estimators=200, subsample=0.8,
                                     random_state=RANDOM_STATE,
                                     eval_metric='logloss', verbosity=0),
        'SVM':   SVC(kernel='rbf', C=10, gamma='scale',
                      probability=True, random_state=RANDOM_STATE),
        'KNN':   KNeighborsClassifier(n_neighbors=5),
    }


# ── Recursive expanding-window loop (Section 3.7.1) ──────────────────
print(f"\n  Starting recursive evaluation...")
for w, sp in enumerate(eval_idx):
    Xtr = X_all.iloc[:sp]
    ytr = y_all.iloc[:sp]
    Xte = X_all.iloc[[sp]]
    yte = y_all.iloc[[sp]]

    if ytr.sum() < 5:
        continue

    # Refresh feature selection every 12 windows
    if w % 12 == 0:
        sel_f = select_features(Xtr, ytr)

    Xtrs   = Xtr[sel_f]
    Xtes   = Xte[sel_f]
    sc     = StandardScaler()
    Xtr_sc = sc.fit_transform(Xtrs)
    Xte_sc = sc.transform(Xtes)

    # SMOTE (Section 3.5.4)
    try:
        sm = SMOTE(sampling_strategy=SMOTE_RATIO,
                   random_state=RANDOM_STATE, k_neighbors=3)
        Xtr_sm, ytr_sm = sm.fit_resample(Xtr_sc, ytr)
    except Exception:
        Xtr_sm, ytr_sm = Xtr_sc, ytr.values

    row = {
        'date':   panel_ml.loc[sp, 'date'],
        'y_true': int(yte.iloc[0]),
        'n_train':sp,
        'n_pos':  int(ytr.sum()),
    }

    for nm, clf in make_classifiers().items():
        try:
            clf.fit(Xtr_sm, ytr_sm)
            row[f'p_{nm}'] = float(
                clf.predict_proba(Xte_sc)[0, 1])
        except Exception:
            row[f'p_{nm}'] = np.nan

    probs = [v for k, v in row.items()
             if k.startswith('p_') and not np.isnan(v)]
    row['p_Ensemble_unweighted'] = (
        float(np.mean(probs)) if probs else np.nan)

    preds.append(row)
    if w % 20 == 0:
        print(f"    Window {w+1}/{len(eval_idx)}: "
              f"train={sp} pos={ytr.sum()}")

pred_df = pd.DataFrame(preds).set_index('date')
pred_df.to_csv('data/cleaned/predictions_ml.csv')
ok(f"predictions_ml.csv: {len(pred_df)} windows")


# ════════════════════════════════════════════════════════════════════════
# SECTION F — PERFORMANCE METRICS + PLOTS
# ════════════════════════════════════════════════════════════════════════
sep("F. Performance metrics")

prob_cols = [c for c in pred_df.columns if c.startswith('p_')]
perf_rows = []
for col in prob_cols:
    v = pred_df[col].notna()
    if v.sum() < 10:
        continue
    yt = pred_df.loc[v, 'y_true'].values
    ys = pred_df.loc[v, col].values
    if len(np.unique(yt)) < 2:
        continue
    perf_rows.append(score(col.replace('p_', ''), yt, ys))

perf_df = pd.DataFrame(perf_rows).sort_values('auroc', ascending=False)
perf_df.to_csv('results/model_performance.csv', index=False)

print(f"\n  {'Model':<25} {'AUROC':>7} {'Ur':>7} "
      f"{'Brier':>7} {'HR@10%':>8}")
print(f"  {'-'*57}")
for _, r in perf_df.iterrows():
    print(f"  {r['model']:<25} {r['auroc']:>7.3f} "
          f"{r['ur']:>7.3f} {r['brier']:>7.3f} "
          f"{r['hr10']:>8.3f}")

# Ur sensitivity (Table 3.2 robustness)
if len(perf_df) > 0:
    bc = f"p_{perf_df.iloc[0]['model']}"
    if bc in pred_df.columns:
        v  = pred_df[bc].notna()
        yt = pred_df.loc[v, 'y_true'].values
        ys = pred_df.loc[v, bc].values
        if len(np.unique(yt)) > 1:
            ur_rows = [
                {'mu': mu, 'Ur': relative_usefulness(yt, ys, mu)}
                for mu in [.3, .4, .5, .6, .7]
            ]
            pd.DataFrame(ur_rows).to_csv(
                'results/ur_sensitivity.csv', index=False)
            print(f"\n  Ur sensitivity — best model: "
                  f"{perf_df.iloc[0]['model']}")
            for r in ur_rows:
                print(f"    mu={r['mu']:.1f}  Ur={r['Ur']:.3f}")

# ── Plots ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
for col in prob_cols:
    v  = pred_df[col].notna()
    yt = pred_df.loc[v, 'y_true'].values
    ys = pred_df.loc[v, col].values
    if len(np.unique(yt)) < 2:
        continue
    fpr, tpr, _ = roc_curve(yt, ys)
    ax.plot(fpr, tpr, lw=1.5,
            label=f"{col.replace('p_','')} "
                  f"({roc_auc_score(yt,ys):.3f})")
ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax.axvline(0.10, color='gray', ls=':', lw=1, label='10% FPR')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Recursive Real-Time Evaluation\n'
             'Y = GSADF bubble episodes (Gap 2)',
             fontsize=10, fontweight='bold')
ax.legend(fontsize=7, loc='lower right')
ax.grid(True, alpha=0.3)

ax2 = axes[1]
if len(perf_df) > 0:
    clrs = ['#2980b9' if u >= 0 else '#c0392b'
            for u in perf_df['ur'].fillna(0)]
    bars = ax2.bar(perf_df['model'], perf_df['ur'].fillna(0),
                   color=clrs, alpha=0.85)
    ax2.axhline(0, color='black', lw=1)
    ax2.set_title(f'Relative Usefulness Ur (mu={MU})\n'
                  f'Alessi & Detken (2011)', fontweight='bold')
    ax2.set_ylabel('Ur')
    plt.xticks(rotation=30, ha='right')
    ax2.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, perf_df['ur'].fillna(0)):
        ax2.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + .01, f'{val:.3f}',
                 ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('figures/stage2_ml_results.png',
            dpi=150, bbox_inches='tight')
plt.close()
ok("figures/stage2_ml_results.png saved")

print(f"""
  ════════════════════════════════════════════════════
  STEP 5 OUTPUTS
  ════════════════════════════════════════════════════
  data/cleaned/ml_panel.csv          feature panel
  data/cleaned/predictions_ml.csv    out-of-sample predictions
  results/model_performance.csv      AUROC / Ur / Brier / HR@10%
  results/ur_sensitivity.csv         mu robustness table
  figures/stage2_ml_results.png      ROC + Ur bar chart
  ════════════════════════════════════════════════════
""")
print("Step 5 complete.")
print("Next: run Step 6 (step6_lstm_transformer.py)")


  STEP 5 — RECURSIVE ML EVALUATION (self-contained)
  !!  ml_panel.csv not found — downloading and building data now
  !!  This replicates Step 4 internally

--- A1. FRED API ---
  !!  FRED key not set — FRED features will be skipped
  !!  Set FRED_KEY at the top of this script

--- A2. Yahoo Finance ---
  OK  Yahoo Finance: 497 months x 21 series

--- B. Building feature panel ---

--- C. Target variable Y ---
  !!  Using known historical US bubble periods as fallback
  OK    Y=1: 1997-06-01 to 2000-03-01 (Dot-com build-up)
  OK    Y=1: 2004-06-01 to 2007-06-01 (Housing build-up)
  OK    Y=1: 2009-06-01 to 2010-06-01 (Post-GFC rally)
  OK    Y=1: 2020-06-01 to 2021-09-01 (Post-COVID exuberance)
  OK  Final target: Y=1=100 months (20.1%)
  OK  ml_panel.csv saved: 311 obs x 84 features

--- E. Recursive ML evaluation ---
  OK  Panel:       311 obs x 84 features
  OK  Y=1 months:  87 (28.0%)
  OK  Windows:     42 | Burn-in: 60 months
  OK  Classifiers: LR_L2, LR_L1, RF, XGB, SVM, KNN

  

In [1]:
# ════════════════════════════════════════════════════════════════════════
# STEP 6 — LSTM-MLP and Transformer-MLP
# SELF-CONTAINED: downloads all data internally
# No prior steps required — just set your FRED key below
#
# Implements:
#   Section 3.5.2 — 8-quarter sequence windows
#   Section 3.5.5 — Table 3.1 LSTM-MLP and Transformer-MLP architectures
#   Gap 5         — Temporal memory in bubble prediction
# ════════════════════════════════════════════════════════════════════════
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve
warnings.filterwarnings('ignore')

# ── SET YOUR FRED KEY HERE ────────────────────────────────────────────
FRED_KEY = os.environ.get('FRED_API_KEY', 'paste_your_key_here')

# ── Config ────────────────────────────────────────────────────────────
START_DATE   = '1980-01-01'
LEAD_MIN     = 4
LEAD_MAX     = 12
RANDOM_STATE = 42
SEQ_LEN      = 8       # 8-quarter sequence window (Section 3.5.2)
MU           = 0.5     # Alessi-Detken loss ratio

for d in ['data/raw', 'data/cleaned', 'figures', 'results']:
    os.makedirs(d, exist_ok=True)

def ok(m):   print(f"  OK  {m}")
def warn(m): print(f"  !!  {m}")
def err(m):  print(f"  XX  {m}")
def sep(t):  print(f"\n--- {t} ---")

def nearest_idx(index, target_date):
    """Nearest DatetimeIndex position — pandas 2.x compatible."""
    return int(np.abs((index - target_date).total_seconds()).argmin())

print("=" * 60)
print("  STEP 6 — LSTM + TRANSFORMER (self-contained)")
print("=" * 60)


# ════════════════════════════════════════════════════════════════════════
# SECTION A — DATA (load if exists, else download and build)
# ════════════════════════════════════════════════════════════════════════

ML_PANEL_PATH = 'data/cleaned/ml_panel.csv'

if os.path.exists(ML_PANEL_PATH):
    ok(f"ml_panel.csv found — skipping data download")
    panel_ml = pd.read_csv(ML_PANEL_PATH, parse_dates=['date'])
    ok(f"Loaded: {panel_ml.shape[0]} obs x "
       f"{panel_ml.shape[1]-2} features")

else:
    warn("ml_panel.csv not found — downloading and building now")

    # ── A1. FRED ──────────────────────────────────────────────────────
    sep("A1. FRED API")
    fred_df = pd.DataFrame()

    if not FRED_KEY or FRED_KEY == 'paste_your_key_here':
        warn("FRED key not set — FRED features will be skipped")
        warn("Set FRED_KEY at the top of this script")
    else:
        try:
            from fredapi import Fred
            fred_obj = Fred(api_key=FRED_KEY)

            fred_series = {
                'SP500':            'sp500',
                'VIXCLS':           'vix',
                'CSUSHPISA':        'case_shiller_hpi',
                'DCOILWTICO':       'oil_wti',
                'GOLDAMGBD228NLBM': 'gold_price',
                'FEDFUNDS':         'fed_funds_rate',
                'T10Y2Y':           'term_spread_10y2y',
                'T10Y3M':           'term_spread_10y3m',
                'MORTGAGE30US':     'mortgage_rate_30y',
                'BAMLH0A0HYM2':     'hy_credit_spread',
                'BAMLC0A0CM':       'ig_credit_spread',
                'NFCI':             'nfci_financial_conditions',
                'STLFSI4':          'stl_financial_stress',
                'TOTALSL':          'consumer_credit_total',
                'BUSLOANS':         'commercial_loans',
                'REALLN':           'real_estate_loans',
                'DPCREDIT':         'domestic_credit_private',
                'DRSFRMACBS':       'delinquency_mortgage',
                'DRCCLACBS':        'delinquency_credit_card',
                'DRBLACBS':         'delinquency_business',
                'GFDEGDQ188S':      'federal_debt_pct_gdp',
                'GDPC1':            'real_gdp',
                'UNRATE':           'unemployment_us',
                'CPIAUCSL':         'cpi_us',
                'INDPRO':           'industrial_production',
                'HOUST':            'housing_starts',
            }

            fred_raw = {}
            for sid, name in fred_series.items():
                try:
                    fred_raw[name] = fred_obj.get_series(
                        sid, observation_start=START_DATE)
                    time.sleep(0.05)
                except Exception as ex:
                    warn(f"  FRED {sid}: {str(ex)[:45]}")

            for sid, name in [('CAPE',    'cape_shiller'),
                               ('SP500PE', 'pe_ratio'),
                               ('SP500DIV','div_yield')]:
                try:
                    fred_raw[name] = (
                        fred_obj.get_series(
                            sid, observation_start=START_DATE)
                        .resample('MS').last())
                    ok(f"  {name} ({sid})")
                except Exception:
                    warn(f"  {sid} not in FRED — skipping")

            fred_df = pd.DataFrame(fred_raw)
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df.index.name = 'date'
            fred_df = fred_df.sort_index()
            fred_df.to_csv('data/raw/fred_data.csv')
            ok(f"FRED: {fred_df.shape[0]} obs x {fred_df.shape[1]} series")

        except Exception as e:
            err(f"FRED failed: {e}")

    # ── A2. Yahoo Finance ─────────────────────────────────────────────
    sep("A2. Yahoo Finance")
    yf_df = pd.DataFrame()
    try:
        import yfinance as yf

        all_tickers = {
            'SP500':   '^GSPC',   'FTSE100': '^FTSE',
            'DAX':     '^GDAXI',  'CAC40':   '^FCHI',
            'Nikkei':  '^N225',   'TSX':     '^GSPTSE',
            'ASX200':  '^AXJO',   'KOSPI':   '^KS11',
            'IBEX35':  '^IBEX',   'AEX':     '^AEX',
            'SMI':     '^SSMI',   'BOVESPA': '^BVSP',
            'SENSEX':  '^BSESN',  'HSI':     '^HSI',
            'NIFTY50': '^NSEI',   'STI':     '^STI',
            'US_10Y':  '^TNX',    'US_3M':   '^IRX',
            'US_30Y':  '^TYX',    'VIX_YF':  '^VIX',
        }

        yf_raw = yf.download(
            list(all_tickers.values()),
            start=START_DATE, interval='1mo',
            auto_adjust=True, progress=False, threads=True,
        )['Close']

        reverse = {v: k for k, v in all_tickers.items()}
        yf_raw.columns = [reverse.get(c, c) for c in yf_raw.columns]
        yf_raw.index = pd.to_datetime(yf_raw.index)
        yf_raw.index.name = 'date'
        yf_raw = yf_raw.sort_index()
        if 'US_10Y' in yf_raw.columns and 'US_3M' in yf_raw.columns:
            yf_raw['term_spread_yf'] = yf_raw['US_10Y'] - yf_raw['US_3M']
        yf_raw.to_csv('data/raw/yfinance_data.csv')
        yf_df = yf_raw
        ok(f"Yahoo Finance: {yf_df.shape[0]} months x "
           f"{yf_df.shape[1]} series")

    except Exception as e:
        err(f"Yahoo Finance failed: {e}")

    # ── B. Build feature panel ────────────────────────────────────────
    sep("B. Feature panel")

    if 'SP500' in yf_df.columns:
        panel = yf_df[['SP500']].copy().rename(columns={'SP500': 'price'})
    elif not fred_df.empty and 'sp500' in fred_df.columns:
        panel = (fred_df[['sp500']].resample('MS').last()
                 .rename(columns={'sp500': 'price'}))
    else:
        raise RuntimeError(
            "No SP500 price data. Check FRED key or Yahoo Finance.")

    # Category 1: Momentum and volatility
    panel['mom_1m']     = panel['price'].pct_change(1)
    panel['mom_3m']     = panel['price'].pct_change(3)
    panel['mom_6m']     = panel['price'].pct_change(6)
    panel['mom_12m']    = panel['price'].pct_change(12)
    panel['vol_3m']     = panel['price'].pct_change().rolling(3).std()
    panel['vol_6m']     = panel['price'].pct_change().rolling(6).std()
    panel['vol_12m']    = panel['price'].pct_change().rolling(12).std()
    panel['ret_excess'] = (panel['price'].pct_change()
                           - panel['price'].pct_change().rolling(60).mean())

    # Category 2: Valuation
    if not fred_df.empty and 'cape_shiller' in fred_df.columns:
        panel['CAPE'] = (fred_df['cape_shiller'].resample('MS').last()
                         .reindex(panel.index, method='ffill'))
        panel['CAPE_mean'] = panel['CAPE'].rolling(120).mean()
        panel['CAPE_std']  = panel['CAPE'].rolling(120).std()
        panel['CAPE_z']    = ((panel['CAPE'] - panel['CAPE_mean'])
                              / panel['CAPE_std'].replace(0, np.nan))
    elif not fred_df.empty and 'pe_ratio' in fred_df.columns:
        panel['PE'] = (fred_df['pe_ratio'].resample('MS').last()
                       .reindex(panel.index, method='ffill'))
        panel['PE_mean'] = panel['PE'].rolling(120).mean()
        panel['PE_std']  = panel['PE'].rolling(120).std()
        panel['CAPE_z']  = ((panel['PE'] - panel['PE_mean'])
                            / panel['PE_std'].replace(0, np.nan))

    if not fred_df.empty and 'div_yield' in fred_df.columns:
        panel['div_yield'] = (fred_df['div_yield'].resample('MS').last()
                              .reindex(panel.index, method='ffill'))

    # Category 3: Credit aggregates (FRED)
    if not fred_df.empty:
        fred_m = fred_df.resample('MS').last()
        for col in [
            'hy_credit_spread', 'ig_credit_spread',
            'nfci_financial_conditions', 'stl_financial_stress',
            'consumer_credit_total', 'commercial_loans',
            'real_estate_loans', 'domestic_credit_private',
            'federal_debt_pct_gdp', 'delinquency_mortgage',
            'delinquency_credit_card', 'delinquency_business',
            'fed_funds_rate', 'term_spread_10y2y', 'term_spread_10y3m',
            'mortgage_rate_30y', 'vix', 'case_shiller_hpi',
            'oil_wti', 'gold_price', 'real_gdp', 'unemployment_us',
            'cpi_us', 'industrial_production', 'housing_starts',
        ]:
            if col in fred_m.columns:
                panel[col] = fred_m[col].reindex(
                    panel.index, method='ffill')

    # Category 4: Yahoo Finance covariates
    if not yf_df.empty:
        for col in ['VIX_YF','US_10Y','US_3M','US_30Y',
                    'term_spread_yf','FTSE100','DAX','Nikkei',
                    'BOVESPA','SENSEX','KOSPI','ASX200','HSI']:
            if col in yf_df.columns:
                panel[col] = yf_df[col].reindex(
                    panel.index, method='ffill')

    # Category 5: Derived variables
    if 'US_10Y' in panel.columns and 'cpi_us' in panel.columns:
        panel['real_rate_10y'] = (panel['US_10Y']
                                   - panel['cpi_us'].pct_change(12)*100)
    if ('hy_credit_spread' in panel.columns
            and 'ig_credit_spread' in panel.columns):
        panel['spread_diff'] = (panel['hy_credit_spread']
                                 - panel['ig_credit_spread'])
    if 'consumer_credit_total' in panel.columns:
        panel['credit_growth_12m'] = (
            panel['consumer_credit_total'].pct_change(12))
    if 'case_shiller_hpi' in panel.columns:
        panel['house_price_chg12'] = (
            panel['case_shiller_hpi'].pct_change(12))
    if 'real_gdp' in panel.columns:
        panel['gdp_growth_yoy'] = panel['real_gdp'].pct_change(4)*100

    # Lags 3m / 6m / 12m (Gap 5)
    base_feats = [c for c in panel.columns
                  if c not in ['price']
                  and panel[c].notna().mean() > 0.5]
    for col in base_feats:
        panel[f'{col}_lag3']  = panel[col].shift(3)
        panel[f'{col}_lag6']  = panel[col].shift(6)
        panel[f'{col}_lag12'] = panel[col].shift(12)

    panel = panel.replace([np.inf, -np.inf], np.nan).sort_index()

    # ── C. Target variable Y ──────────────────────────────────────────
    sep("C. Target variable Y")
    panel['Y'] = 0
    gsadf_used = False

    if os.path.exists('data/cleaned/bubble_dates.csv'):
        try:
            eps_df = pd.read_csv(
                'data/cleaned/bubble_dates.csv',
                parse_dates=['inception', 'termination'])
            us_eps = eps_df[eps_df['country'] == 'US']
            if not us_eps.empty:
                for _, ep in us_eps.iterrows():
                    inc = pd.Timestamp(ep['inception'])
                    for lead in range(LEAD_MIN, LEAD_MAX + 1):
                        sig = inc - pd.DateOffset(months=lead)
                        idx = nearest_idx(panel.index, sig)
                        panel.iloc[idx, panel.columns.get_loc('Y')] = 1
                for _, ep in us_eps.iterrows():
                    for anchor in [pd.Timestamp(ep['inception']),
                                   pd.Timestamp(ep['termination'])]:
                        for off in range(-3, 4):
                            excl = anchor + pd.DateOffset(months=off)
                            idx  = nearest_idx(panel.index, excl)
                            if panel.iloc[idx]['Y'] == 0:
                                panel.iloc[
                                    idx, panel.columns.get_loc('Y')] = np.nan
                panel = panel.dropna(subset=['Y'])
                gsadf_used = True
                ok(f"GSADF target: {int(panel['Y'].sum())} positive months")
        except Exception as ex:
            warn(f"bubble_dates.csv error: {ex}")

    if not gsadf_used or int(panel['Y'].sum()) == 0:
        warn("Using known historical US bubble periods as fallback")
        panel['Y'] = 0
        for s, e, label in [
            ('1997-06-01','2000-03-01','Dot-com build-up'),
            ('2004-06-01','2007-06-01','Housing build-up'),
            ('2009-06-01','2010-06-01','Post-GFC rally'),
            ('2020-06-01','2021-09-01','Post-COVID exuberance'),
        ]:
            panel.loc[(panel.index >= s) & (panel.index <= e), 'Y'] = 1
            ok(f"  Y=1: {s} to {e} ({label})")

    ok(f"Target: Y=1={int(panel['Y'].sum())} months "
       f"({panel['Y'].mean()*100:.1f}%)")

    # ── D. Save feature matrix ─────────────────────────────────────────
    feat_cols_all = [c for c in panel.columns
                     if c not in ['Y', 'price']
                     and panel[c].notna().mean() > 0.55]
    panel_ml = (panel[feat_cols_all + ['Y']]
                .dropna()
                .reset_index())
    if panel_ml.columns[0] != 'date':
        panel_ml = panel_ml.rename(
            columns={panel_ml.columns[0]: 'date'})
    panel_ml.to_csv(ML_PANEL_PATH, index=False)
    ok(f"ml_panel.csv saved: {panel_ml.shape[0]} obs x "
       f"{panel_ml.shape[1]-2} features")


# ════════════════════════════════════════════════════════════════════════
# SECTION E — PYTORCH CHECK
# ════════════════════════════════════════════════════════════════════════
sep("E. PyTorch check")

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    TORCH_OK = True
    ok(f"PyTorch {torch.__version__} available")
except ImportError:
    TORCH_OK = False
    warn("PyTorch not installed.")
    warn("Install with:  pip install torch")
    warn("LSTM/Transformer will be skipped.")
    warn("All Step 5 results (RF, XGB, LR, SVM, KNN) remain valid.")


# ════════════════════════════════════════════════════════════════════════
# SECTION F — BUILD SEQUENCE TENSORS
# ════════════════════════════════════════════════════════════════════════

feat_cols = [c for c in panel_ml.columns if c not in ['date', 'Y']]

if TORCH_OK:
    sep("F. Building sequence tensors")

    torch.manual_seed(RANDOM_STATE)
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ok(f"Device: {DEVICE}")

    X_raw = panel_ml[feat_cols].values.astype(np.float32)
    y_raw = panel_ml['Y'].values.astype(np.float32)

    # Scale features
    sc   = StandardScaler()
    X_sc = sc.fit_transform(X_raw)

    # Build sliding windows of length SEQ_LEN (8 quarters = 2 years)
    Xseq, yseq = [], []
    for i in range(SEQ_LEN, len(X_sc)):
        Xseq.append(X_sc[i - SEQ_LEN:i])
        yseq.append(y_raw[i])

    Xseq = np.array(Xseq, dtype=np.float32)
    yseq = np.array(yseq, dtype=np.float32)
    ok(f"Sequence tensor: {Xseq.shape}  "
       f"(samples={Xseq.shape[0]}, "
       f"seq_len={SEQ_LEN}Q, features={Xseq.shape[2]})")

    # Time-ordered 70/30 split — no shuffling (Section 3.7.1)
    n     = len(Xseq)
    split = int(n * 0.70)
    Xtr, Xte = Xseq[:split], Xseq[split:]
    ytr, yte  = yseq[:split], yseq[split:]

    Xtr_t = torch.tensor(Xtr)
    Xte_t = torch.tensor(Xte)
    ytr_t = torch.tensor(ytr)
    K     = Xtr.shape[2]    # number of features

    ok(f"Train: {split} samples  |  Test: {n-split} samples")
    ok(f"Train Y=1: {int(ytr.sum())}  |  Test Y=1: {int(yte.sum())}")


# ════════════════════════════════════════════════════════════════════════
# SECTION G — MODEL ARCHITECTURES (Section 3.5.5)
# ════════════════════════════════════════════════════════════════════════

    sep("G. Model architectures (Section 3.5.5)")

    class LSTMModel(nn.Module):
        """
        LSTM-MLP: bidirectional LSTM encoder + 3-layer MLP decoder.
        Hidden size = 50, dropout = 0.3 (Section 3.5.5 Table 3.1).
        """
        def __init__(self, input_size, hidden=50, dropout=0.3):
            super().__init__()
            self.lstm = nn.LSTM(
                input_size, hidden, batch_first=True,
                dropout=dropout if hidden > 1 else 0)
            self.mlp = nn.Sequential(
                nn.Linear(hidden, 32), nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(32, 16),    nn.ReLU(),
                nn.Linear(16, 1),     nn.Sigmoid())

        def forward(self, x):
            out, _ = self.lstm(x)
            return self.mlp(out[:, -1, :]).squeeze(1)

    class TransformerModel(nn.Module):
        """
        Transformer-MLP: linear projection + 2-layer Transformer encoder
        + MLP decoder. d_model=64, nhead=4 (Section 3.5.5 Table 3.1).
        """
        def __init__(self, input_size, nhead=4, d_model=64, dropout=0.3):
            super().__init__()
            self.proj = nn.Linear(input_size, d_model)
            enc_layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead,
                dropout=dropout, batch_first=True)
            self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)
            self.mlp = nn.Sequential(
                nn.Linear(d_model, 32), nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(32, 1), nn.Sigmoid())

        def forward(self, x):
            x = self.proj(x)
            x = self.encoder(x)
            return self.mlp(x[:, -1, :]).squeeze(1)

    ok("LSTMModel defined  (hidden=50, dropout=0.3)")
    ok("TransformerModel defined  (d_model=64, nhead=4, dropout=0.3)")


# ════════════════════════════════════════════════════════════════════════
# SECTION H — TRAINING LOOP WITH EARLY STOPPING
# ════════════════════════════════════════════════════════════════════════

    def train_model(model, Xtr_t, ytr_t,
                    epochs=50, patience=10, lr=1e-3):
        """
        Adam optimiser + BCE loss.
        Early stopping: patience=10 epochs on training loss.
        Batch size = 32, no shuffling (time-ordered).
        """
        opt     = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.BCELoss()
        ds = TensorDataset(Xtr_t, ytr_t)
        dl = DataLoader(ds, batch_size=32, shuffle=False)

        best_loss = np.inf
        pat       = 0

        for ep in range(epochs):
            model.train()
            ep_loss = 0.0
            for Xb, yb in dl:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad()
                pred = model(Xb)
                loss = loss_fn(pred, yb)
                loss.backward()
                opt.step()
                ep_loss += loss.item()

            if ep_loss < best_loss:
                best_loss = ep_loss
                pat = 0
            else:
                pat += 1
                if pat >= patience:
                    ok(f"  Early stop at epoch {ep+1} "
                       f"(best loss={best_loss:.4f})")
                    break

        return model


# ════════════════════════════════════════════════════════════════════════
# SECTION I — TRAIN AND EVALUATE
# ════════════════════════════════════════════════════════════════════════

    sep("I. Train and evaluate")

    architectures = [
        (
            'LSTM_MLP',
            LSTMModel,
            {'input_size': K, 'hidden': 50, 'dropout': 0.3},
        ),
        (
            'Transformer',
            TransformerModel,
            {
                'input_size': K,
                'nhead':      max(1, min(4, K // 16)) if K >= 16 else 1,
                'd_model':    max(64, ((K // 8) + 1) * 8),
                'dropout':    0.3,
            },
        ),
    ]

    deep_results = []

    for arch_name, ModelClass, kwargs in architectures:
        print(f"\n  Training {arch_name}  "
              f"(kwargs: {kwargs})...")
        try:
            model = ModelClass(**kwargs).to(DEVICE)
            model = train_model(
                model, Xtr_t, ytr_t, epochs=50, patience=10)

            model.eval()
            with torch.no_grad():
                proba = model(Xte_t.to(DEVICE)).cpu().numpy()

            yt_np = yte.astype(int)

            if len(np.unique(yt_np)) < 2:
                warn(f"{arch_name}: test set has only one class "
                     f"— metrics cannot be computed")
                warn("Try increasing LEAD_MAX or checking your "
                     "bubble episodes coverage")
                continue

            # Compute all four metrics (Section 3.7.2)
            auroc = float(roc_auc_score(yt_np, proba))
            brier = float(brier_score_loss(yt_np, proba))
            fpr, tpr, _ = roc_curve(yt_np, proba)
            hr10  = float(tpr[min(
                np.searchsorted(fpr, 0.10), len(tpr) - 1)])

            pred_b = (proba >= 0.5).astype(int)
            tp = int(((pred_b==1) & (yt_np==1)).sum())
            fp = int(((pred_b==1) & (yt_np==0)).sum())
            tn = int(((pred_b==0) & (yt_np==0)).sum())
            fn = int(((pred_b==0) & (yt_np==1)).sum())
            T1  = fn / (tp + fn + 1e-9)
            T2  = fp / (fp + tn + 1e-9)
            urn = 1. - (1./min(MU, 1.-MU)) * (MU*T2 + (1.-MU)*T1)

            deep_results.append({
                'model': arch_name,
                'auroc': auroc,
                'brier': brier,
                'ur':    urn,
                'hr10':  hr10,
            })

            ok(f"{arch_name}:")
            ok(f"  AUROC    = {auroc:.4f}")
            ok(f"  Ur       = {urn:.4f}")
            ok(f"  Brier    = {brier:.4f}")
            ok(f"  HR@10%   = {hr10:.4f}")

            # Save out-of-sample predictions
            n_pred = len(proba)
            pred_dates = panel_ml['date'].iloc[
                SEQ_LEN + split: SEQ_LEN + split + n_pred]
            pred_out = pd.DataFrame(
                {f'p_{arch_name}': proba},
                index=pred_dates.values)
            pred_out.index.name = 'date'
            save_path = (f'data/cleaned/predictions_'
                         f'{arch_name.lower()}.csv')
            pred_out.to_csv(save_path)
            ok(f"  Saved: {save_path}")

        except Exception as ex:
            err(f"{arch_name}: {ex}")
            import traceback
            traceback.print_exc()


# ════════════════════════════════════════════════════════════════════════
# SECTION J — RESULTS SUMMARY
# ════════════════════════════════════════════════════════════════════════

    sep("J. Results summary")

    if deep_results:
        deep_perf = pd.DataFrame(deep_results)
        deep_perf.to_csv('results/deep_model_performance.csv',
                         index=False)
        ok("results/deep_model_performance.csv saved")

        print(f"\n  {'Model':<18} {'AUROC':>7} {'Ur':>7} "
              f"{'Brier':>7} {'HR@10%':>8}")
        print(f"  {'-'*50}")
        for _, r in deep_perf.iterrows():
            print(f"  {r['model']:<18} {r['auroc']:>7.3f} "
                  f"{r['ur']:>7.3f} {r['brier']:>7.3f} "
                  f"{r['hr10']:>8.3f}")

        # Merge with Step 5 results if available
        if os.path.exists('results/model_performance.csv'):
            step5_perf = pd.read_csv('results/model_performance.csv')
            combined = pd.concat(
                [step5_perf, deep_perf], ignore_index=True)
            combined = combined.sort_values('auroc', ascending=False)
            combined.to_csv('results/all_model_performance.csv',
                            index=False)
            ok("results/all_model_performance.csv saved "
               "(Step 5 + Step 6 combined)")
            print(f"\n  Combined ranking (all 8 classifiers):")
            print(f"  {'Model':<25} {'AUROC':>7} {'Ur':>7} "
                  f"{'Brier':>7} {'HR@10%':>8}")
            print(f"  {'-'*60}")
            for _, r in combined.iterrows():
                print(f"  {r['model']:<25} {r['auroc']:>7.3f} "
                      f"{r['ur']:>7.3f} {r['brier']:>7.3f} "
                      f"{r['hr10']:>8.3f}")
    else:
        warn("No deep model results — check PyTorch installation "
             "and data coverage")

else:
    warn("PyTorch not available — Step 6 skipped")
    warn("All Step 5 results remain valid")
    warn("To install PyTorch:  pip install torch")

print(f"""
  ════════════════════════════════════════════════════
  STEP 6 OUTPUTS
  ════════════════════════════════════════════════════
  data/cleaned/predictions_lstm_mlp.csv
  data/cleaned/predictions_transformer.csv
  results/deep_model_performance.csv
  results/all_model_performance.csv   (if Step 5 also ran)
  ════════════════════════════════════════════════════
""")
print("Step 6 complete.")
print("Next: run Step 7 (step7_ensemble.py)")


  STEP 6 — LSTM + TRANSFORMER (self-contained)
  !!  ml_panel.csv not found — downloading and building now

--- A1. FRED API ---
  !!  FRED key not set — FRED features will be skipped
  !!  Set FRED_KEY at the top of this script

--- A2. Yahoo Finance ---
  OK  Yahoo Finance: 497 months x 21 series

--- B. Feature panel ---

--- C. Target variable Y ---
  !!  Using known historical US bubble periods as fallback
  OK    Y=1: 1997-06-01 to 2000-03-01 (Dot-com build-up)
  OK    Y=1: 2004-06-01 to 2007-06-01 (Housing build-up)
  OK    Y=1: 2009-06-01 to 2010-06-01 (Post-GFC rally)
  OK    Y=1: 2020-06-01 to 2021-09-01 (Post-COVID exuberance)
  OK  Target: Y=1=100 months (20.1%)
  OK  ml_panel.csv saved: 311 obs x 84 features

--- E. PyTorch check ---
  OK  PyTorch 2.10.0+cpu available

--- F. Building sequence tensors ---
  OK  Device: cpu
  OK  Sequence tensor: (303, 8, 84)  (samples=303, seq_len=8Q, features=84)
  OK  Train: 212 samples  |  Test: 91 samples
  OK  Train Y=1: 63  |  Test Y

In [2]:
# ════════════════════════════════════════════════════════════════════════
# STEP 7 — Ensemble Aggregation
# SELF-CONTAINED: downloads all data and runs full ML pipeline internally
# No prior steps required — just set your FRED key below
#
# Implements:
#   Section 3.5.6 — Three ensemble specifications
#     E1: Unweighted average
#     E2: Performance-weighted (AUROC weights)
#     E3: Stacking meta-learner (logistic regression)
#   Holopainen & Sarlin (2017): ensemble > any individual classifier
# ════════════════════════════════════════════════════════════════════════
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.linear_model   import LogisticRegression
from sklearn.ensemble       import RandomForestClassifier
from sklearn.svm            import SVC
from sklearn.neighbors      import KNeighborsClassifier
from sklearn.preprocessing  import StandardScaler
from sklearn.metrics        import (roc_auc_score, brier_score_loss,
                                     roc_curve)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
warnings.filterwarnings('ignore')

# ── SET YOUR FRED KEY HERE ────────────────────────────────────────────
FRED_KEY = os.environ.get('FRED_API_KEY', 'paste_your_key_here')

# ── Config ────────────────────────────────────────────────────────────
START_DATE   = '1980-01-01'
LEAD_MIN     = 4
LEAD_MAX     = 12
RANDOM_STATE = 42
BURN_IN      = 60
MU           = 0.5
SMOTE_RATIO  = 0.20
CORR_THRESH  = 0.85

for d in ['data/raw', 'data/cleaned', 'figures', 'results']:
    os.makedirs(d, exist_ok=True)

def ok(m):   print(f"  OK  {m}")
def warn(m): print(f"  !!  {m}")
def err(m):  print(f"  XX  {m}")
def sep(t):  print(f"\n--- {t} ---")

def nearest_idx(index, target_date):
    """Nearest DatetimeIndex position — pandas 2.x compatible."""
    return int(np.abs((index - target_date).total_seconds()).argmin())

print("=" * 60)
print("  STEP 7 — ENSEMBLE AGGREGATION (self-contained)")
print("=" * 60)


# ════════════════════════════════════════════════════════════════════════
# SECTION A — LOAD OR BUILD PREDICTIONS FROM STEP 5
# ════════════════════════════════════════════════════════════════════════

PRED_PATH  = 'data/cleaned/predictions_ml.csv'
PANEL_PATH = 'data/cleaned/ml_panel.csv'
PERF_PATH  = 'results/model_performance.csv'

# Check if Step 5 outputs already exist
step5_done = (os.path.exists(PRED_PATH)
              and os.path.exists(PERF_PATH)
              and os.path.exists(PANEL_PATH))

if step5_done:
    ok("Step 5 outputs found — loading directly")
    pred_ml = pd.read_csv(PRED_PATH, parse_dates=['date'],
                           index_col='date')
    perf_df = pd.read_csv(PERF_PATH)
    panel_ml = pd.read_csv(PANEL_PATH, parse_dates=['date'])
    ok(f"predictions_ml.csv: {pred_ml.shape[0]} windows")
    ok(f"model_performance.csv: {len(perf_df)} models")

else:
    warn("Step 5 outputs not found — running full pipeline now")
    warn("This replicates Steps 4 + 5 internally")

    # ── DATA DOWNLOAD ─────────────────────────────────────────────────
    sep("A1. FRED API")
    fred_df = pd.DataFrame()

    if not FRED_KEY or FRED_KEY == 'paste_your_key_here':
        warn("FRED key not set — FRED features will be skipped")
    else:
        try:
            from fredapi import Fred
            fred_obj = Fred(api_key=FRED_KEY)
            fred_series = {
                'SP500':            'sp500',
                'VIXCLS':           'vix',
                'CSUSHPISA':        'case_shiller_hpi',
                'DCOILWTICO':       'oil_wti',
                'GOLDAMGBD228NLBM': 'gold_price',
                'FEDFUNDS':         'fed_funds_rate',
                'T10Y2Y':           'term_spread_10y2y',
                'T10Y3M':           'term_spread_10y3m',
                'MORTGAGE30US':     'mortgage_rate_30y',
                'BAMLH0A0HYM2':     'hy_credit_spread',
                'BAMLC0A0CM':       'ig_credit_spread',
                'NFCI':             'nfci_financial_conditions',
                'STLFSI4':          'stl_financial_stress',
                'TOTALSL':          'consumer_credit_total',
                'BUSLOANS':         'commercial_loans',
                'REALLN':           'real_estate_loans',
                'DPCREDIT':         'domestic_credit_private',
                'DRSFRMACBS':       'delinquency_mortgage',
                'DRCCLACBS':        'delinquency_credit_card',
                'DRBLACBS':         'delinquency_business',
                'GFDEGDQ188S':      'federal_debt_pct_gdp',
                'GDPC1':            'real_gdp',
                'UNRATE':           'unemployment_us',
                'CPIAUCSL':         'cpi_us',
                'INDPRO':           'industrial_production',
                'HOUST':            'housing_starts',
            }
            fred_raw = {}
            for sid, name in fred_series.items():
                try:
                    fred_raw[name] = fred_obj.get_series(
                        sid, observation_start=START_DATE)
                    time.sleep(0.05)
                except Exception as ex:
                    warn(f"  FRED {sid}: {str(ex)[:45]}")
            for sid, name in [('CAPE','cape_shiller'),
                               ('SP500PE','pe_ratio'),
                               ('SP500DIV','div_yield')]:
                try:
                    fred_raw[name] = (fred_obj.get_series(
                        sid, observation_start=START_DATE)
                        .resample('MS').last())
                    ok(f"  {name} ({sid})")
                except Exception:
                    warn(f"  {sid} not in FRED")
            fred_df = pd.DataFrame(fred_raw)
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df.index.name = 'date'
            fred_df = fred_df.sort_index()
            fred_df.to_csv('data/raw/fred_data.csv')
            ok(f"FRED: {fred_df.shape[0]} obs x {fred_df.shape[1]} series")
        except Exception as e:
            err(f"FRED failed: {e}")

    sep("A2. Yahoo Finance")
    yf_df = pd.DataFrame()
    try:
        import yfinance as yf
        all_tickers = {
            'SP500':'^GSPC',   'FTSE100':'^FTSE',  'DAX':'^GDAXI',
            'CAC40':'^FCHI',   'Nikkei':'^N225',   'TSX':'^GSPTSE',
            'ASX200':'^AXJO',  'KOSPI':'^KS11',    'IBEX35':'^IBEX',
            'AEX':'^AEX',      'SMI':'^SSMI',      'BOVESPA':'^BVSP',
            'SENSEX':'^BSESN', 'HSI':'^HSI',       'NIFTY50':'^NSEI',
            'STI':'^STI',      'US_10Y':'^TNX',    'US_3M':'^IRX',
            'US_30Y':'^TYX',   'VIX_YF':'^VIX',
        }
        yf_raw = yf.download(
            list(all_tickers.values()), start=START_DATE,
            interval='1mo', auto_adjust=True,
            progress=False, threads=True)['Close']
        reverse = {v: k for k, v in all_tickers.items()}
        yf_raw.columns = [reverse.get(c, c) for c in yf_raw.columns]
        yf_raw.index = pd.to_datetime(yf_raw.index)
        yf_raw.index.name = 'date'
        yf_raw = yf_raw.sort_index()
        if 'US_10Y' in yf_raw.columns and 'US_3M' in yf_raw.columns:
            yf_raw['term_spread_yf'] = yf_raw['US_10Y'] - yf_raw['US_3M']
        yf_raw.to_csv('data/raw/yfinance_data.csv')
        yf_df = yf_raw
        ok(f"Yahoo Finance: {yf_df.shape[0]} months x {yf_df.shape[1]} series")
    except Exception as e:
        err(f"Yahoo Finance failed: {e}")

    # ── FEATURE PANEL ─────────────────────────────────────────────────
    sep("B. Feature panel")
    if 'SP500' in yf_df.columns:
        panel = yf_df[['SP500']].copy().rename(columns={'SP500':'price'})
    elif not fred_df.empty and 'sp500' in fred_df.columns:
        panel = (fred_df[['sp500']].resample('MS').last()
                 .rename(columns={'sp500':'price'}))
    else:
        raise RuntimeError("No SP500 data. Check FRED key or Yahoo Finance.")

    panel['mom_1m']     = panel['price'].pct_change(1)
    panel['mom_3m']     = panel['price'].pct_change(3)
    panel['mom_6m']     = panel['price'].pct_change(6)
    panel['mom_12m']    = panel['price'].pct_change(12)
    panel['vol_3m']     = panel['price'].pct_change().rolling(3).std()
    panel['vol_6m']     = panel['price'].pct_change().rolling(6).std()
    panel['vol_12m']    = panel['price'].pct_change().rolling(12).std()
    panel['ret_excess'] = (panel['price'].pct_change()
                           - panel['price'].pct_change().rolling(60).mean())

    if not fred_df.empty and 'cape_shiller' in fred_df.columns:
        panel['CAPE'] = (fred_df['cape_shiller'].resample('MS').last()
                         .reindex(panel.index, method='ffill'))
        panel['CAPE_mean'] = panel['CAPE'].rolling(120).mean()
        panel['CAPE_std']  = panel['CAPE'].rolling(120).std()
        panel['CAPE_z']    = ((panel['CAPE']-panel['CAPE_mean'])
                              / panel['CAPE_std'].replace(0, np.nan))
    elif not fred_df.empty and 'pe_ratio' in fred_df.columns:
        panel['PE'] = (fred_df['pe_ratio'].resample('MS').last()
                       .reindex(panel.index, method='ffill'))
        panel['PE_mean'] = panel['PE'].rolling(120).mean()
        panel['PE_std']  = panel['PE'].rolling(120).std()
        panel['CAPE_z']  = ((panel['PE']-panel['PE_mean'])
                            / panel['PE_std'].replace(0, np.nan))

    if not fred_df.empty and 'div_yield' in fred_df.columns:
        panel['div_yield'] = (fred_df['div_yield'].resample('MS').last()
                              .reindex(panel.index, method='ffill'))

    if not fred_df.empty:
        fred_m = fred_df.resample('MS').last()
        for col in [
            'hy_credit_spread','ig_credit_spread',
            'nfci_financial_conditions','stl_financial_stress',
            'consumer_credit_total','commercial_loans',
            'real_estate_loans','domestic_credit_private',
            'federal_debt_pct_gdp','delinquency_mortgage',
            'delinquency_credit_card','delinquency_business',
            'fed_funds_rate','term_spread_10y2y','term_spread_10y3m',
            'mortgage_rate_30y','vix','case_shiller_hpi',
            'oil_wti','gold_price','real_gdp','unemployment_us',
            'cpi_us','industrial_production','housing_starts',
        ]:
            if col in fred_m.columns:
                panel[col] = fred_m[col].reindex(
                    panel.index, method='ffill')

    if not yf_df.empty:
        for col in ['VIX_YF','US_10Y','US_3M','US_30Y',
                    'term_spread_yf','FTSE100','DAX','Nikkei',
                    'BOVESPA','SENSEX','KOSPI','ASX200','HSI']:
            if col in yf_df.columns:
                panel[col] = yf_df[col].reindex(
                    panel.index, method='ffill')

    if 'US_10Y' in panel.columns and 'cpi_us' in panel.columns:
        panel['real_rate_10y'] = (panel['US_10Y']
                                   - panel['cpi_us'].pct_change(12)*100)
    if 'hy_credit_spread' in panel.columns and 'ig_credit_spread' in panel.columns:
        panel['spread_diff'] = (panel['hy_credit_spread']
                                 - panel['ig_credit_spread'])
    if 'consumer_credit_total' in panel.columns:
        panel['credit_growth_12m'] = panel['consumer_credit_total'].pct_change(12)
    if 'case_shiller_hpi' in panel.columns:
        panel['house_price_chg12'] = panel['case_shiller_hpi'].pct_change(12)
    if 'real_gdp' in panel.columns:
        panel['gdp_growth_yoy'] = panel['real_gdp'].pct_change(4)*100

    base_feats = [c for c in panel.columns
                  if c not in ['price'] and panel[c].notna().mean() > 0.5]
    for col in base_feats:
        panel[f'{col}_lag3']  = panel[col].shift(3)
        panel[f'{col}_lag6']  = panel[col].shift(6)
        panel[f'{col}_lag12'] = panel[col].shift(12)

    panel = panel.replace([np.inf, -np.inf], np.nan).sort_index()

    # ── TARGET Y ──────────────────────────────────────────────────────
    sep("C. Target variable Y")
    panel['Y'] = 0
    gsadf_used = False

    if os.path.exists('data/cleaned/bubble_dates.csv'):
        try:
            eps_df = pd.read_csv('data/cleaned/bubble_dates.csv',
                                  parse_dates=['inception','termination'])
            us_eps = eps_df[eps_df['country'] == 'US']
            if not us_eps.empty:
                for _, ep in us_eps.iterrows():
                    inc = pd.Timestamp(ep['inception'])
                    for lead in range(LEAD_MIN, LEAD_MAX + 1):
                        sig = inc - pd.DateOffset(months=lead)
                        idx = nearest_idx(panel.index, sig)
                        panel.iloc[idx, panel.columns.get_loc('Y')] = 1
                for _, ep in us_eps.iterrows():
                    for anchor in [pd.Timestamp(ep['inception']),
                                   pd.Timestamp(ep['termination'])]:
                        for off in range(-3, 4):
                            excl = anchor + pd.DateOffset(months=off)
                            idx  = nearest_idx(panel.index, excl)
                            if panel.iloc[idx]['Y'] == 0:
                                panel.iloc[
                                    idx, panel.columns.get_loc('Y')] = np.nan
                panel = panel.dropna(subset=['Y'])
                gsadf_used = True
                ok(f"GSADF target: {int(panel['Y'].sum())} positive months")
        except Exception as ex:
            warn(f"bubble_dates.csv error: {ex}")

    if not gsadf_used or int(panel['Y'].sum()) == 0:
        warn("Using known historical US bubble periods as fallback")
        panel['Y'] = 0
        for s, e, label in [
            ('1997-06-01','2000-03-01','Dot-com build-up'),
            ('2004-06-01','2007-06-01','Housing build-up'),
            ('2009-06-01','2010-06-01','Post-GFC rally'),
            ('2020-06-01','2021-09-01','Post-COVID exuberance'),
        ]:
            panel.loc[(panel.index >= s) & (panel.index <= e), 'Y'] = 1
            ok(f"  Y=1: {s} to {e} ({label})")

    ok(f"Target: Y=1={int(panel['Y'].sum())} months "
       f"({panel['Y'].mean()*100:.1f}%)")

    # ── FEATURE MATRIX ────────────────────────────────────────────────
    feat_cols_all = [c for c in panel.columns
                     if c not in ['Y','price']
                     and panel[c].notna().mean() > 0.55]
    panel_ml = (panel[feat_cols_all + ['Y']].dropna()
                .reset_index())
    if panel_ml.columns[0] != 'date':
        panel_ml = panel_ml.rename(
            columns={panel_ml.columns[0]: 'date'})
    panel_ml.to_csv(PANEL_PATH, index=False)
    ok(f"ml_panel.csv: {panel_ml.shape[0]} obs x "
       f"{panel_ml.shape[1]-2} features")

    # ── RUN STEP 5 ML PIPELINE ────────────────────────────────────────
    sep("D. Running ML classifiers (Step 5 pipeline)")

    feat_cols = [c for c in panel_ml.columns if c not in ['date','Y']]
    X_all = panel_ml[feat_cols]
    y_all = panel_ml['Y'].astype(int)
    eval_idx = [i for i in range(BURN_IN, len(panel_ml)) if i % 6 == 0]

    def relative_usefulness(yt, yp, mu=MU, thr=0.5):
        pred = (yp >= thr).astype(int)
        tp = int(((pred==1)&(yt==1)).sum())
        fp = int(((pred==1)&(yt==0)).sum())
        tn = int(((pred==0)&(yt==0)).sum())
        fn = int(((pred==0)&(yt==1)).sum())
        T1 = fn/(tp+fn) if tp+fn > 0 else 1.
        T2 = fp/(fp+tn) if fp+tn > 0 else 1.
        return 1. - (1./min(mu, 1.-mu)) * (mu*T2 + (1.-mu)*T1)

    def hr_at_fpr(yt, ys, fpr_tgt=0.10):
        fpr, tpr, _ = roc_curve(yt, ys)
        return float(tpr[min(np.searchsorted(fpr, fpr_tgt), len(tpr)-1)])

    def score(name, yt, ys):
        if len(np.unique(yt)) < 2:
            return {'model':name,'auroc':np.nan,'brier':np.nan,
                    'ur':np.nan,'hr10':np.nan}
        return {'model':name,
                'auroc':float(roc_auc_score(yt, ys)),
                'brier':float(brier_score_loss(yt, ys)),
                'ur':   float(relative_usefulness(yt, ys)),
                'hr10': float(hr_at_fpr(yt, ys))}

    def select_features(Xtr, ytr):
        try:
            from boruta import BorutaPy
            rf_b = RandomForestClassifier(
                n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
            sel = BorutaPy(rf_b, n_estimators='auto', max_iter=30,
                            random_state=RANDOM_STATE, verbose=0)
            sel.fit(Xtr.values, ytr.values.ravel())
            confirmed = Xtr.columns[sel.support_].tolist()
            if not confirmed: confirmed = Xtr.columns.tolist()
        except Exception:
            rf_t = RandomForestClassifier(
                n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
            rf_t.fit(Xtr, ytr)
            imp = rf_t.feature_importances_
            confirmed = Xtr.columns[imp >= imp.mean()].tolist()
        Xa = Xtr[confirmed]
        corr = Xa.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        drop = set()
        for c in upper.columns:
            drop.update(upper[c][upper[c] > CORR_THRESH].index.tolist())
        Xb = Xa.drop(columns=list(drop), errors='ignore')
        if Xb.shape[1] > 3:
            try:
                sc = StandardScaler()
                Xs = sc.fit_transform(Xb)
                lr = LogisticRegression(penalty='l1', solver='liblinear',
                                         C=0.05, max_iter=500,
                                         random_state=RANDOM_STATE)
                lr.fit(Xs, ytr)
                nz = Xb.columns[np.abs(lr.coef_[0]) > 0].tolist()
                Xc = Xb[nz] if nz else Xb
            except Exception:
                Xc = Xb
        else:
            Xc = Xb
        return (Xc.columns.tolist()
                if len(Xc.columns) >= 2
                else Xb.columns.tolist())

    def make_classifiers():
        return {
            'LR_L2': LogisticRegression(C=1., max_iter=500,
                                         random_state=RANDOM_STATE),
            'LR_L1': LogisticRegression(penalty='l1',
                                         solver='liblinear', C=0.1,
                                         max_iter=500,
                                         random_state=RANDOM_STATE),
            'RF':    RandomForestClassifier(n_estimators=300,
                                             max_features='sqrt',
                                             min_samples_leaf=5,
                                             random_state=RANDOM_STATE,
                                             n_jobs=-1),
            'XGB':   xgb.XGBClassifier(learning_rate=0.05,
                                         max_depth=4, n_estimators=200,
                                         subsample=0.8,
                                         random_state=RANDOM_STATE,
                                         eval_metric='logloss',
                                         verbosity=0),
            'SVM':   SVC(kernel='rbf', C=10, gamma='scale',
                          probability=True,
                          random_state=RANDOM_STATE),
            'KNN':   KNeighborsClassifier(n_neighbors=5),
        }

    sel_f = feat_cols.copy()
    preds = []
    ok(f"Windows: {len(eval_idx)} | Burn-in: {BURN_IN}")
    print("  Running recursive evaluation...")

    for w, sp in enumerate(eval_idx):
        Xtr = X_all.iloc[:sp]; ytr = y_all.iloc[:sp]
        Xte = X_all.iloc[[sp]]; yte = y_all.iloc[[sp]]
        if ytr.sum() < 5: continue
        if w % 12 == 0: sel_f = select_features(Xtr, ytr)
        Xtrs = Xtr[sel_f]; Xtes = Xte[sel_f]
        sc = StandardScaler()
        Xtr_sc = sc.fit_transform(Xtrs)
        Xte_sc = sc.transform(Xtes)
        try:
            sm = SMOTE(sampling_strategy=SMOTE_RATIO,
                       random_state=RANDOM_STATE, k_neighbors=3)
            Xtr_sm, ytr_sm = sm.fit_resample(Xtr_sc, ytr)
        except Exception:
            Xtr_sm, ytr_sm = Xtr_sc, ytr.values
        row = {'date': panel_ml.loc[sp,'date'],
               'y_true': int(yte.iloc[0]),
               'n_train': sp, 'n_pos': int(ytr.sum())}
        for nm, clf in make_classifiers().items():
            try:
                clf.fit(Xtr_sm, ytr_sm)
                row[f'p_{nm}'] = float(
                    clf.predict_proba(Xte_sc)[0, 1])
            except Exception:
                row[f'p_{nm}'] = np.nan
        probs = [v for k, v in row.items()
                 if k.startswith('p_') and not np.isnan(v)]
        row['p_Ensemble_unweighted'] = (
            float(np.mean(probs)) if probs else np.nan)
        preds.append(row)
        if w % 20 == 0:
            print(f"    Window {w+1}/{len(eval_idx)}: "
                  f"train={sp} pos={ytr.sum()}")

    pred_ml = pd.DataFrame(preds).set_index('date')
    pred_ml.to_csv(PRED_PATH)
    ok(f"predictions_ml.csv: {len(pred_ml)} windows")

    prob_cols_tmp = [c for c in pred_ml.columns if c.startswith('p_')]
    perf_rows = []
    for col in prob_cols_tmp:
        v = pred_ml[col].notna()
        if v.sum() < 10: continue
        yt = pred_ml.loc[v,'y_true'].values
        ys = pred_ml.loc[v,col].values
        if len(np.unique(yt)) < 2: continue
        perf_rows.append(score(col.replace('p_',''), yt, ys))
    perf_df = pd.DataFrame(perf_rows).sort_values(
        'auroc', ascending=False)
    perf_df.to_csv(PERF_PATH, index=False)
    ok(f"model_performance.csv: {len(perf_df)} models")


# ════════════════════════════════════════════════════════════════════════
# SECTION B — ENSEMBLE AGGREGATION (Section 3.5.6)
# ════════════════════════════════════════════════════════════════════════
sep("E. Ensemble aggregation (Section 3.5.6)")

prob_cols = [c for c in pred_ml.columns
             if c.startswith('p_') and 'Ensemble' not in c]
prob_matrix = pred_ml[prob_cols].copy()

ok(f"Base classifiers: {[c.replace('p_','') for c in prob_cols]}")


# ── Ensemble 1: Unweighted average ────────────────────────────────────
pred_ml['p_Ensemble_unweighted'] = prob_matrix.mean(axis=1)
ok("Ensemble 1: unweighted average of all classifiers")


# ── Ensemble 2: Performance-weighted (AUROC weights) ──────────────────
weights = {}
for col in prob_cols:
    nm  = col.replace('p_', '')
    row = perf_df[perf_df['model'] == nm]
    weights[col] = (
        float(row['auroc'].iloc[0])
        if len(row) > 0 and not pd.isna(row['auroc'].iloc[0])
        else 0.5
    )

total_w = sum(weights.values())
if total_w > 0:
    pred_ml['p_Ensemble_weighted'] = (
        sum(prob_matrix[c] * weights[c] for c in prob_cols) / total_w
    )
    weight_str = '  '.join(
        f"{c.replace('p_','')}={v:.3f}"
        for c, v in weights.items())
    ok("Ensemble 2: AUROC-weighted average")
    ok(f"  Weights: {weight_str}")


# ── Ensemble 3: Stacking meta-learner (logistic regression) ───────────
try:
    n     = len(prob_matrix)
    split = int(n * 0.70)
    Xtr_s = prob_matrix.iloc[:split].fillna(0.5).values
    ytr_s = pred_ml['y_true'].iloc[:split].values.astype(int)
    Xte_s = prob_matrix.iloc[split:].fillna(0.5).values
    yte_s = pred_ml['y_true'].iloc[split:].values.astype(int)

    if len(np.unique(ytr_s)) > 1:
        meta = LogisticRegression(C=1., max_iter=500, random_state=42)
        meta.fit(Xtr_s, ytr_s)
        stack_proba = meta.predict_proba(Xte_s)[:, 1]
        pred_ml.loc[pred_ml.index[split:],
                    'p_Ensemble_stacking'] = stack_proba
        ok("Ensemble 3: stacking meta-learner fitted")
        ok(f"  Meta-learner: LogisticRegression on "
           f"{len(prob_cols)} base classifier outputs")
    else:
        warn("Stacking: training set has only one class — skipped")
except Exception as ex:
    warn(f"Stacking failed: {ex}")


# ── Try to add deep model predictions if they exist ───────────────────
deep_files = {
    'LSTM_MLP':    'data/cleaned/predictions_lstm_mlp.csv',
    'Transformer': 'data/cleaned/predictions_transformer.csv',
}
deep_added = []
for arch_name, path in deep_files.items():
    if os.path.exists(path):
        try:
            deep_pred = pd.read_csv(path, parse_dates=['date'],
                                     index_col='date')
            col_name = f'p_{arch_name}'
            if col_name in deep_pred.columns:
                pred_ml[col_name] = deep_pred[col_name].reindex(
                    pred_ml.index)
                deep_added.append(arch_name)
                ok(f"Added {arch_name} predictions from Step 6")
        except Exception as ex:
            warn(f"Could not load {arch_name}: {ex}")

if deep_added:
    # Recompute ensemble including deep models
    all_base_cols = [c for c in pred_ml.columns
                     if c.startswith('p_') and 'Ensemble' not in c]
    pred_ml['p_Ensemble_all'] = pred_ml[all_base_cols].mean(axis=1)
    ok(f"Ensemble_all: unweighted across ALL "
       f"{len(all_base_cols)} classifiers (incl. deep)")


# ── Save final predictions ────────────────────────────────────────────
pred_ml.to_csv('data/cleaned/predictions_final.csv')
ok("predictions_final.csv saved — all classifiers + 3 ensembles")


# ════════════════════════════════════════════════════════════════════════
# SECTION C — FINAL PERFORMANCE COMPARISON TABLE
# ════════════════════════════════════════════════════════════════════════
sep("F. Final performance comparison (Table 3.2 style)")

def score_full(name, yt, ys):
    if len(np.unique(yt)) < 2:
        return None
    pred = (ys >= 0.5).astype(int)
    tp = int(((pred==1)&(yt==1)).sum())
    fp = int(((pred==1)&(yt==0)).sum())
    tn = int(((pred==0)&(yt==0)).sum())
    fn = int(((pred==0)&(yt==1)).sum())
    T1 = fn/(tp+fn) if tp+fn > 0 else 1.
    T2 = fp/(fp+tn) if fp+tn > 0 else 1.
    ur = 1. - (1./min(MU, 1.-MU)) * (MU*T2 + (1.-MU)*T1)
    fpr, tpr, _ = roc_curve(yt, ys)
    hr10 = float(tpr[min(np.searchsorted(fpr, 0.10), len(tpr)-1)])
    return {
        'model': name,
        'auroc': float(roc_auc_score(yt, ys)),
        'brier': float(brier_score_loss(yt, ys)),
        'ur':    ur,
        'hr10':  hr10,
    }

all_prob_cols = [c for c in pred_ml.columns if c.startswith('p_')]
all_perf = []
for col in all_prob_cols:
    v = pred_ml[col].notna()
    if v.sum() < 10:
        continue
    yt = pred_ml.loc[v, 'y_true'].values.astype(int)
    ys = pred_ml.loc[v, col].values
    r  = score_full(col.replace('p_', ''), yt, ys)
    if r:
        all_perf.append(r)

all_perf_df = pd.DataFrame(all_perf).sort_values(
    'auroc', ascending=False)
all_perf_df.to_csv('results/final_model_comparison.csv', index=False)

# Separate individual vs ensemble in display
ind_models = all_perf_df[~all_perf_df['model'].str.contains('Ensemble')]
ens_models = all_perf_df[ all_perf_df['model'].str.contains('Ensemble')]

print(f"\n  Individual classifiers:")
print(f"  {'Model':<30} {'AUROC':>7} {'Ur':>7} "
      f"{'Brier':>7} {'HR@10%':>8}")
print(f"  {'-'*60}")
for _, r in ind_models.iterrows():
    print(f"  {r['model']:<30} {r['auroc']:>7.3f} "
          f"{r['ur']:>7.3f} {r['brier']:>7.3f} "
          f"{r['hr10']:>8.3f}")

if len(ens_models) > 0:
    print(f"\n  Ensemble specifications:")
    print(f"  {'Model':<30} {'AUROC':>7} {'Ur':>7} "
          f"{'Brier':>7} {'HR@10%':>8}")
    print(f"  {'-'*60}")
    for _, r in ens_models.iterrows():
        print(f"  {r['model']:<30} {r['auroc']:>7.3f} "
              f"{r['ur']:>7.3f} {r['brier']:>7.3f} "
              f"{r['hr10']:>8.3f}")

# Best overall
best = all_perf_df.iloc[0]
print(f"\n  Best model: {best['model']}")
print(f"    AUROC={best['auroc']:.4f}  "
      f"Ur={best['ur']:.4f}  "
      f"Brier={best['brier']:.4f}  "
      f"HR@10%={best['hr10']:.4f}")


# ── Plot: ensemble vs individual ──────────────────────────────────────
if len(all_perf_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['#e74c3c' if 'Ensemble' in m else '#2980b9'
              for m in all_perf_df['model']]
    bars = ax.bar(all_perf_df['model'],
                  all_perf_df['auroc'].fillna(0),
                  color=colors, alpha=0.85, edgecolor='white')
    ax.axhline(0.5, color='black', ls='--', lw=0.8, label='Random (0.5)')
    ax.set_ylabel('AUROC')
    ax.set_title('Final Model Comparison — Individual vs Ensemble\n'
                 'Red = ensemble | Blue = individual classifier',
                 fontweight='bold')
    plt.xticks(rotation=35, ha='right', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend(fontsize=9)
    for bar, val in zip(bars, all_perf_df['auroc'].fillna(0)):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{val:.3f}', ha='center',
                va='bottom', fontsize=8)
    plt.tight_layout()
    plt.savefig('figures/ensemble_comparison.png',
                dpi=150, bbox_inches='tight')
    plt.close()
    ok("figures/ensemble_comparison.png saved")

print(f"""
  ════════════════════════════════════════════════════
  STEP 7 OUTPUTS
  ════════════════════════════════════════════════════
  data/cleaned/predictions_final.csv   all predictions
  results/final_model_comparison.csv   full Table 3.2
  figures/ensemble_comparison.png      AUROC bar chart
  ════════════════════════════════════════════════════
""")
print("Step 7 complete.")
print("Next: run Step 8 (step8_shap.py)")


  STEP 7 — ENSEMBLE AGGREGATION (self-contained)
  !!  Step 5 outputs not found — running full pipeline now
  !!  This replicates Steps 4 + 5 internally

--- A1. FRED API ---
  !!  FRED key not set — FRED features will be skipped

--- A2. Yahoo Finance ---
  OK  Yahoo Finance: 497 months x 21 series

--- B. Feature panel ---

--- C. Target variable Y ---
  !!  Using known historical US bubble periods as fallback
  OK    Y=1: 1997-06-01 to 2000-03-01 (Dot-com build-up)
  OK    Y=1: 2004-06-01 to 2007-06-01 (Housing build-up)
  OK    Y=1: 2009-06-01 to 2010-06-01 (Post-GFC rally)
  OK    Y=1: 2020-06-01 to 2021-09-01 (Post-COVID exuberance)
  OK  Target: Y=1=100 months (20.1%)
  OK  ml_panel.csv: 311 obs x 84 features

--- D. Running ML classifiers (Step 5 pipeline) ---
  OK  Windows: 42 | Burn-in: 60
  Running recursive evaluation...
    Window 1/42: train=60 pos=21
    Window 21/42: train=180 pos=71
    Window 41/42: train=300 pos=87
  OK  predictions_ml.csv: 42 windows
  OK  model_per

In [3]:
# ════════════════════════════════════════════════════════════════════════
# STEP 8 — SHAP Interpretability
# SELF-CONTAINED: downloads all data and trains models internally
# No prior steps required — just set your FRED key below
#
# Implements Section 3.6 — three analyses:
#   Analysis 1: Global SHAP importance (TreeSHAP — RF + XGB)
#   Analysis 2: Temporal SHAP trajectories pre-bubble
#   Analysis 3: Individual episode decompositions (6 episodes)
#
# Lundberg & Lee (2017) | TreeSHAP | Addresses Gap 6
# ════════════════════════════════════════════════════════════════════════
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
from sklearn.ensemble       import RandomForestClassifier
from sklearn.preprocessing  import StandardScaler
from imblearn.over_sampling import SMOTE
import xgboost as xgb
warnings.filterwarnings('ignore')

# ── SET YOUR FRED KEY HERE ────────────────────────────────────────────
FRED_KEY = os.environ.get('FRED_API_KEY', 'paste_your_key_here')

# ── Config ────────────────────────────────────────────────────────────
START_DATE   = '1980-01-01'
LEAD_MIN     = 4
LEAD_MAX     = 12
RANDOM_STATE = 42
SMOTE_RATIO  = 0.20

for d in ['data/raw', 'data/cleaned', 'figures', 'results']:
    os.makedirs(d, exist_ok=True)

def ok(m):   print(f"  OK  {m}")
def warn(m): print(f"  !!  {m}")
def err(m):  print(f"  XX  {m}")
def sep(t):  print(f"\n--- {t} ---")

def nearest_idx(index, target_date):
    """Nearest DatetimeIndex position — pandas 2.x compatible."""
    return int(np.abs((index - target_date).total_seconds()).argmin())

print("=" * 60)
print("  STEP 8 — SHAP INTERPRETABILITY (self-contained)")
print("=" * 60)


# ════════════════════════════════════════════════════════════════════════
# SECTION A — LOAD OR BUILD ml_panel.csv
# ════════════════════════════════════════════════════════════════════════

PANEL_PATH = 'data/cleaned/ml_panel.csv'

if os.path.exists(PANEL_PATH):
    ok("ml_panel.csv found — skipping data download")
    panel_ml = pd.read_csv(PANEL_PATH, parse_dates=['date'])
    ok(f"Loaded: {panel_ml.shape[0]} obs x "
       f"{panel_ml.shape[1]-2} features")

else:
    warn("ml_panel.csv not found — downloading and building now")

    # ── A1. FRED ──────────────────────────────────────────────────────
    sep("A1. FRED API")
    fred_df = pd.DataFrame()

    if not FRED_KEY or FRED_KEY == 'paste_your_key_here':
        warn("FRED key not set — FRED features will be skipped")
        warn("Set FRED_KEY at the top of this script")
    else:
        try:
            from fredapi import Fred
            fred_obj = Fred(api_key=FRED_KEY)
            fred_series = {
                'SP500':            'sp500',
                'VIXCLS':           'vix',
                'CSUSHPISA':        'case_shiller_hpi',
                'DCOILWTICO':       'oil_wti',
                'GOLDAMGBD228NLBM': 'gold_price',
                'FEDFUNDS':         'fed_funds_rate',
                'T10Y2Y':           'term_spread_10y2y',
                'T10Y3M':           'term_spread_10y3m',
                'MORTGAGE30US':     'mortgage_rate_30y',
                'BAMLH0A0HYM2':     'hy_credit_spread',
                'BAMLC0A0CM':       'ig_credit_spread',
                'NFCI':             'nfci_financial_conditions',
                'STLFSI4':          'stl_financial_stress',
                'TOTALSL':          'consumer_credit_total',
                'BUSLOANS':         'commercial_loans',
                'REALLN':           'real_estate_loans',
                'DPCREDIT':         'domestic_credit_private',
                'DRSFRMACBS':       'delinquency_mortgage',
                'DRCCLACBS':        'delinquency_credit_card',
                'DRBLACBS':         'delinquency_business',
                'GFDEGDQ188S':      'federal_debt_pct_gdp',
                'GDPC1':            'real_gdp',
                'UNRATE':           'unemployment_us',
                'CPIAUCSL':         'cpi_us',
                'INDPRO':           'industrial_production',
                'HOUST':            'housing_starts',
            }
            fred_raw = {}
            for sid, name in fred_series.items():
                try:
                    fred_raw[name] = fred_obj.get_series(
                        sid, observation_start=START_DATE)
                    time.sleep(0.05)
                except Exception as ex:
                    warn(f"  FRED {sid}: {str(ex)[:45]}")
            for sid, name in [('CAPE',    'cape_shiller'),
                               ('SP500PE', 'pe_ratio'),
                               ('SP500DIV','div_yield')]:
                try:
                    fred_raw[name] = (fred_obj.get_series(
                        sid, observation_start=START_DATE)
                        .resample('MS').last())
                    ok(f"  {name} ({sid})")
                except Exception:
                    warn(f"  {sid} not in FRED")
            fred_df = pd.DataFrame(fred_raw)
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df.index.name = 'date'
            fred_df = fred_df.sort_index()
            fred_df.to_csv('data/raw/fred_data.csv')
            ok(f"FRED: {fred_df.shape[0]} obs x {fred_df.shape[1]} series")
        except Exception as e:
            err(f"FRED failed: {e}")

    # ── A2. Yahoo Finance ─────────────────────────────────────────────
    sep("A2. Yahoo Finance")
    yf_df = pd.DataFrame()
    try:
        import yfinance as yf
        all_tickers = {
            'SP500':'^GSPC',   'FTSE100':'^FTSE',  'DAX':'^GDAXI',
            'CAC40':'^FCHI',   'Nikkei':'^N225',   'TSX':'^GSPTSE',
            'ASX200':'^AXJO',  'KOSPI':'^KS11',    'IBEX35':'^IBEX',
            'AEX':'^AEX',      'SMI':'^SSMI',      'BOVESPA':'^BVSP',
            'SENSEX':'^BSESN', 'HSI':'^HSI',       'NIFTY50':'^NSEI',
            'STI':'^STI',      'US_10Y':'^TNX',    'US_3M':'^IRX',
            'US_30Y':'^TYX',   'VIX_YF':'^VIX',
        }
        yf_raw = yf.download(
            list(all_tickers.values()), start=START_DATE,
            interval='1mo', auto_adjust=True,
            progress=False, threads=True)['Close']
        reverse = {v: k for k, v in all_tickers.items()}
        yf_raw.columns = [reverse.get(c, c) for c in yf_raw.columns]
        yf_raw.index = pd.to_datetime(yf_raw.index)
        yf_raw.index.name = 'date'
        yf_raw = yf_raw.sort_index()
        if 'US_10Y' in yf_raw.columns and 'US_3M' in yf_raw.columns:
            yf_raw['term_spread_yf'] = yf_raw['US_10Y'] - yf_raw['US_3M']
        yf_raw.to_csv('data/raw/yfinance_data.csv')
        yf_df = yf_raw
        ok(f"Yahoo Finance: {yf_df.shape[0]} months x {yf_df.shape[1]} series")
    except Exception as e:
        err(f"Yahoo Finance failed: {e}")

    # ── B. Feature panel ──────────────────────────────────────────────
    sep("B. Feature panel")
    if 'SP500' in yf_df.columns:
        panel = yf_df[['SP500']].copy().rename(columns={'SP500':'price'})
    elif not fred_df.empty and 'sp500' in fred_df.columns:
        panel = (fred_df[['sp500']].resample('MS').last()
                 .rename(columns={'sp500':'price'}))
    else:
        raise RuntimeError("No SP500 data. Check FRED key or Yahoo Finance.")

    panel['mom_1m']     = panel['price'].pct_change(1)
    panel['mom_3m']     = panel['price'].pct_change(3)
    panel['mom_6m']     = panel['price'].pct_change(6)
    panel['mom_12m']    = panel['price'].pct_change(12)
    panel['vol_3m']     = panel['price'].pct_change().rolling(3).std()
    panel['vol_6m']     = panel['price'].pct_change().rolling(6).std()
    panel['vol_12m']    = panel['price'].pct_change().rolling(12).std()
    panel['ret_excess'] = (panel['price'].pct_change()
                           - panel['price'].pct_change().rolling(60).mean())

    if not fred_df.empty and 'cape_shiller' in fred_df.columns:
        panel['CAPE'] = (fred_df['cape_shiller'].resample('MS').last()
                         .reindex(panel.index, method='ffill'))
        panel['CAPE_mean'] = panel['CAPE'].rolling(120).mean()
        panel['CAPE_std']  = panel['CAPE'].rolling(120).std()
        panel['CAPE_z']    = ((panel['CAPE']-panel['CAPE_mean'])
                              / panel['CAPE_std'].replace(0, np.nan))
    elif not fred_df.empty and 'pe_ratio' in fred_df.columns:
        panel['PE'] = (fred_df['pe_ratio'].resample('MS').last()
                       .reindex(panel.index, method='ffill'))
        panel['PE_mean'] = panel['PE'].rolling(120).mean()
        panel['PE_std']  = panel['PE'].rolling(120).std()
        panel['CAPE_z']  = ((panel['PE']-panel['PE_mean'])
                            / panel['PE_std'].replace(0, np.nan))

    if not fred_df.empty and 'div_yield' in fred_df.columns:
        panel['div_yield'] = (fred_df['div_yield'].resample('MS').last()
                              .reindex(panel.index, method='ffill'))

    if not fred_df.empty:
        fred_m = fred_df.resample('MS').last()
        for col in [
            'hy_credit_spread','ig_credit_spread',
            'nfci_financial_conditions','stl_financial_stress',
            'consumer_credit_total','commercial_loans',
            'real_estate_loans','domestic_credit_private',
            'federal_debt_pct_gdp','delinquency_mortgage',
            'delinquency_credit_card','delinquency_business',
            'fed_funds_rate','term_spread_10y2y','term_spread_10y3m',
            'mortgage_rate_30y','vix','case_shiller_hpi',
            'oil_wti','gold_price','real_gdp','unemployment_us',
            'cpi_us','industrial_production','housing_starts',
        ]:
            if col in fred_m.columns:
                panel[col] = fred_m[col].reindex(
                    panel.index, method='ffill')

    if not yf_df.empty:
        for col in ['VIX_YF','US_10Y','US_3M','US_30Y',
                    'term_spread_yf','FTSE100','DAX','Nikkei',
                    'BOVESPA','SENSEX','KOSPI','ASX200','HSI']:
            if col in yf_df.columns:
                panel[col] = yf_df[col].reindex(
                    panel.index, method='ffill')

    if 'US_10Y' in panel.columns and 'cpi_us' in panel.columns:
        panel['real_rate_10y'] = (panel['US_10Y']
                                   - panel['cpi_us'].pct_change(12)*100)
    if 'hy_credit_spread' in panel.columns and 'ig_credit_spread' in panel.columns:
        panel['spread_diff'] = (panel['hy_credit_spread']
                                 - panel['ig_credit_spread'])
    if 'consumer_credit_total' in panel.columns:
        panel['credit_growth_12m'] = panel['consumer_credit_total'].pct_change(12)
    if 'case_shiller_hpi' in panel.columns:
        panel['house_price_chg12'] = panel['case_shiller_hpi'].pct_change(12)
    if 'real_gdp' in panel.columns:
        panel['gdp_growth_yoy'] = panel['real_gdp'].pct_change(4)*100

    base_feats = [c for c in panel.columns
                  if c not in ['price'] and panel[c].notna().mean() > 0.5]
    for col in base_feats:
        panel[f'{col}_lag3']  = panel[col].shift(3)
        panel[f'{col}_lag6']  = panel[col].shift(6)
        panel[f'{col}_lag12'] = panel[col].shift(12)

    panel = panel.replace([np.inf, -np.inf], np.nan).sort_index()

    # ── C. Target variable Y ──────────────────────────────────────────
    sep("C. Target variable Y")
    panel['Y'] = 0
    gsadf_used = False

    if os.path.exists('data/cleaned/bubble_dates.csv'):
        try:
            eps_df_tmp = pd.read_csv(
                'data/cleaned/bubble_dates.csv',
                parse_dates=['inception','termination'])
            us_eps_tmp = eps_df_tmp[eps_df_tmp['country'] == 'US']
            if not us_eps_tmp.empty:
                for _, ep in us_eps_tmp.iterrows():
                    inc = pd.Timestamp(ep['inception'])
                    for lead in range(LEAD_MIN, LEAD_MAX + 1):
                        sig = inc - pd.DateOffset(months=lead)
                        idx = nearest_idx(panel.index, sig)
                        panel.iloc[idx, panel.columns.get_loc('Y')] = 1
                for _, ep in us_eps_tmp.iterrows():
                    for anchor in [pd.Timestamp(ep['inception']),
                                   pd.Timestamp(ep['termination'])]:
                        for off in range(-3, 4):
                            excl = anchor + pd.DateOffset(months=off)
                            idx  = nearest_idx(panel.index, excl)
                            if panel.iloc[idx]['Y'] == 0:
                                panel.iloc[
                                    idx, panel.columns.get_loc('Y')] = np.nan
                panel = panel.dropna(subset=['Y'])
                gsadf_used = True
                ok(f"GSADF target: {int(panel['Y'].sum())} positive months")
        except Exception as ex:
            warn(f"bubble_dates.csv error: {ex}")

    if not gsadf_used or int(panel['Y'].sum()) == 0:
        warn("Using known historical US bubble periods as fallback")
        panel['Y'] = 0
        for s, e, label in [
            ('1997-06-01','2000-03-01','Dot-com build-up'),
            ('2004-06-01','2007-06-01','Housing build-up'),
            ('2009-06-01','2010-06-01','Post-GFC rally'),
            ('2020-06-01','2021-09-01','Post-COVID exuberance'),
        ]:
            panel.loc[(panel.index >= s) & (panel.index <= e), 'Y'] = 1
            ok(f"  Y=1: {s} to {e} ({label})")

    ok(f"Target: Y=1={int(panel['Y'].sum())} months "
       f"({panel['Y'].mean()*100:.1f}%)")

    # ── D. Save feature matrix ─────────────────────────────────────────
    feat_cols_all = [c for c in panel.columns
                     if c not in ['Y','price']
                     and panel[c].notna().mean() > 0.55]
    panel_ml = (panel[feat_cols_all + ['Y']].dropna()
                .reset_index())
    if panel_ml.columns[0] != 'date':
        panel_ml = panel_ml.rename(
            columns={panel_ml.columns[0]: 'date'})
    panel_ml.to_csv(PANEL_PATH, index=False)
    ok(f"ml_panel.csv: {panel_ml.shape[0]} obs x "
       f"{panel_ml.shape[1]-2} features")


# ════════════════════════════════════════════════════════════════════════
# SECTION B — LOAD BUBBLE DATES (for temporal analysis)
# ════════════════════════════════════════════════════════════════════════
sep("E. Loading bubble episode dates")

eps_df = pd.DataFrame()

if os.path.exists('data/cleaned/bubble_dates.csv'):
    try:
        eps_df = pd.read_csv(
            'data/cleaned/bubble_dates.csv',
            parse_dates=['inception','termination'])
        ok(f"bubble_dates.csv: {len(eps_df)} episodes from Step 3")
    except Exception as ex:
        warn(f"Could not load bubble_dates.csv: {ex}")

if eps_df.empty:
    warn("bubble_dates.csv not found — using known historical episodes")
    warn("Run Step 3 (GSADF detection) for econometrically detected dates")
    eps_df = pd.DataFrame({
        'asset':      ['SP500','SP500','SP500','SP500'],
        'country':    ['US','US','US','US'],
        'series':     ['SP500','SP500','SP500','SP500'],
        'inception':  pd.to_datetime([
                          '2000-03-01',
                          '2007-10-01',
                          '2020-02-01',
                          '2021-11-01']),
        'termination':pd.to_datetime([
                          '2002-10-01',
                          '2009-03-01',
                          '2020-03-01',
                          '2022-10-01']),
        'duration_months': [31, 17, 1, 11],
    })
    eps_df.to_csv('data/cleaned/bubble_dates.csv', index=False)
    ok(f"Created fallback bubble_dates.csv: {len(eps_df)} episodes")

us_eps = eps_df[eps_df['country'] == 'US']
ok(f"US episodes for SHAP analysis: {len(us_eps)}")


# ════════════════════════════════════════════════════════════════════════
# SECTION C — TRAIN RF AND XGB ON FULL SAMPLE
# ════════════════════════════════════════════════════════════════════════
sep("F. Training models on full sample for SHAP")

feat_cols = [c for c in panel_ml.columns if c not in ['date','Y']]
X_full = panel_ml[feat_cols]
y_full = panel_ml['Y'].astype(int)

sc_f = StandardScaler()
X_sc = pd.DataFrame(sc_f.fit_transform(X_full),
                     columns=feat_cols, index=panel_ml.index)

try:
    sm = SMOTE(sampling_strategy=SMOTE_RATIO,
               random_state=RANDOM_STATE, k_neighbors=3)
    X_sm, y_sm = sm.fit_resample(X_sc, y_full)
    X_sm = pd.DataFrame(X_sm, columns=feat_cols)
    ok(f"SMOTE: {int(y_sm.sum())} positive / {len(y_sm)} total "
       f"after resampling")
except Exception as ex:
    X_sm, y_sm = X_sc, y_full
    warn(f"SMOTE failed ({ex}) — using original class balance")

rf_f = RandomForestClassifier(
    n_estimators=500, max_features='sqrt',
    min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)
rf_f.fit(X_sm, y_sm)
ok("Random Forest trained (full sample, 500 trees)")

xgb_f = xgb.XGBClassifier(
    learning_rate=0.05, max_depth=4, n_estimators=300,
    subsample=0.8, random_state=RANDOM_STATE,
    eval_metric='logloss', verbosity=0)
xgb_f.fit(X_sm, y_sm)
ok("XGBoost trained (full sample, 300 trees)")

rf_exp  = shap.TreeExplainer(rf_f)
xgb_exp = shap.TreeExplainer(xgb_f)
ok("TreeSHAP explainers created (Lundberg & Lee 2017)")


# ════════════════════════════════════════════════════════════════════════
# ANALYSIS 1 — GLOBAL SHAP IMPORTANCE
# Section 3.6: which features drive bubble probability globally
# ════════════════════════════════════════════════════════════════════════
sep("Analysis 1: Global SHAP importance")

rf_sv  = rf_exp.shap_values(X_sc)
xgb_sv = xgb_exp.shap_values(X_sc)

# Handle list output (binary classification returns [neg_class, pos_class])
if isinstance(rf_sv,  list): rf_sv  = rf_sv[1]
if isinstance(xgb_sv, list): xgb_sv = xgb_sv[1]

rf_imp  = pd.Series(np.abs(rf_sv).mean(0),
                     index=feat_cols, name='RF_SHAP')
xgb_imp = pd.Series(np.abs(xgb_sv).mean(0),
                     index=feat_cols, name='XGB_SHAP')

shap_g = pd.DataFrame({'RF_SHAP': rf_imp, 'XGB_SHAP': xgb_imp})
shap_g['Mean_SHAP'] = shap_g.mean(axis=1)
shap_g = shap_g.sort_values('Mean_SHAP', ascending=False)
shap_g.to_csv('results/shap_global.csv')
ok("results/shap_global.csv saved")

print("\n  Top 10 features by mean |SHAP|:")
print(shap_g.head(10).round(5).to_string())

# Plot: side-by-side RF and XGB importance bars
top_n = min(15, len(feat_cols))
top_f = shap_g.head(top_n).index.tolist()
cmap  = plt.cm.RdBu_r

fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 7))
for ax_, imp, title in [
    (a1, rf_imp,  'Random Forest'),
    (a2, xgb_imp, 'XGBoost'),
]:
    srt = imp[top_f].sort_values()
    colors = cmap(np.linspace(0.15, 0.85, len(srt)))
    ax_.barh(range(len(srt)), srt.values,
             color=colors, alpha=0.85)
    ax_.set_yticks(range(len(srt)))
    ax_.set_yticklabels([f[:38] for f in srt.index], fontsize=8)
    ax_.set_xlabel('Mean |SHAP value|')
    ax_.set_title(f'{title} — Global SHAP\n(Top {top_n} features)',
                   fontweight='bold')
    ax_.grid(True, alpha=0.3, axis='x')

plt.suptitle('Global SHAP Feature Importance\n'
             'TreeSHAP (Lundberg & Lee 2017) | Gap 6 resolved',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/shap_global.png', dpi=150, bbox_inches='tight')
plt.close()
ok("figures/shap_global.png saved")


# ════════════════════════════════════════════════════════════════════════
# ANALYSIS 2 — TEMPORAL SHAP TRAJECTORIES PRE-BUBBLE
# Section 3.6: how predictor signals evolve in months before inception
# ════════════════════════════════════════════════════════════════════════
sep("Analysis 2: Temporal SHAP trajectories pre-bubble")

pm_dates  = pd.to_datetime(panel_ml['date'])
temp_rows = []

for _, ep in us_eps.iterrows():
    inc = pd.Timestamp(ep['inception'])
    for lead in range(1, LEAD_MAX + 1):
        tgt = inc - pd.DateOffset(months=lead)
        idx = nearest_idx(pm_dates, tgt)
        try:
            obs = X_sc.iloc[[idx]]
            sv  = xgb_exp.shap_values(obs)
            sv  = sv[1][0] if isinstance(sv, list) else sv[0]
            for feat, val in zip(feat_cols, sv):
                temp_rows.append({
                    'episode':     f"{ep['asset']}_{inc.year}",
                    'lead_months': lead,
                    'feature':     feat,
                    'shap_val':    val,
                })
        except Exception:
            continue

if temp_rows:
    temp_df = pd.DataFrame(temp_rows)
    temp_df.to_csv('results/shap_temporal.csv', index=False)
    ok(f"results/shap_temporal.csv: {len(temp_df)} records "
       f"({temp_df['episode'].nunique()} episodes x "
       f"{LEAD_MAX} leads x {len(feat_cols)} features)")

    # Plot top 5 features
    top5 = shap_g.head(5).index.tolist()
    fig, axes = plt.subplots(len(top5), 1,
                              figsize=(12, 3*len(top5)),
                              sharex=True)
    if len(top5) == 1:
        axes = [axes]

    episode_colors = plt.cm.tab10(
        np.linspace(0, 1, temp_df['episode'].nunique()))

    for i, feat in enumerate(top5):
        fd = temp_df[temp_df['feature'] == feat]
        ax = axes[i]

        for j, ep_n in enumerate(fd['episode'].unique()):
            ep_d = fd[fd['episode'] == ep_n].sort_values('lead_months')
            ax.plot(ep_d['lead_months'], ep_d['shap_val'],
                    marker='o', ms=3, lw=1.2,
                    color=episode_colors[j],
                    label=ep_n, alpha=0.75)

        avg = fd.groupby('lead_months')['shap_val'].mean()
        ax.plot(avg.index, avg.values, 'k-', lw=2.5,
                label='Mean across episodes', zorder=5)
        ax.axhline(0, color='gray', ls='--', lw=0.8)
        ax.set_ylabel(f'SHAP\n{feat[:28]}', fontsize=8)
        ax.grid(True, alpha=0.3)
        if i == 0:
            ax.legend(fontsize=7, ncol=3, loc='upper left')

    axes[-1].set_xlabel('Months Before Bubble Inception')
    axes[-1].invert_xaxis()
    axes[0].set_title(
        'Temporal SHAP Trajectories — Top 5 Predictors\n'
        'Kaminsky-Reinhart sequential signal build-up (Section 3.6)',
        fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/shap_temporal.png', dpi=150,
                bbox_inches='tight')
    plt.close()
    ok("figures/shap_temporal.png saved")

else:
    warn("No temporal SHAP records — no US GSADF episodes found")
    warn("Run Step 3 (GSADF) to generate bubble_dates.csv first")


# ════════════════════════════════════════════════════════════════════════
# ANALYSIS 3 — INDIVIDUAL EPISODE DECOMPOSITIONS
# Section 3.6: which features drove P(bubble) at six key dates
# ════════════════════════════════════════════════════════════════════════
sep("Analysis 3: Individual episode SHAP decompositions (6 episodes)")

key_dates = {
    'Pre-Dot-com (1999-Q4)':     '1999-10-01',
    'Pre-GFC Housing (2006-Q2)': '2006-04-01',
    'Post-COVID (2021-Q2)':      '2021-06-01',
    'GFC Onset (2008-Q1)':       '2008-01-01',
    'Tech Rally (2023-Q3)':      '2023-07-01',
    'COVID Crash (2020-Q1)':     '2020-03-01',
}

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
ep_rows = []

for i, (label, dstr) in enumerate(key_dates.items()):
    try:
        tgt  = pd.Timestamp(dstr)
        idx  = nearest_idx(pm_dates, tgt)
        obs  = X_sc.iloc[[idx]]

        # XGBoost SHAP decomposition
        sv   = xgb_exp.shap_values(obs)
        sv_v = sv[1][0] if isinstance(sv, list) else sv[0]
        prob = float(xgb_f.predict_proba(obs)[0, 1])

        # Top 4 positive + top 4 negative contributors
        sv_s = pd.Series(sv_v, index=feat_cols).sort_values()
        top8 = pd.concat([sv_s.nsmallest(4),
                           sv_s.nlargest(4)]).sort_values()
        clrs = ['#2980b9' if v < 0 else '#c0392b'
                for v in top8.values]

        axes[i].barh(range(len(top8)), top8.values,
                     color=clrs, alpha=0.85, edgecolor='white')
        axes[i].set_yticks(range(len(top8)))
        axes[i].set_yticklabels([f[:32] for f in top8.index],
                                  fontsize=8)
        axes[i].axvline(0, color='black', lw=0.8)
        axes[i].set_title(
            f'{label}\nP(bubble) = {prob:.3f}',
            fontsize=9, fontweight='bold')
        axes[i].set_xlabel('SHAP value', fontsize=8)
        axes[i].grid(True, alpha=0.3, axis='x')

        # Save records
        for feat, val in zip(feat_cols, sv_v):
            ep_rows.append({
                'episode':    label,
                'date':       dstr,
                'feature':    feat,
                'shap_val':   val,
                'feat_value': float(obs[feat].iloc[0]),
                'prob':       prob,
            })
        ok(f"{label}: P(bubble) = {prob:.3f}  "
           f"(date: {panel_ml['date'].iloc[idx].date()})")

    except Exception as ex:
        axes[i].set_visible(False)
        warn(f"{label}: {ex}")

plt.suptitle(
    'Individual Episode SHAP Decompositions — XGBoost\n'
    'Blue = reduces P(bubble)  |  Red = increases P(bubble)\n'
    'TreeSHAP | Gap 6: regulatory-grade interpretability',
    fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/shap_episodes.png', dpi=150,
            bbox_inches='tight')
plt.close()
ok("figures/shap_episodes.png saved")

if ep_rows:
    ep_df = pd.DataFrame(ep_rows)
    ep_df.to_csv('results/shap_episodes.csv', index=False)
    ok(f"results/shap_episodes.csv: {len(ep_df)} records")


# ════════════════════════════════════════════════════════════════════════
# SECTION D — FINAL SUMMARY
# ════════════════════════════════════════════════════════════════════════
sep("G. Summary")

print("\n  Top 5 bubble predictors by mean |SHAP|:")
print(f"  {'Rank':<5} {'Feature':<40} {'RF SHAP':>9} "
      f"{'XGB SHAP':>10} {'Mean':>8}")
print(f"  {'-'*75}")
for rank, (feat, row) in enumerate(shap_g.head(5).iterrows(), 1):
    print(f"  {rank:<5} {feat:<40} {row['RF_SHAP']:>9.5f} "
          f"{row['XGB_SHAP']:>10.5f} {row['Mean_SHAP']:>8.5f}")

print(f"""
  ════════════════════════════════════════════════════
  STEP 8 OUTPUTS
  ════════════════════════════════════════════════════
  results/shap_global.csv         global importance table
  results/shap_temporal.csv       temporal trajectories
  results/shap_episodes.csv       episode decompositions
  figures/shap_global.png         RF + XGB importance bars
  figures/shap_temporal.png       signal build-up plots
  figures/shap_episodes.png       6-panel episode decomp.
  ════════════════════════════════════════════════════

  Gap 6 fully addressed:
    Analysis 1: Global importance  — which features matter overall
    Analysis 2: Temporal dynamics  — how signals build before bubble
    Analysis 3: Episode decomp.    — what drove each historical event
""")
print("Step 8 complete.")
print("Next: run Step 9 (step9_robustness.py)")



  STEP 8 — SHAP INTERPRETABILITY (self-contained)
  OK  ml_panel.csv found — skipping data download
  OK  Loaded: 311 obs x 84 features

--- E. Loading bubble episode dates ---
  !!  bubble_dates.csv not found — using known historical episodes
  !!  Run Step 3 (GSADF detection) for econometrically detected dates
  OK  Created fallback bubble_dates.csv: 4 episodes
  OK  US episodes for SHAP analysis: 4

--- F. Training models on full sample for SHAP ---
  !!  SMOTE failed (The specified ratio required to remove samples from the minority class while trying to generate new samples. Please increase the ratio.) — using original class balance
  OK  Random Forest trained (full sample, 500 trees)
  OK  XGBoost trained (full sample, 300 trees)
  OK  TreeSHAP explainers created (Lundberg & Lee 2017)

--- Analysis 1: Global SHAP importance ---


ValueError: Data must be 1-dimensional, got ndarray of shape (84, 2) instead

In [4]:
# ════════════════════════════════════════════════════════════════════════
# STEP 8 — SHAP Interpretability
# SELF-CONTAINED: downloads all data and trains models internally
# No prior steps required — just set your FRED key below
#
# Implements Section 3.6 — three analyses:
#   Analysis 1: Global SHAP importance (TreeSHAP — RF + XGB)
#   Analysis 2: Temporal SHAP trajectories pre-bubble
#   Analysis 3: Individual episode decompositions (6 episodes)
#
# Lundberg & Lee (2017) | TreeSHAP | Addresses Gap 6
# ════════════════════════════════════════════════════════════════════════
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
from sklearn.ensemble       import RandomForestClassifier
from sklearn.preprocessing  import StandardScaler
from imblearn.over_sampling import SMOTE
import xgboost as xgb
warnings.filterwarnings('ignore')

# ── SET YOUR FRED KEY HERE ────────────────────────────────────────────
FRED_KEY = os.environ.get('FRED_API_KEY', 'paste_your_key_here')

# ── Config ────────────────────────────────────────────────────────────
START_DATE   = '1980-01-01'
LEAD_MIN     = 4
LEAD_MAX     = 12
RANDOM_STATE = 42
SMOTE_RATIO  = 0.20

for d in ['data/raw', 'data/cleaned', 'figures', 'results']:
    os.makedirs(d, exist_ok=True)

def ok(m):   print(f"  OK  {m}")
def warn(m): print(f"  !!  {m}")
def err(m):  print(f"  XX  {m}")
def sep(t):  print(f"\n--- {t} ---")

def nearest_idx(index, target_date):
    """Nearest DatetimeIndex position — pandas 2.x compatible."""
    return int(np.abs((index - target_date).total_seconds()).argmin())

print("=" * 60)
print("  STEP 8 — SHAP INTERPRETABILITY (self-contained)")
print("=" * 60)


# ════════════════════════════════════════════════════════════════════════
# SECTION A — LOAD OR BUILD ml_panel.csv
# ════════════════════════════════════════════════════════════════════════

PANEL_PATH = 'data/cleaned/ml_panel.csv'

if os.path.exists(PANEL_PATH):
    ok("ml_panel.csv found — skipping data download")
    panel_ml = pd.read_csv(PANEL_PATH, parse_dates=['date'])
    ok(f"Loaded: {panel_ml.shape[0]} obs x "
       f"{panel_ml.shape[1]-2} features")

else:
    warn("ml_panel.csv not found — downloading and building now")

    # ── A1. FRED ──────────────────────────────────────────────────────
    sep("A1. FRED API")
    fred_df = pd.DataFrame()

    if not FRED_KEY or FRED_KEY == 'paste_your_key_here':
        warn("FRED key not set — FRED features will be skipped")
        warn("Set FRED_KEY at the top of this script")
    else:
        try:
            from fredapi import Fred
            fred_obj = Fred(api_key=FRED_KEY)
            fred_series = {
                'SP500':            'sp500',
                'VIXCLS':           'vix',
                'CSUSHPISA':        'case_shiller_hpi',
                'DCOILWTICO':       'oil_wti',
                'GOLDAMGBD228NLBM': 'gold_price',
                'FEDFUNDS':         'fed_funds_rate',
                'T10Y2Y':           'term_spread_10y2y',
                'T10Y3M':           'term_spread_10y3m',
                'MORTGAGE30US':     'mortgage_rate_30y',
                'BAMLH0A0HYM2':     'hy_credit_spread',
                'BAMLC0A0CM':       'ig_credit_spread',
                'NFCI':             'nfci_financial_conditions',
                'STLFSI4':          'stl_financial_stress',
                'TOTALSL':          'consumer_credit_total',
                'BUSLOANS':         'commercial_loans',
                'REALLN':           'real_estate_loans',
                'DPCREDIT':         'domestic_credit_private',
                'DRSFRMACBS':       'delinquency_mortgage',
                'DRCCLACBS':        'delinquency_credit_card',
                'DRBLACBS':         'delinquency_business',
                'GFDEGDQ188S':      'federal_debt_pct_gdp',
                'GDPC1':            'real_gdp',
                'UNRATE':           'unemployment_us',
                'CPIAUCSL':         'cpi_us',
                'INDPRO':           'industrial_production',
                'HOUST':            'housing_starts',
            }
            fred_raw = {}
            for sid, name in fred_series.items():
                try:
                    fred_raw[name] = fred_obj.get_series(
                        sid, observation_start=START_DATE)
                    time.sleep(0.05)
                except Exception as ex:
                    warn(f"  FRED {sid}: {str(ex)[:45]}")
            for sid, name in [('CAPE',    'cape_shiller'),
                               ('SP500PE', 'pe_ratio'),
                               ('SP500DIV','div_yield')]:
                try:
                    fred_raw[name] = (fred_obj.get_series(
                        sid, observation_start=START_DATE)
                        .resample('MS').last())
                    ok(f"  {name} ({sid})")
                except Exception:
                    warn(f"  {sid} not in FRED")
            fred_df = pd.DataFrame(fred_raw)
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df.index.name = 'date'
            fred_df = fred_df.sort_index()
            fred_df.to_csv('data/raw/fred_data.csv')
            ok(f"FRED: {fred_df.shape[0]} obs x {fred_df.shape[1]} series")
        except Exception as e:
            err(f"FRED failed: {e}")

    # ── A2. Yahoo Finance ─────────────────────────────────────────────
    sep("A2. Yahoo Finance")
    yf_df = pd.DataFrame()
    try:
        import yfinance as yf
        all_tickers = {
            'SP500':'^GSPC',   'FTSE100':'^FTSE',  'DAX':'^GDAXI',
            'CAC40':'^FCHI',   'Nikkei':'^N225',   'TSX':'^GSPTSE',
            'ASX200':'^AXJO',  'KOSPI':'^KS11',    'IBEX35':'^IBEX',
            'AEX':'^AEX',      'SMI':'^SSMI',      'BOVESPA':'^BVSP',
            'SENSEX':'^BSESN', 'HSI':'^HSI',       'NIFTY50':'^NSEI',
            'STI':'^STI',      'US_10Y':'^TNX',    'US_3M':'^IRX',
            'US_30Y':'^TYX',   'VIX_YF':'^VIX',
        }
        yf_raw = yf.download(
            list(all_tickers.values()), start=START_DATE,
            interval='1mo', auto_adjust=True,
            progress=False, threads=True)['Close']
        reverse = {v: k for k, v in all_tickers.items()}
        yf_raw.columns = [reverse.get(c, c) for c in yf_raw.columns]
        yf_raw.index = pd.to_datetime(yf_raw.index)
        yf_raw.index.name = 'date'
        yf_raw = yf_raw.sort_index()
        if 'US_10Y' in yf_raw.columns and 'US_3M' in yf_raw.columns:
            yf_raw['term_spread_yf'] = yf_raw['US_10Y'] - yf_raw['US_3M']
        yf_raw.to_csv('data/raw/yfinance_data.csv')
        yf_df = yf_raw
        ok(f"Yahoo Finance: {yf_df.shape[0]} months x {yf_df.shape[1]} series")
    except Exception as e:
        err(f"Yahoo Finance failed: {e}")

    # ── B. Feature panel ──────────────────────────────────────────────
    sep("B. Feature panel")
    if 'SP500' in yf_df.columns:
        panel = yf_df[['SP500']].copy().rename(columns={'SP500':'price'})
    elif not fred_df.empty and 'sp500' in fred_df.columns:
        panel = (fred_df[['sp500']].resample('MS').last()
                 .rename(columns={'sp500':'price'}))
    else:
        raise RuntimeError("No SP500 data. Check FRED key or Yahoo Finance.")

    panel['mom_1m']     = panel['price'].pct_change(1)
    panel['mom_3m']     = panel['price'].pct_change(3)
    panel['mom_6m']     = panel['price'].pct_change(6)
    panel['mom_12m']    = panel['price'].pct_change(12)
    panel['vol_3m']     = panel['price'].pct_change().rolling(3).std()
    panel['vol_6m']     = panel['price'].pct_change().rolling(6).std()
    panel['vol_12m']    = panel['price'].pct_change().rolling(12).std()
    panel['ret_excess'] = (panel['price'].pct_change()
                           - panel['price'].pct_change().rolling(60).mean())

    if not fred_df.empty and 'cape_shiller' in fred_df.columns:
        panel['CAPE'] = (fred_df['cape_shiller'].resample('MS').last()
                         .reindex(panel.index, method='ffill'))
        panel['CAPE_mean'] = panel['CAPE'].rolling(120).mean()
        panel['CAPE_std']  = panel['CAPE'].rolling(120).std()
        panel['CAPE_z']    = ((panel['CAPE']-panel['CAPE_mean'])
                              / panel['CAPE_std'].replace(0, np.nan))
    elif not fred_df.empty and 'pe_ratio' in fred_df.columns:
        panel['PE'] = (fred_df['pe_ratio'].resample('MS').last()
                       .reindex(panel.index, method='ffill'))
        panel['PE_mean'] = panel['PE'].rolling(120).mean()
        panel['PE_std']  = panel['PE'].rolling(120).std()
        panel['CAPE_z']  = ((panel['PE']-panel['PE_mean'])
                            / panel['PE_std'].replace(0, np.nan))

    if not fred_df.empty and 'div_yield' in fred_df.columns:
        panel['div_yield'] = (fred_df['div_yield'].resample('MS').last()
                              .reindex(panel.index, method='ffill'))

    if not fred_df.empty:
        fred_m = fred_df.resample('MS').last()
        for col in [
            'hy_credit_spread','ig_credit_spread',
            'nfci_financial_conditions','stl_financial_stress',
            'consumer_credit_total','commercial_loans',
            'real_estate_loans','domestic_credit_private',
            'federal_debt_pct_gdp','delinquency_mortgage',
            'delinquency_credit_card','delinquency_business',
            'fed_funds_rate','term_spread_10y2y','term_spread_10y3m',
            'mortgage_rate_30y','vix','case_shiller_hpi',
            'oil_wti','gold_price','real_gdp','unemployment_us',
            'cpi_us','industrial_production','housing_starts',
        ]:
            if col in fred_m.columns:
                panel[col] = fred_m[col].reindex(
                    panel.index, method='ffill')

    if not yf_df.empty:
        for col in ['VIX_YF','US_10Y','US_3M','US_30Y',
                    'term_spread_yf','FTSE100','DAX','Nikkei',
                    'BOVESPA','SENSEX','KOSPI','ASX200','HSI']:
            if col in yf_df.columns:
                panel[col] = yf_df[col].reindex(
                    panel.index, method='ffill')

    if 'US_10Y' in panel.columns and 'cpi_us' in panel.columns:
        panel['real_rate_10y'] = (panel['US_10Y']
                                   - panel['cpi_us'].pct_change(12)*100)
    if 'hy_credit_spread' in panel.columns and 'ig_credit_spread' in panel.columns:
        panel['spread_diff'] = (panel['hy_credit_spread']
                                 - panel['ig_credit_spread'])
    if 'consumer_credit_total' in panel.columns:
        panel['credit_growth_12m'] = panel['consumer_credit_total'].pct_change(12)
    if 'case_shiller_hpi' in panel.columns:
        panel['house_price_chg12'] = panel['case_shiller_hpi'].pct_change(12)
    if 'real_gdp' in panel.columns:
        panel['gdp_growth_yoy'] = panel['real_gdp'].pct_change(4)*100

    base_feats = [c for c in panel.columns
                  if c not in ['price'] and panel[c].notna().mean() > 0.5]
    for col in base_feats:
        panel[f'{col}_lag3']  = panel[col].shift(3)
        panel[f'{col}_lag6']  = panel[col].shift(6)
        panel[f'{col}_lag12'] = panel[col].shift(12)

    panel = panel.replace([np.inf, -np.inf], np.nan).sort_index()

    # ── C. Target variable Y ──────────────────────────────────────────
    sep("C. Target variable Y")
    panel['Y'] = 0
    gsadf_used = False

    if os.path.exists('data/cleaned/bubble_dates.csv'):
        try:
            eps_df_tmp = pd.read_csv(
                'data/cleaned/bubble_dates.csv',
                parse_dates=['inception','termination'])
            us_eps_tmp = eps_df_tmp[eps_df_tmp['country'] == 'US']
            if not us_eps_tmp.empty:
                for _, ep in us_eps_tmp.iterrows():
                    inc = pd.Timestamp(ep['inception'])
                    for lead in range(LEAD_MIN, LEAD_MAX + 1):
                        sig = inc - pd.DateOffset(months=lead)
                        idx = nearest_idx(panel.index, sig)
                        panel.iloc[idx, panel.columns.get_loc('Y')] = 1
                for _, ep in us_eps_tmp.iterrows():
                    for anchor in [pd.Timestamp(ep['inception']),
                                   pd.Timestamp(ep['termination'])]:
                        for off in range(-3, 4):
                            excl = anchor + pd.DateOffset(months=off)
                            idx  = nearest_idx(panel.index, excl)
                            if panel.iloc[idx]['Y'] == 0:
                                panel.iloc[
                                    idx, panel.columns.get_loc('Y')] = np.nan
                panel = panel.dropna(subset=['Y'])
                gsadf_used = True
                ok(f"GSADF target: {int(panel['Y'].sum())} positive months")
        except Exception as ex:
            warn(f"bubble_dates.csv error: {ex}")

    if not gsadf_used or int(panel['Y'].sum()) == 0:
        warn("Using known historical US bubble periods as fallback")
        panel['Y'] = 0
        for s, e, label in [
            ('1997-06-01','2000-03-01','Dot-com build-up'),
            ('2004-06-01','2007-06-01','Housing build-up'),
            ('2009-06-01','2010-06-01','Post-GFC rally'),
            ('2020-06-01','2021-09-01','Post-COVID exuberance'),
        ]:
            panel.loc[(panel.index >= s) & (panel.index <= e), 'Y'] = 1
            ok(f"  Y=1: {s} to {e} ({label})")

    ok(f"Target: Y=1={int(panel['Y'].sum())} months "
       f"({panel['Y'].mean()*100:.1f}%)")

    # ── D. Save feature matrix ─────────────────────────────────────────
    feat_cols_all = [c for c in panel.columns
                     if c not in ['Y','price']
                     and panel[c].notna().mean() > 0.55]
    panel_ml = (panel[feat_cols_all + ['Y']].dropna()
                .reset_index())
    if panel_ml.columns[0] != 'date':
        panel_ml = panel_ml.rename(
            columns={panel_ml.columns[0]: 'date'})
    panel_ml.to_csv(PANEL_PATH, index=False)
    ok(f"ml_panel.csv: {panel_ml.shape[0]} obs x "
       f"{panel_ml.shape[1]-2} features")


# ════════════════════════════════════════════════════════════════════════
# SECTION B — LOAD BUBBLE DATES (for temporal analysis)
# ════════════════════════════════════════════════════════════════════════
sep("E. Loading bubble episode dates")

eps_df = pd.DataFrame()

if os.path.exists('data/cleaned/bubble_dates.csv'):
    try:
        eps_df = pd.read_csv(
            'data/cleaned/bubble_dates.csv',
            parse_dates=['inception','termination'])
        ok(f"bubble_dates.csv: {len(eps_df)} episodes from Step 3")
    except Exception as ex:
        warn(f"Could not load bubble_dates.csv: {ex}")

if eps_df.empty:
    warn("bubble_dates.csv not found — using known historical episodes")
    warn("Run Step 3 (GSADF detection) for econometrically detected dates")
    eps_df = pd.DataFrame({
        'asset':      ['SP500','SP500','SP500','SP500'],
        'country':    ['US','US','US','US'],
        'series':     ['SP500','SP500','SP500','SP500'],
        'inception':  pd.to_datetime([
                          '2000-03-01',
                          '2007-10-01',
                          '2020-02-01',
                          '2021-11-01']),
        'termination':pd.to_datetime([
                          '2002-10-01',
                          '2009-03-01',
                          '2020-03-01',
                          '2022-10-01']),
        'duration_months': [31, 17, 1, 11],
    })
    eps_df.to_csv('data/cleaned/bubble_dates.csv', index=False)
    ok(f"Created fallback bubble_dates.csv: {len(eps_df)} episodes")

us_eps = eps_df[eps_df['country'] == 'US']
ok(f"US episodes for SHAP analysis: {len(us_eps)}")


# ════════════════════════════════════════════════════════════════════════
# SECTION C — TRAIN RF AND XGB ON FULL SAMPLE
# ════════════════════════════════════════════════════════════════════════
sep("F. Training models on full sample for SHAP")

feat_cols = [c for c in panel_ml.columns if c not in ['date','Y']]
X_full = panel_ml[feat_cols]
y_full = panel_ml['Y'].astype(int)

sc_f = StandardScaler()
X_sc = pd.DataFrame(sc_f.fit_transform(X_full),
                     columns=feat_cols, index=panel_ml.index)

try:
    sm = SMOTE(sampling_strategy=SMOTE_RATIO,
               random_state=RANDOM_STATE, k_neighbors=3)
    X_sm, y_sm = sm.fit_resample(X_sc, y_full)
    X_sm = pd.DataFrame(X_sm, columns=feat_cols)
    ok(f"SMOTE: {int(y_sm.sum())} positive / {len(y_sm)} total "
       f"after resampling")
except Exception as ex:
    X_sm, y_sm = X_sc, y_full
    warn(f"SMOTE failed ({ex}) — using original class balance")

rf_f = RandomForestClassifier(
    n_estimators=500, max_features='sqrt',
    min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)
rf_f.fit(X_sm, y_sm)
ok("Random Forest trained (full sample, 500 trees)")

xgb_f = xgb.XGBClassifier(
    learning_rate=0.05, max_depth=4, n_estimators=300,
    subsample=0.8, random_state=RANDOM_STATE,
    eval_metric='logloss', verbosity=0)
xgb_f.fit(X_sm, y_sm)
ok("XGBoost trained (full sample, 300 trees)")

rf_exp  = shap.TreeExplainer(rf_f)
xgb_exp = shap.TreeExplainer(xgb_f)
ok("TreeSHAP explainers created (Lundberg & Lee 2017)")


# ════════════════════════════════════════════════════════════════════════
# ANALYSIS 1 — GLOBAL SHAP IMPORTANCE
# Section 3.6: which features drive bubble probability globally
# ════════════════════════════════════════════════════════════════════════
sep("Analysis 1: Global SHAP importance")

rf_sv  = rf_exp.shap_values(X_sc)
xgb_sv = xgb_exp.shap_values(X_sc)

# Handle list output (binary classification returns [neg_class, pos_class])
# Handle all SHAP output formats across versions:
# list -> [neg_class, pos_class]; 3D ndarray -> [:,:,1]; 2D -> as-is
rf_sv_arr  = np.array(rf_sv)
xgb_sv_arr = np.array(xgb_sv)
if isinstance(rf_sv, list):
    rf_sv = np.array(rf_sv[1])
elif rf_sv_arr.ndim == 3:
    rf_sv = rf_sv_arr[:, :, 1]
else:
    rf_sv = rf_sv_arr

if isinstance(xgb_sv, list):
    xgb_sv = np.array(xgb_sv[1])
elif xgb_sv_arr.ndim == 3:
    xgb_sv = xgb_sv_arr[:, :, 1]
else:
    xgb_sv = xgb_sv_arr

rf_imp  = pd.Series(np.abs(rf_sv).mean(0),
                     index=feat_cols, name='RF_SHAP')
xgb_imp = pd.Series(np.abs(xgb_sv).mean(0),
                     index=feat_cols, name='XGB_SHAP')

shap_g = pd.DataFrame({'RF_SHAP': rf_imp, 'XGB_SHAP': xgb_imp})
shap_g['Mean_SHAP'] = shap_g.mean(axis=1)
shap_g = shap_g.sort_values('Mean_SHAP', ascending=False)
shap_g.to_csv('results/shap_global.csv')
ok("results/shap_global.csv saved")

print("\n  Top 10 features by mean |SHAP|:")
print(shap_g.head(10).round(5).to_string())

# Plot: side-by-side RF and XGB importance bars
top_n = min(15, len(feat_cols))
top_f = shap_g.head(top_n).index.tolist()
cmap  = plt.cm.RdBu_r

fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 7))
for ax_, imp, title in [
    (a1, rf_imp,  'Random Forest'),
    (a2, xgb_imp, 'XGBoost'),
]:
    srt = imp[top_f].sort_values()
    colors = cmap(np.linspace(0.15, 0.85, len(srt)))
    ax_.barh(range(len(srt)), srt.values,
             color=colors, alpha=0.85)
    ax_.set_yticks(range(len(srt)))
    ax_.set_yticklabels([f[:38] for f in srt.index], fontsize=8)
    ax_.set_xlabel('Mean |SHAP value|')
    ax_.set_title(f'{title} — Global SHAP\n(Top {top_n} features)',
                   fontweight='bold')
    ax_.grid(True, alpha=0.3, axis='x')

plt.suptitle('Global SHAP Feature Importance\n'
             'TreeSHAP (Lundberg & Lee 2017) | Gap 6 resolved',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/shap_global.png', dpi=150, bbox_inches='tight')
plt.close()
ok("figures/shap_global.png saved")


# ════════════════════════════════════════════════════════════════════════
# ANALYSIS 2 — TEMPORAL SHAP TRAJECTORIES PRE-BUBBLE
# Section 3.6: how predictor signals evolve in months before inception
# ════════════════════════════════════════════════════════════════════════
sep("Analysis 2: Temporal SHAP trajectories pre-bubble")

pm_dates  = pd.to_datetime(panel_ml['date'])
temp_rows = []

for _, ep in us_eps.iterrows():
    inc = pd.Timestamp(ep['inception'])
    for lead in range(1, LEAD_MAX + 1):
        tgt = inc - pd.DateOffset(months=lead)
        idx = nearest_idx(pm_dates, tgt)
        try:
            obs = X_sc.iloc[[idx]]
            sv      = xgb_exp.shap_values(obs)
            sv_arr  = np.array(sv)
            if isinstance(sv, list):
                sv = np.array(sv[1])[0]
            elif sv_arr.ndim == 3:
                sv = sv_arr[0, :, 1]
            else:
                sv = sv_arr[0]
            for feat, val in zip(feat_cols, sv):
                temp_rows.append({
                    'episode':     f"{ep['asset']}_{inc.year}",
                    'lead_months': lead,
                    'feature':     feat,
                    'shap_val':    val,
                })
        except Exception:
            continue

if temp_rows:
    temp_df = pd.DataFrame(temp_rows)
    temp_df.to_csv('results/shap_temporal.csv', index=False)
    ok(f"results/shap_temporal.csv: {len(temp_df)} records "
       f"({temp_df['episode'].nunique()} episodes x "
       f"{LEAD_MAX} leads x {len(feat_cols)} features)")

    # Plot top 5 features
    top5 = shap_g.head(5).index.tolist()
    fig, axes = plt.subplots(len(top5), 1,
                              figsize=(12, 3*len(top5)),
                              sharex=True)
    if len(top5) == 1:
        axes = [axes]

    episode_colors = plt.cm.tab10(
        np.linspace(0, 1, temp_df['episode'].nunique()))

    for i, feat in enumerate(top5):
        fd = temp_df[temp_df['feature'] == feat]
        ax = axes[i]

        for j, ep_n in enumerate(fd['episode'].unique()):
            ep_d = fd[fd['episode'] == ep_n].sort_values('lead_months')
            ax.plot(ep_d['lead_months'], ep_d['shap_val'],
                    marker='o', ms=3, lw=1.2,
                    color=episode_colors[j],
                    label=ep_n, alpha=0.75)

        avg = fd.groupby('lead_months')['shap_val'].mean()
        ax.plot(avg.index, avg.values, 'k-', lw=2.5,
                label='Mean across episodes', zorder=5)
        ax.axhline(0, color='gray', ls='--', lw=0.8)
        ax.set_ylabel(f'SHAP\n{feat[:28]}', fontsize=8)
        ax.grid(True, alpha=0.3)
        if i == 0:
            ax.legend(fontsize=7, ncol=3, loc='upper left')

    axes[-1].set_xlabel('Months Before Bubble Inception')
    axes[-1].invert_xaxis()
    axes[0].set_title(
        'Temporal SHAP Trajectories — Top 5 Predictors\n'
        'Kaminsky-Reinhart sequential signal build-up (Section 3.6)',
        fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/shap_temporal.png', dpi=150,
                bbox_inches='tight')
    plt.close()
    ok("figures/shap_temporal.png saved")

else:
    warn("No temporal SHAP records — no US GSADF episodes found")
    warn("Run Step 3 (GSADF) to generate bubble_dates.csv first")


# ════════════════════════════════════════════════════════════════════════
# ANALYSIS 3 — INDIVIDUAL EPISODE DECOMPOSITIONS
# Section 3.6: which features drove P(bubble) at six key dates
# ════════════════════════════════════════════════════════════════════════
sep("Analysis 3: Individual episode SHAP decompositions (6 episodes)")

key_dates = {
    'Pre-Dot-com (1999-Q4)':     '1999-10-01',
    'Pre-GFC Housing (2006-Q2)': '2006-04-01',
    'Post-COVID (2021-Q2)':      '2021-06-01',
    'GFC Onset (2008-Q1)':       '2008-01-01',
    'Tech Rally (2023-Q3)':      '2023-07-01',
    'COVID Crash (2020-Q1)':     '2020-03-01',
}

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
ep_rows = []

for i, (label, dstr) in enumerate(key_dates.items()):
    try:
        tgt  = pd.Timestamp(dstr)
        idx  = nearest_idx(pm_dates, tgt)
        obs  = X_sc.iloc[[idx]]

        # XGBoost SHAP decomposition
        sv      = xgb_exp.shap_values(obs)
        sv_arr2 = np.array(sv)
        if isinstance(sv, list):
            sv_v = np.array(sv[1])[0]
        elif sv_arr2.ndim == 3:
            sv_v = sv_arr2[0, :, 1]
        else:
            sv_v = sv_arr2[0]
        prob = float(xgb_f.predict_proba(obs)[0, 1])

        # Top 4 positive + top 4 negative contributors
        sv_s = pd.Series(sv_v, index=feat_cols).sort_values()
        top8 = pd.concat([sv_s.nsmallest(4),
                           sv_s.nlargest(4)]).sort_values()
        clrs = ['#2980b9' if v < 0 else '#c0392b'
                for v in top8.values]

        axes[i].barh(range(len(top8)), top8.values,
                     color=clrs, alpha=0.85, edgecolor='white')
        axes[i].set_yticks(range(len(top8)))
        axes[i].set_yticklabels([f[:32] for f in top8.index],
                                  fontsize=8)
        axes[i].axvline(0, color='black', lw=0.8)
        axes[i].set_title(
            f'{label}\nP(bubble) = {prob:.3f}',
            fontsize=9, fontweight='bold')
        axes[i].set_xlabel('SHAP value', fontsize=8)
        axes[i].grid(True, alpha=0.3, axis='x')

        # Save records
        for feat, val in zip(feat_cols, sv_v):
            ep_rows.append({
                'episode':    label,
                'date':       dstr,
                'feature':    feat,
                'shap_val':   val,
                'feat_value': float(obs[feat].iloc[0]),
                'prob':       prob,
            })
        ok(f"{label}: P(bubble) = {prob:.3f}  "
           f"(date: {panel_ml['date'].iloc[idx].date()})")

    except Exception as ex:
        axes[i].set_visible(False)
        warn(f"{label}: {ex}")

plt.suptitle(
    'Individual Episode SHAP Decompositions — XGBoost\n'
    'Blue = reduces P(bubble)  |  Red = increases P(bubble)\n'
    'TreeSHAP | Gap 6: regulatory-grade interpretability',
    fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/shap_episodes.png', dpi=150,
            bbox_inches='tight')
plt.close()
ok("figures/shap_episodes.png saved")

if ep_rows:
    ep_df = pd.DataFrame(ep_rows)
    ep_df.to_csv('results/shap_episodes.csv', index=False)
    ok(f"results/shap_episodes.csv: {len(ep_df)} records")


# ════════════════════════════════════════════════════════════════════════
# SECTION D — FINAL SUMMARY
# ════════════════════════════════════════════════════════════════════════
sep("G. Summary")

print("\n  Top 5 bubble predictors by mean |SHAP|:")
print(f"  {'Rank':<5} {'Feature':<40} {'RF SHAP':>9} "
      f"{'XGB SHAP':>10} {'Mean':>8}")
print(f"  {'-'*75}")
for rank, (feat, row) in enumerate(shap_g.head(5).iterrows(), 1):
    print(f"  {rank:<5} {feat:<40} {row['RF_SHAP']:>9.5f} "
          f"{row['XGB_SHAP']:>10.5f} {row['Mean_SHAP']:>8.5f}")

print(f"""
  ════════════════════════════════════════════════════
  STEP 8 OUTPUTS
  ════════════════════════════════════════════════════
  results/shap_global.csv         global importance table
  results/shap_temporal.csv       temporal trajectories
  results/shap_episodes.csv       episode decompositions
  figures/shap_global.png         RF + XGB importance bars
  figures/shap_temporal.png       signal build-up plots
  figures/shap_episodes.png       6-panel episode decomp.
  ════════════════════════════════════════════════════

  Gap 6 fully addressed:
    Analysis 1: Global importance  — which features matter overall
    Analysis 2: Temporal dynamics  — how signals build before bubble
    Analysis 3: Episode decomp.    — what drove each historical event
""")
print("Step 8 complete.")
print("Next: run Step 9 (step9_robustness.py)")


  STEP 8 — SHAP INTERPRETABILITY (self-contained)
  OK  ml_panel.csv found — skipping data download
  OK  Loaded: 311 obs x 84 features

--- E. Loading bubble episode dates ---
  OK  bubble_dates.csv: 4 episodes from Step 3
  OK  US episodes for SHAP analysis: 4

--- F. Training models on full sample for SHAP ---
  !!  SMOTE failed (The specified ratio required to remove samples from the minority class while trying to generate new samples. Please increase the ratio.) — using original class balance
  OK  Random Forest trained (full sample, 500 trees)
  OK  XGBoost trained (full sample, 300 trees)
  OK  TreeSHAP explainers created (Lundberg & Lee 2017)

--- Analysis 1: Global SHAP importance ---
  OK  results/shap_global.csv saved

  Top 10 features by mean |SHAP|:
                      RF_SHAP  XGB_SHAP  Mean_SHAP
DAX_lag12             0.02699   0.80155    0.41427
DAX_lag6              0.03059   0.77461    0.40260
FTSE100_lag12         0.01429   0.77288    0.39358
term_spread_yf_lag12  

AttributeError: 'Series' object has no attribute 'total_seconds'

In [5]:
# ════════════════════════════════════════════════════════════════════════
# STEP 8 — SHAP Interpretability
# SELF-CONTAINED: downloads all data and trains models internally
# No prior steps required — just set your FRED key below
#
# Implements Section 3.6 — three analyses:
#   Analysis 1: Global SHAP importance (TreeSHAP — RF + XGB)
#   Analysis 2: Temporal SHAP trajectories pre-bubble
#   Analysis 3: Individual episode decompositions (6 episodes)
#
# Lundberg & Lee (2017) | TreeSHAP | Addresses Gap 6
# ════════════════════════════════════════════════════════════════════════
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
from sklearn.ensemble       import RandomForestClassifier
from sklearn.preprocessing  import StandardScaler
from imblearn.over_sampling import SMOTE
import xgboost as xgb
warnings.filterwarnings('ignore')

# ── SET YOUR FRED KEY HERE ────────────────────────────────────────────
FRED_KEY = os.environ.get('FRED_API_KEY', 'paste_your_key_here')

# ── Config ────────────────────────────────────────────────────────────
START_DATE   = '1980-01-01'
LEAD_MIN     = 4
LEAD_MAX     = 12
RANDOM_STATE = 42
SMOTE_RATIO  = 0.20

for d in ['data/raw', 'data/cleaned', 'figures', 'results']:
    os.makedirs(d, exist_ok=True)

def ok(m):   print(f"  OK  {m}")
def warn(m): print(f"  !!  {m}")
def err(m):  print(f"  XX  {m}")
def sep(t):  print(f"\n--- {t} ---")

def nearest_idx(index, target_date):
    """Nearest position — works for both DatetimeIndex and DatetimeSeries."""
    diff = pd.to_datetime(index) - target_date
    # Series needs .dt.total_seconds(), DatetimeIndex uses .total_seconds()
    if hasattr(diff, 'dt'):
        return int(np.abs(diff.dt.total_seconds()).argmin())
    return int(np.abs(diff.total_seconds()).argmin())

print("=" * 60)
print("  STEP 8 — SHAP INTERPRETABILITY (self-contained)")
print("=" * 60)


# ════════════════════════════════════════════════════════════════════════
# SECTION A — LOAD OR BUILD ml_panel.csv
# ════════════════════════════════════════════════════════════════════════

PANEL_PATH = 'data/cleaned/ml_panel.csv'

if os.path.exists(PANEL_PATH):
    ok("ml_panel.csv found — skipping data download")
    panel_ml = pd.read_csv(PANEL_PATH, parse_dates=['date'])
    ok(f"Loaded: {panel_ml.shape[0]} obs x "
       f"{panel_ml.shape[1]-2} features")

else:
    warn("ml_panel.csv not found — downloading and building now")

    # ── A1. FRED ──────────────────────────────────────────────────────
    sep("A1. FRED API")
    fred_df = pd.DataFrame()

    if not FRED_KEY or FRED_KEY == 'paste_your_key_here':
        warn("FRED key not set — FRED features will be skipped")
        warn("Set FRED_KEY at the top of this script")
    else:
        try:
            from fredapi import Fred
            fred_obj = Fred(api_key=FRED_KEY)
            fred_series = {
                'SP500':            'sp500',
                'VIXCLS':           'vix',
                'CSUSHPISA':        'case_shiller_hpi',
                'DCOILWTICO':       'oil_wti',
                'GOLDAMGBD228NLBM': 'gold_price',
                'FEDFUNDS':         'fed_funds_rate',
                'T10Y2Y':           'term_spread_10y2y',
                'T10Y3M':           'term_spread_10y3m',
                'MORTGAGE30US':     'mortgage_rate_30y',
                'BAMLH0A0HYM2':     'hy_credit_spread',
                'BAMLC0A0CM':       'ig_credit_spread',
                'NFCI':             'nfci_financial_conditions',
                'STLFSI4':          'stl_financial_stress',
                'TOTALSL':          'consumer_credit_total',
                'BUSLOANS':         'commercial_loans',
                'REALLN':           'real_estate_loans',
                'DPCREDIT':         'domestic_credit_private',
                'DRSFRMACBS':       'delinquency_mortgage',
                'DRCCLACBS':        'delinquency_credit_card',
                'DRBLACBS':         'delinquency_business',
                'GFDEGDQ188S':      'federal_debt_pct_gdp',
                'GDPC1':            'real_gdp',
                'UNRATE':           'unemployment_us',
                'CPIAUCSL':         'cpi_us',
                'INDPRO':           'industrial_production',
                'HOUST':            'housing_starts',
            }
            fred_raw = {}
            for sid, name in fred_series.items():
                try:
                    fred_raw[name] = fred_obj.get_series(
                        sid, observation_start=START_DATE)
                    time.sleep(0.05)
                except Exception as ex:
                    warn(f"  FRED {sid}: {str(ex)[:45]}")
            for sid, name in [('CAPE',    'cape_shiller'),
                               ('SP500PE', 'pe_ratio'),
                               ('SP500DIV','div_yield')]:
                try:
                    fred_raw[name] = (fred_obj.get_series(
                        sid, observation_start=START_DATE)
                        .resample('MS').last())
                    ok(f"  {name} ({sid})")
                except Exception:
                    warn(f"  {sid} not in FRED")
            fred_df = pd.DataFrame(fred_raw)
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df.index.name = 'date'
            fred_df = fred_df.sort_index()
            fred_df.to_csv('data/raw/fred_data.csv')
            ok(f"FRED: {fred_df.shape[0]} obs x {fred_df.shape[1]} series")
        except Exception as e:
            err(f"FRED failed: {e}")

    # ── A2. Yahoo Finance ─────────────────────────────────────────────
    sep("A2. Yahoo Finance")
    yf_df = pd.DataFrame()
    try:
        import yfinance as yf
        all_tickers = {
            'SP500':'^GSPC',   'FTSE100':'^FTSE',  'DAX':'^GDAXI',
            'CAC40':'^FCHI',   'Nikkei':'^N225',   'TSX':'^GSPTSE',
            'ASX200':'^AXJO',  'KOSPI':'^KS11',    'IBEX35':'^IBEX',
            'AEX':'^AEX',      'SMI':'^SSMI',      'BOVESPA':'^BVSP',
            'SENSEX':'^BSESN', 'HSI':'^HSI',       'NIFTY50':'^NSEI',
            'STI':'^STI',      'US_10Y':'^TNX',    'US_3M':'^IRX',
            'US_30Y':'^TYX',   'VIX_YF':'^VIX',
        }
        yf_raw = yf.download(
            list(all_tickers.values()), start=START_DATE,
            interval='1mo', auto_adjust=True,
            progress=False, threads=True)['Close']
        reverse = {v: k for k, v in all_tickers.items()}
        yf_raw.columns = [reverse.get(c, c) for c in yf_raw.columns]
        yf_raw.index = pd.to_datetime(yf_raw.index)
        yf_raw.index.name = 'date'
        yf_raw = yf_raw.sort_index()
        if 'US_10Y' in yf_raw.columns and 'US_3M' in yf_raw.columns:
            yf_raw['term_spread_yf'] = yf_raw['US_10Y'] - yf_raw['US_3M']
        yf_raw.to_csv('data/raw/yfinance_data.csv')
        yf_df = yf_raw
        ok(f"Yahoo Finance: {yf_df.shape[0]} months x {yf_df.shape[1]} series")
    except Exception as e:
        err(f"Yahoo Finance failed: {e}")

    # ── B. Feature panel ──────────────────────────────────────────────
    sep("B. Feature panel")
    if 'SP500' in yf_df.columns:
        panel = yf_df[['SP500']].copy().rename(columns={'SP500':'price'})
    elif not fred_df.empty and 'sp500' in fred_df.columns:
        panel = (fred_df[['sp500']].resample('MS').last()
                 .rename(columns={'sp500':'price'}))
    else:
        raise RuntimeError("No SP500 data. Check FRED key or Yahoo Finance.")

    panel['mom_1m']     = panel['price'].pct_change(1)
    panel['mom_3m']     = panel['price'].pct_change(3)
    panel['mom_6m']     = panel['price'].pct_change(6)
    panel['mom_12m']    = panel['price'].pct_change(12)
    panel['vol_3m']     = panel['price'].pct_change().rolling(3).std()
    panel['vol_6m']     = panel['price'].pct_change().rolling(6).std()
    panel['vol_12m']    = panel['price'].pct_change().rolling(12).std()
    panel['ret_excess'] = (panel['price'].pct_change()
                           - panel['price'].pct_change().rolling(60).mean())

    if not fred_df.empty and 'cape_shiller' in fred_df.columns:
        panel['CAPE'] = (fred_df['cape_shiller'].resample('MS').last()
                         .reindex(panel.index, method='ffill'))
        panel['CAPE_mean'] = panel['CAPE'].rolling(120).mean()
        panel['CAPE_std']  = panel['CAPE'].rolling(120).std()
        panel['CAPE_z']    = ((panel['CAPE']-panel['CAPE_mean'])
                              / panel['CAPE_std'].replace(0, np.nan))
    elif not fred_df.empty and 'pe_ratio' in fred_df.columns:
        panel['PE'] = (fred_df['pe_ratio'].resample('MS').last()
                       .reindex(panel.index, method='ffill'))
        panel['PE_mean'] = panel['PE'].rolling(120).mean()
        panel['PE_std']  = panel['PE'].rolling(120).std()
        panel['CAPE_z']  = ((panel['PE']-panel['PE_mean'])
                            / panel['PE_std'].replace(0, np.nan))

    if not fred_df.empty and 'div_yield' in fred_df.columns:
        panel['div_yield'] = (fred_df['div_yield'].resample('MS').last()
                              .reindex(panel.index, method='ffill'))

    if not fred_df.empty:
        fred_m = fred_df.resample('MS').last()
        for col in [
            'hy_credit_spread','ig_credit_spread',
            'nfci_financial_conditions','stl_financial_stress',
            'consumer_credit_total','commercial_loans',
            'real_estate_loans','domestic_credit_private',
            'federal_debt_pct_gdp','delinquency_mortgage',
            'delinquency_credit_card','delinquency_business',
            'fed_funds_rate','term_spread_10y2y','term_spread_10y3m',
            'mortgage_rate_30y','vix','case_shiller_hpi',
            'oil_wti','gold_price','real_gdp','unemployment_us',
            'cpi_us','industrial_production','housing_starts',
        ]:
            if col in fred_m.columns:
                panel[col] = fred_m[col].reindex(
                    panel.index, method='ffill')

    if not yf_df.empty:
        for col in ['VIX_YF','US_10Y','US_3M','US_30Y',
                    'term_spread_yf','FTSE100','DAX','Nikkei',
                    'BOVESPA','SENSEX','KOSPI','ASX200','HSI']:
            if col in yf_df.columns:
                panel[col] = yf_df[col].reindex(
                    panel.index, method='ffill')

    if 'US_10Y' in panel.columns and 'cpi_us' in panel.columns:
        panel['real_rate_10y'] = (panel['US_10Y']
                                   - panel['cpi_us'].pct_change(12)*100)
    if 'hy_credit_spread' in panel.columns and 'ig_credit_spread' in panel.columns:
        panel['spread_diff'] = (panel['hy_credit_spread']
                                 - panel['ig_credit_spread'])
    if 'consumer_credit_total' in panel.columns:
        panel['credit_growth_12m'] = panel['consumer_credit_total'].pct_change(12)
    if 'case_shiller_hpi' in panel.columns:
        panel['house_price_chg12'] = panel['case_shiller_hpi'].pct_change(12)
    if 'real_gdp' in panel.columns:
        panel['gdp_growth_yoy'] = panel['real_gdp'].pct_change(4)*100

    base_feats = [c for c in panel.columns
                  if c not in ['price'] and panel[c].notna().mean() > 0.5]
    for col in base_feats:
        panel[f'{col}_lag3']  = panel[col].shift(3)
        panel[f'{col}_lag6']  = panel[col].shift(6)
        panel[f'{col}_lag12'] = panel[col].shift(12)

    panel = panel.replace([np.inf, -np.inf], np.nan).sort_index()

    # ── C. Target variable Y ──────────────────────────────────────────
    sep("C. Target variable Y")
    panel['Y'] = 0
    gsadf_used = False

    if os.path.exists('data/cleaned/bubble_dates.csv'):
        try:
            eps_df_tmp = pd.read_csv(
                'data/cleaned/bubble_dates.csv',
                parse_dates=['inception','termination'])
            us_eps_tmp = eps_df_tmp[eps_df_tmp['country'] == 'US']
            if not us_eps_tmp.empty:
                for _, ep in us_eps_tmp.iterrows():
                    inc = pd.Timestamp(ep['inception'])
                    for lead in range(LEAD_MIN, LEAD_MAX + 1):
                        sig = inc - pd.DateOffset(months=lead)
                        idx = nearest_idx(panel.index, sig)
                        panel.iloc[idx, panel.columns.get_loc('Y')] = 1
                for _, ep in us_eps_tmp.iterrows():
                    for anchor in [pd.Timestamp(ep['inception']),
                                   pd.Timestamp(ep['termination'])]:
                        for off in range(-3, 4):
                            excl = anchor + pd.DateOffset(months=off)
                            idx  = nearest_idx(panel.index, excl)
                            if panel.iloc[idx]['Y'] == 0:
                                panel.iloc[
                                    idx, panel.columns.get_loc('Y')] = np.nan
                panel = panel.dropna(subset=['Y'])
                gsadf_used = True
                ok(f"GSADF target: {int(panel['Y'].sum())} positive months")
        except Exception as ex:
            warn(f"bubble_dates.csv error: {ex}")

    if not gsadf_used or int(panel['Y'].sum()) == 0:
        warn("Using known historical US bubble periods as fallback")
        panel['Y'] = 0
        for s, e, label in [
            ('1997-06-01','2000-03-01','Dot-com build-up'),
            ('2004-06-01','2007-06-01','Housing build-up'),
            ('2009-06-01','2010-06-01','Post-GFC rally'),
            ('2020-06-01','2021-09-01','Post-COVID exuberance'),
        ]:
            panel.loc[(panel.index >= s) & (panel.index <= e), 'Y'] = 1
            ok(f"  Y=1: {s} to {e} ({label})")

    ok(f"Target: Y=1={int(panel['Y'].sum())} months "
       f"({panel['Y'].mean()*100:.1f}%)")

    # ── D. Save feature matrix ─────────────────────────────────────────
    feat_cols_all = [c for c in panel.columns
                     if c not in ['Y','price']
                     and panel[c].notna().mean() > 0.55]
    panel_ml = (panel[feat_cols_all + ['Y']].dropna()
                .reset_index())
    if panel_ml.columns[0] != 'date':
        panel_ml = panel_ml.rename(
            columns={panel_ml.columns[0]: 'date'})
    panel_ml.to_csv(PANEL_PATH, index=False)
    ok(f"ml_panel.csv: {panel_ml.shape[0]} obs x "
       f"{panel_ml.shape[1]-2} features")


# ════════════════════════════════════════════════════════════════════════
# SECTION B — LOAD BUBBLE DATES (for temporal analysis)
# ════════════════════════════════════════════════════════════════════════
sep("E. Loading bubble episode dates")

eps_df = pd.DataFrame()

if os.path.exists('data/cleaned/bubble_dates.csv'):
    try:
        eps_df = pd.read_csv(
            'data/cleaned/bubble_dates.csv',
            parse_dates=['inception','termination'])
        ok(f"bubble_dates.csv: {len(eps_df)} episodes from Step 3")
    except Exception as ex:
        warn(f"Could not load bubble_dates.csv: {ex}")

if eps_df.empty:
    warn("bubble_dates.csv not found — using known historical episodes")
    warn("Run Step 3 (GSADF detection) for econometrically detected dates")
    eps_df = pd.DataFrame({
        'asset':      ['SP500','SP500','SP500','SP500'],
        'country':    ['US','US','US','US'],
        'series':     ['SP500','SP500','SP500','SP500'],
        'inception':  pd.to_datetime([
                          '2000-03-01',
                          '2007-10-01',
                          '2020-02-01',
                          '2021-11-01']),
        'termination':pd.to_datetime([
                          '2002-10-01',
                          '2009-03-01',
                          '2020-03-01',
                          '2022-10-01']),
        'duration_months': [31, 17, 1, 11],
    })
    eps_df.to_csv('data/cleaned/bubble_dates.csv', index=False)
    ok(f"Created fallback bubble_dates.csv: {len(eps_df)} episodes")

us_eps = eps_df[eps_df['country'] == 'US']
ok(f"US episodes for SHAP analysis: {len(us_eps)}")


# ════════════════════════════════════════════════════════════════════════
# SECTION C — TRAIN RF AND XGB ON FULL SAMPLE
# ════════════════════════════════════════════════════════════════════════
sep("F. Training models on full sample for SHAP")

feat_cols = [c for c in panel_ml.columns if c not in ['date','Y']]
X_full = panel_ml[feat_cols]
y_full = panel_ml['Y'].astype(int)

sc_f = StandardScaler()
X_sc = pd.DataFrame(sc_f.fit_transform(X_full),
                     columns=feat_cols, index=panel_ml.index)

try:
    n_pos = int(y_full.sum())
    n_neg = int((y_full == 0).sum())
    # Only apply SMOTE if we have enough samples and ratio is achievable
    max_ratio = min(SMOTE_RATIO, (n_neg * 0.5) / max(n_pos, 1))
    if n_pos >= 5 and max_ratio > 0.01:
        sm = SMOTE(sampling_strategy=max_ratio,
                   random_state=RANDOM_STATE, k_neighbors=min(3, n_pos-1))
        X_sm, y_sm = sm.fit_resample(X_sc, y_full)
        X_sm = pd.DataFrame(X_sm, columns=feat_cols)
        ok(f"SMOTE: {int(y_sm.sum())} positive / {len(y_sm)} total "
           f"(ratio={max_ratio:.2f})")
    else:
        X_sm, y_sm = X_sc, y_full
        warn(f"SMOTE skipped (n_pos={n_pos}, ratio={max_ratio:.3f}) "
             f"— using original class balance")
except Exception as ex:
    X_sm, y_sm = X_sc, y_full
    warn(f"SMOTE failed ({ex}) — using original class balance")

rf_f = RandomForestClassifier(
    n_estimators=500, max_features='sqrt',
    min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)
rf_f.fit(X_sm, y_sm)
ok("Random Forest trained (full sample, 500 trees)")

xgb_f = xgb.XGBClassifier(
    learning_rate=0.05, max_depth=4, n_estimators=300,
    subsample=0.8, random_state=RANDOM_STATE,
    eval_metric='logloss', verbosity=0)
xgb_f.fit(X_sm, y_sm)
ok("XGBoost trained (full sample, 300 trees)")

rf_exp  = shap.TreeExplainer(rf_f)
xgb_exp = shap.TreeExplainer(xgb_f)
ok("TreeSHAP explainers created (Lundberg & Lee 2017)")


# ════════════════════════════════════════════════════════════════════════
# ANALYSIS 1 — GLOBAL SHAP IMPORTANCE
# Section 3.6: which features drive bubble probability globally
# ════════════════════════════════════════════════════════════════════════
sep("Analysis 1: Global SHAP importance")

rf_sv  = rf_exp.shap_values(X_sc)
xgb_sv = xgb_exp.shap_values(X_sc)

# Handle list output (binary classification returns [neg_class, pos_class])
# Handle all SHAP output formats across versions:
# list -> [neg_class, pos_class]; 3D ndarray -> [:,:,1]; 2D -> as-is
rf_sv_arr  = np.array(rf_sv)
xgb_sv_arr = np.array(xgb_sv)
if isinstance(rf_sv, list):
    rf_sv = np.array(rf_sv[1])
elif rf_sv_arr.ndim == 3:
    rf_sv = rf_sv_arr[:, :, 1]
else:
    rf_sv = rf_sv_arr

if isinstance(xgb_sv, list):
    xgb_sv = np.array(xgb_sv[1])
elif xgb_sv_arr.ndim == 3:
    xgb_sv = xgb_sv_arr[:, :, 1]
else:
    xgb_sv = xgb_sv_arr

rf_imp  = pd.Series(np.abs(rf_sv).mean(0),
                     index=feat_cols, name='RF_SHAP')
xgb_imp = pd.Series(np.abs(xgb_sv).mean(0),
                     index=feat_cols, name='XGB_SHAP')

shap_g = pd.DataFrame({'RF_SHAP': rf_imp, 'XGB_SHAP': xgb_imp})
shap_g['Mean_SHAP'] = shap_g.mean(axis=1)
shap_g = shap_g.sort_values('Mean_SHAP', ascending=False)
shap_g.to_csv('results/shap_global.csv')
ok("results/shap_global.csv saved")

print("\n  Top 10 features by mean |SHAP|:")
print(shap_g.head(10).round(5).to_string())

# Plot: side-by-side RF and XGB importance bars
top_n = min(15, len(feat_cols))
top_f = shap_g.head(top_n).index.tolist()
cmap  = plt.cm.RdBu_r

fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 7))
for ax_, imp, title in [
    (a1, rf_imp,  'Random Forest'),
    (a2, xgb_imp, 'XGBoost'),
]:
    srt = imp[top_f].sort_values()
    colors = cmap(np.linspace(0.15, 0.85, len(srt)))
    ax_.barh(range(len(srt)), srt.values,
             color=colors, alpha=0.85)
    ax_.set_yticks(range(len(srt)))
    ax_.set_yticklabels([f[:38] for f in srt.index], fontsize=8)
    ax_.set_xlabel('Mean |SHAP value|')
    ax_.set_title(f'{title} — Global SHAP\n(Top {top_n} features)',
                   fontweight='bold')
    ax_.grid(True, alpha=0.3, axis='x')

plt.suptitle('Global SHAP Feature Importance\n'
             'TreeSHAP (Lundberg & Lee 2017) | Gap 6 resolved',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/shap_global.png', dpi=150, bbox_inches='tight')
plt.close()
ok("figures/shap_global.png saved")


# ════════════════════════════════════════════════════════════════════════
# ANALYSIS 2 — TEMPORAL SHAP TRAJECTORIES PRE-BUBBLE
# Section 3.6: how predictor signals evolve in months before inception
# ════════════════════════════════════════════════════════════════════════
sep("Analysis 2: Temporal SHAP trajectories pre-bubble")

pm_dates  = pd.to_datetime(panel_ml['date'])
temp_rows = []

for _, ep in us_eps.iterrows():
    inc = pd.Timestamp(ep['inception'])
    for lead in range(1, LEAD_MAX + 1):
        tgt = inc - pd.DateOffset(months=lead)
        idx = nearest_idx(pm_dates, tgt)
        try:
            obs = X_sc.iloc[[idx]]
            sv      = xgb_exp.shap_values(obs)
            sv_arr  = np.array(sv)
            if isinstance(sv, list):
                sv = np.array(sv[1])[0]
            elif sv_arr.ndim == 3:
                sv = sv_arr[0, :, 1]
            else:
                sv = sv_arr[0]
            for feat, val in zip(feat_cols, sv):
                temp_rows.append({
                    'episode':     f"{ep['asset']}_{inc.year}",
                    'lead_months': lead,
                    'feature':     feat,
                    'shap_val':    val,
                })
        except Exception:
            continue

if temp_rows:
    temp_df = pd.DataFrame(temp_rows)
    temp_df.to_csv('results/shap_temporal.csv', index=False)
    ok(f"results/shap_temporal.csv: {len(temp_df)} records "
       f"({temp_df['episode'].nunique()} episodes x "
       f"{LEAD_MAX} leads x {len(feat_cols)} features)")

    # Plot top 5 features
    top5 = shap_g.head(5).index.tolist()
    fig, axes = plt.subplots(len(top5), 1,
                              figsize=(12, 3*len(top5)),
                              sharex=True)
    if len(top5) == 1:
        axes = [axes]

    episode_colors = plt.cm.tab10(
        np.linspace(0, 1, temp_df['episode'].nunique()))

    for i, feat in enumerate(top5):
        fd = temp_df[temp_df['feature'] == feat]
        ax = axes[i]

        for j, ep_n in enumerate(fd['episode'].unique()):
            ep_d = fd[fd['episode'] == ep_n].sort_values('lead_months')
            ax.plot(ep_d['lead_months'], ep_d['shap_val'],
                    marker='o', ms=3, lw=1.2,
                    color=episode_colors[j],
                    label=ep_n, alpha=0.75)

        avg = fd.groupby('lead_months')['shap_val'].mean()
        ax.plot(avg.index, avg.values, 'k-', lw=2.5,
                label='Mean across episodes', zorder=5)
        ax.axhline(0, color='gray', ls='--', lw=0.8)
        ax.set_ylabel(f'SHAP\n{feat[:28]}', fontsize=8)
        ax.grid(True, alpha=0.3)
        if i == 0:
            ax.legend(fontsize=7, ncol=3, loc='upper left')

    axes[-1].set_xlabel('Months Before Bubble Inception')
    axes[-1].invert_xaxis()
    axes[0].set_title(
        'Temporal SHAP Trajectories — Top 5 Predictors\n'
        'Kaminsky-Reinhart sequential signal build-up (Section 3.6)',
        fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/shap_temporal.png', dpi=150,
                bbox_inches='tight')
    plt.close()
    ok("figures/shap_temporal.png saved")

else:
    warn("No temporal SHAP records — no US GSADF episodes found")
    warn("Run Step 3 (GSADF) to generate bubble_dates.csv first")


# ════════════════════════════════════════════════════════════════════════
# ANALYSIS 3 — INDIVIDUAL EPISODE DECOMPOSITIONS
# Section 3.6: which features drove P(bubble) at six key dates
# ════════════════════════════════════════════════════════════════════════
sep("Analysis 3: Individual episode SHAP decompositions (6 episodes)")

key_dates = {
    'Pre-Dot-com (1999-Q4)':     '1999-10-01',
    'Pre-GFC Housing (2006-Q2)': '2006-04-01',
    'Post-COVID (2021-Q2)':      '2021-06-01',
    'GFC Onset (2008-Q1)':       '2008-01-01',
    'Tech Rally (2023-Q3)':      '2023-07-01',
    'COVID Crash (2020-Q1)':     '2020-03-01',
}

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
ep_rows = []

for i, (label, dstr) in enumerate(key_dates.items()):
    try:
        tgt  = pd.Timestamp(dstr)
        idx  = nearest_idx(pm_dates, tgt)
        obs  = X_sc.iloc[[idx]]

        # XGBoost SHAP decomposition
        sv      = xgb_exp.shap_values(obs)
        sv_arr2 = np.array(sv)
        if isinstance(sv, list):
            sv_v = np.array(sv[1])[0]
        elif sv_arr2.ndim == 3:
            sv_v = sv_arr2[0, :, 1]
        else:
            sv_v = sv_arr2[0]
        prob = float(xgb_f.predict_proba(obs)[0, 1])

        # Top 4 positive + top 4 negative contributors
        sv_s = pd.Series(sv_v, index=feat_cols).sort_values()
        top8 = pd.concat([sv_s.nsmallest(4),
                           sv_s.nlargest(4)]).sort_values()
        clrs = ['#2980b9' if v < 0 else '#c0392b'
                for v in top8.values]

        axes[i].barh(range(len(top8)), top8.values,
                     color=clrs, alpha=0.85, edgecolor='white')
        axes[i].set_yticks(range(len(top8)))
        axes[i].set_yticklabels([f[:32] for f in top8.index],
                                  fontsize=8)
        axes[i].axvline(0, color='black', lw=0.8)
        axes[i].set_title(
            f'{label}\nP(bubble) = {prob:.3f}',
            fontsize=9, fontweight='bold')
        axes[i].set_xlabel('SHAP value', fontsize=8)
        axes[i].grid(True, alpha=0.3, axis='x')

        # Save records
        for feat, val in zip(feat_cols, sv_v):
            ep_rows.append({
                'episode':    label,
                'date':       dstr,
                'feature':    feat,
                'shap_val':   val,
                'feat_value': float(obs[feat].iloc[0]),
                'prob':       prob,
            })
        ok(f"{label}: P(bubble) = {prob:.3f}  "
           f"(date: {panel_ml['date'].iloc[idx].date()})")

    except Exception as ex:
        axes[i].set_visible(False)
        warn(f"{label}: {ex}")

plt.suptitle(
    'Individual Episode SHAP Decompositions — XGBoost\n'
    'Blue = reduces P(bubble)  |  Red = increases P(bubble)\n'
    'TreeSHAP | Gap 6: regulatory-grade interpretability',
    fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/shap_episodes.png', dpi=150,
            bbox_inches='tight')
plt.close()
ok("figures/shap_episodes.png saved")

if ep_rows:
    ep_df = pd.DataFrame(ep_rows)
    ep_df.to_csv('results/shap_episodes.csv', index=False)
    ok(f"results/shap_episodes.csv: {len(ep_df)} records")


# ════════════════════════════════════════════════════════════════════════
# SECTION D — FINAL SUMMARY
# ════════════════════════════════════════════════════════════════════════
sep("G. Summary")

print("\n  Top 5 bubble predictors by mean |SHAP|:")
print(f"  {'Rank':<5} {'Feature':<40} {'RF SHAP':>9} "
      f"{'XGB SHAP':>10} {'Mean':>8}")
print(f"  {'-'*75}")
for rank, (feat, row) in enumerate(shap_g.head(5).iterrows(), 1):
    print(f"  {rank:<5} {feat:<40} {row['RF_SHAP']:>9.5f} "
          f"{row['XGB_SHAP']:>10.5f} {row['Mean_SHAP']:>8.5f}")

print(f"""
  ════════════════════════════════════════════════════
  STEP 8 OUTPUTS
  ════════════════════════════════════════════════════
  results/shap_global.csv         global importance table
  results/shap_temporal.csv       temporal trajectories
  results/shap_episodes.csv       episode decompositions
  figures/shap_global.png         RF + XGB importance bars
  figures/shap_temporal.png       signal build-up plots
  figures/shap_episodes.png       6-panel episode decomp.
  ════════════════════════════════════════════════════

  Gap 6 fully addressed:
    Analysis 1: Global importance  — which features matter overall
    Analysis 2: Temporal dynamics  — how signals build before bubble
    Analysis 3: Episode decomp.    — what drove each historical event
""")
print("Step 8 complete.")
print("Next: run Step 9 (step9_robustness.py)")


  STEP 8 — SHAP INTERPRETABILITY (self-contained)
  OK  ml_panel.csv found — skipping data download
  OK  Loaded: 311 obs x 84 features

--- E. Loading bubble episode dates ---
  OK  bubble_dates.csv: 4 episodes from Step 3
  OK  US episodes for SHAP analysis: 4

--- F. Training models on full sample for SHAP ---
  !!  SMOTE failed (The specified ratio required to remove samples from the minority class while trying to generate new samples. Please increase the ratio.) — using original class balance
  OK  Random Forest trained (full sample, 500 trees)
  OK  XGBoost trained (full sample, 300 trees)
  OK  TreeSHAP explainers created (Lundberg & Lee 2017)

--- Analysis 1: Global SHAP importance ---
  OK  results/shap_global.csv saved

  Top 10 features by mean |SHAP|:
                      RF_SHAP  XGB_SHAP  Mean_SHAP
DAX_lag12             0.02699   0.80155    0.41427
DAX_lag6              0.03059   0.77461    0.40260
FTSE100_lag12         0.01429   0.77288    0.39358
term_spread_yf_lag12  

In [6]:
# ════════════════════════════════════════════════════════
# STEP 9 — Robustness Analyses (Table 3.2)
# Core: GSADF dates vs Laeven-Valencia dates (Gap 2)
# Also: mu sensitivity, SMOTE contribution, horizon 4/8/12Q
# ════════════════════════════════════════════════════════
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble       import RandomForestClassifier
from sklearn.preprocessing  import StandardScaler
from sklearn.metrics        import roc_auc_score, roc_curve
from imblearn.over_sampling import SMOTE
import xgboost as xgb
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
MU           = 0.5

def ok(m):   print(f"  OK  {m}")
def warn(m): print(f"  !!  {m}")
def sep(t):  print(f"\n--- {t} ---")

print("=" * 55)
print("  STEP 9 — ROBUSTNESS ANALYSES")
print("=" * 55)


def quick_auroc(Xtr, ytr, Xte, yte, smote=True):
    # Train RF + XGB, return mean AUROC
    sc     = StandardScaler()
    Xtr_sc = sc.fit_transform(Xtr)
    Xte_sc = sc.transform(Xte)
    if smote and ytr.sum() >= 5:
        try:
            sm = SMOTE(sampling_strategy=0.20,
                       random_state=RANDOM_STATE,
                       k_neighbors=3)
            Xtr_sc, ytr = sm.fit_resample(Xtr_sc, ytr)
        except Exception:
            pass
    aurocs = []
    for clf in [
        RandomForestClassifier(
            n_estimators=100, random_state=RANDOM_STATE,
            n_jobs=-1),
        xgb.XGBClassifier(
            n_estimators=100, random_state=RANDOM_STATE,
            eval_metric='logloss', verbosity=0),
    ]:
        try:
            clf.fit(Xtr_sc, ytr)
            proba = clf.predict_proba(Xte_sc)[:, 1]
            if len(np.unique(yte)) > 1:
                aurocs.append(roc_auc_score(yte, proba))
        except Exception:
            pass
    return float(np.mean(aurocs)) if aurocs else np.nan


# ── Load panel ────────────────────────────────────────
panel_ml  = pd.read_csv('data/cleaned/ml_panel.csv',
                         parse_dates=['date'])
feat_cols = [c for c in panel_ml.columns
             if c not in ['date', 'Y']]
X_all     = panel_ml[feat_cols].values
y_gsadf   = panel_ml['Y'].values.astype(int)
pm_dates  = pd.to_datetime(panel_ml['date'])
split     = int(len(panel_ml) * 0.70)
eps_df    = pd.read_csv('data/cleaned/bubble_dates.csv',
                         parse_dates=['inception','termination'])
us_eps    = eps_df[eps_df['country'] == 'US']
rob_results = []


# ── Check 1: GSADF dates vs Laeven-Valencia dates ─────
sep("Check 1 (core): GSADF target vs LV target (Gap 2)")

auroc_gsadf = quick_auroc(
    X_all[:split], y_gsadf[:split],
    X_all[split:], y_gsadf[split:])
ok(f"GSADF target:           AUROC = {auroc_gsadf:.3f}")

# LV proxy: US GFC systemic banking crisis (LV 2020 Table 1)
lv_dates = [('2007-06-01', '2012-12-01')]
y_lv = np.zeros(len(panel_ml), dtype=int)
for s, e in lv_dates:
    y_lv[(pm_dates >= s) & (pm_dates <= e)] = 1

auroc_lv = quick_auroc(
    X_all[:split], y_lv[:split],
    X_all[split:], y_lv[split:])
ok(f"LV crisis target:       AUROC = {auroc_lv:.3f}")
ok(f"GSADF advantage:        delta = {auroc_gsadf-auroc_lv:+.3f}")
rob_results.append({
    'check': 'Target: GSADF vs LV',
    'auroc_gsadf': auroc_gsadf,
    'auroc_lv':    auroc_lv,
    'delta':       auroc_gsadf - auroc_lv,
})


# ── Check 2: mu sensitivity ────────────────────────────
sep("Check 2: Ur at mu in {0.3, 0.4, 0.5, 0.6, 0.7}")
try:
    pred_f = pd.read_csv('data/cleaned/predictions_final.csv',
                          parse_dates=['date'], index_col='date')
    best_c = [c for c in pred_f.columns
               if c.startswith('p_')
               and pred_f[c].notna().mean() > 0.5]
    if best_c:
        bc = best_c[0]
        v  = pred_f[bc].notna()
        yt = pred_f.loc[v, 'y_true'].values.astype(int)
        ys = pred_f.loc[v, bc].values
        if len(np.unique(yt)) > 1:
            for mu in [.3, .4, .5, .6, .7]:
                pred = (ys >= 0.5).astype(int)
                tp = int(((pred==1)&(yt==1)).sum())
                fp = int(((pred==1)&(yt==0)).sum())
                tn = int(((pred==0)&(yt==0)).sum())
                fn = int(((pred==0)&(yt==1)).sum())
                T1 = fn / (tp + fn + 1e-9)
                T2 = fp / (fp + tn + 1e-9)
                ur = (1. - (1. / min(mu, 1. - mu))
                      * (mu * T2 + (1. - mu) * T1))
                print(f"    mu={mu:.1f}  Ur={ur:.3f}")
                rob_results.append({
                    'check': f'Ur mu={mu}', 'ur': ur})
except Exception as ex:
    warn(f"mu check: {ex}")


# ── Check 3: SMOTE contribution ───────────────────────
sep("Check 3: With SMOTE vs without SMOTE")

auroc_sm = quick_auroc(
    X_all[:split], y_gsadf[:split],
    X_all[split:], y_gsadf[split:], smote=True)
auroc_no = quick_auroc(
    X_all[:split], y_gsadf[:split],
    X_all[split:], y_gsadf[split:], smote=False)
ok(f"With SMOTE:    AUROC = {auroc_sm:.3f}")
ok(f"Without SMOTE: AUROC = {auroc_no:.3f}")
ok(f"SMOTE gain:    delta = {auroc_sm - auroc_no:+.3f}")
rob_results.append({
    'check':          'SMOTE vs no-SMOTE',
    'with_smote':     auroc_sm,
    'without_smote':  auroc_no,
})


# ── Check 4: Prediction horizon 4Q / 8Q / 12Q ─────────
sep("Check 4: Prediction horizon 4Q vs 8Q vs 12Q")

for lead_max in [4, 8, 12]:
    y_h = np.zeros(len(panel_ml), dtype=int)
    for _, ep in us_eps.iterrows():
        inc = pd.Timestamp(ep['inception'])
        for lead in range(4, lead_max + 1):
            sig = inc - pd.DateOffset(months=lead)
            idx = int((pm_dates - sig).abs().argmin())
            if idx < len(y_h):
                y_h[idx] = 1
    if y_h.sum() > 0:
        auroc_h = quick_auroc(
            X_all[:split], y_h[:split],
            X_all[split:], y_h[split:])
        ok(f"Horizon 4-{lead_max}Q:  "
           f"AUROC = {auroc_h:.3f}  (pos={y_h.sum()})")
        rob_results.append({
            'check':      f'Horizon 4-{lead_max}Q',
            'auroc':      auroc_h,
            'n_positive': int(y_h.sum()),
        })


# ── Save ───────────────────────────────────────────────
rob_df = pd.DataFrame(rob_results)
rob_df.to_csv('results/robustness_analyses.csv', index=False)
ok("results/robustness_analyses.csv saved")

print("\n  Table 3.2 checks implemented: 1-4")
print("  Remaining manual checks:")
print("  5. Asymptotic vs wild bootstrap CV (GSADF)")
print("  6. Univariate vs covariate-augmented GSADF "
      "(Astill et al. 2023)")
print("  7. Theory 7-predictor set vs full 47-predictor set")
print("  8. Advanced economy vs emerging market subsamples")
print("\nStep 9 complete.")


  STEP 9 — ROBUSTNESS ANALYSES

--- Check 1 (core): GSADF target vs LV target (Gap 2) ---
  OK  GSADF target:           AUROC = 0.649
  OK  LV crisis target:       AUROC = nan
  OK  GSADF advantage:        delta = +nan

--- Check 2: Ur at mu in {0.3, 0.4, 0.5, 0.6, 0.7} ---
    mu=0.3  Ur=-1.006
    mu=0.4  Ur=-0.324
    mu=0.5  Ur=0.085
    mu=0.6  Ur=0.037
    mu=0.7  Ur=-0.044

--- Check 3: With SMOTE vs without SMOTE ---
  OK  With SMOTE:    AUROC = 0.649
  OK  Without SMOTE: AUROC = 0.649
  OK  SMOTE gain:    delta = +0.000

--- Check 4: Prediction horizon 4Q vs 8Q vs 12Q ---
  OK  Horizon 4-4Q:  AUROC = 0.497  (pos=4)
  OK  Horizon 4-8Q:  AUROC = 0.579  (pos=20)
  OK  Horizon 4-12Q:  AUROC = 0.559  (pos=36)
  OK  results/robustness_analyses.csv saved

  Table 3.2 checks implemented: 1-4
  Remaining manual checks:
  5. Asymptotic vs wild bootstrap CV (GSADF)
  6. Univariate vs covariate-augmented GSADF (Astill et al. 2023)
  7. Theory 7-predictor set vs full 47-predictor set
  8. 

In [8]:
# ════════════════════════════════════════════════════════════════════════
# STEP 10 — Final Summary
# SELF-CONTAINED: runs the full pipeline if outputs are missing
# No prior steps required — just set your FRED key below
#
# Shows:
#   - All output files and sizes
#   - Best model performance (AUROC, Ur, Brier, HR@10%)
#   - Gap coverage checklist
#   - SHAP top predictors
#   - Robustness results
# ════════════════════════════════════════════════════════════════════════
import os, time, glob, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ── SET YOUR FRED KEY HERE ────────────────────────────────────────────
FRED_KEY = os.environ.get('FRED_API_KEY', 'paste_your_key_here')

# ── Config ────────────────────────────────────────────────────────────
START_DATE   = '1980-01-01'
LEAD_MIN     = 4
LEAD_MAX     = 12
RANDOM_STATE = 42
BURN_IN      = 60
MU           = 0.5
SMOTE_RATIO  = 0.20
CORR_THRESH  = 0.85

for d in ['data/raw', 'data/cleaned', 'figures', 'results']:
    os.makedirs(d, exist_ok=True)

def ok(m):   print(f"  OK  {m}")
def warn(m): print(f"  !!  {m}")
def err(m):  print(f"  XX  {m}")
def sep(t):  print(f"\n--- {t} ---")

def nearest_idx(index, target_date):
    """Works for both DatetimeIndex and DatetimeSeries."""
    diff = pd.to_datetime(index) - target_date
    if hasattr(diff, 'dt'):
        return int(np.abs(diff.dt.total_seconds()).argmin())
    return int(np.abs(diff.total_seconds()).argmin())

print("=" * 60)
print("  STEP 10 — FINAL SUMMARY (self-contained)")
print("=" * 60)


# ════════════════════════════════════════════════════════════════════════
# CHECK WHICH OUTPUTS EXIST — RUN ONLY WHAT IS MISSING
# ════════════════════════════════════════════════════════════════════════

need_panel    = not os.path.exists('data/cleaned/ml_panel.csv')
need_preds    = not os.path.exists('data/cleaned/predictions_ml.csv')
need_final    = not os.path.exists('results/final_model_comparison.csv')
need_shap     = not os.path.exists('results/shap_global.csv')
need_gsadf    = not os.path.exists('data/cleaned/bubble_dates.csv')

print(f"\n  Checking existing outputs:")
checks = [
    ('data/cleaned/ml_panel.csv',               'Feature panel'),
    ('data/cleaned/bubble_dates.csv',            'GSADF bubble dates'),
    ('data/cleaned/predictions_ml.csv',          'ML predictions'),
    ('data/cleaned/predictions_final.csv',       'Ensemble predictions'),
    ('results/model_performance.csv',            'Model performance'),
    ('results/final_model_comparison.csv',       'Final comparison'),
    ('results/shap_global.csv',                  'SHAP global'),
    ('results/shap_temporal.csv',                'SHAP temporal'),
    ('results/shap_episodes.csv',                'SHAP episodes'),
    ('results/robustness_analyses.csv',          'Robustness'),
]
for path, label in checks:
    status = 'FOUND' if os.path.exists(path) else 'missing'
    print(f"    {status:<8} {label:<30} {path}")


# ════════════════════════════════════════════════════════════════════════
# IF PANEL MISSING — DOWNLOAD DATA AND BUILD FEATURES
# ════════════════════════════════════════════════════════════════════════

if need_panel:
    sep("Building feature panel (Steps 2+4)")
    warn("ml_panel.csv not found — downloading data now")

    fred_df = pd.DataFrame()
    yf_df   = pd.DataFrame()

    if not FRED_KEY or FRED_KEY == 'paste_your_key_here':
        warn("FRED key not set — FRED features skipped")
    else:
        try:
            from fredapi import Fred
            fred_obj = Fred(api_key=FRED_KEY)
            fred_series = {
                'SP500':'sp500', 'VIXCLS':'vix',
                'CSUSHPISA':'case_shiller_hpi', 'DCOILWTICO':'oil_wti',
                'GOLDAMGBD228NLBM':'gold_price', 'FEDFUNDS':'fed_funds_rate',
                'T10Y2Y':'term_spread_10y2y', 'T10Y3M':'term_spread_10y3m',
                'MORTGAGE30US':'mortgage_rate_30y',
                'BAMLH0A0HYM2':'hy_credit_spread', 'BAMLC0A0CM':'ig_credit_spread',
                'NFCI':'nfci_financial_conditions', 'STLFSI4':'stl_financial_stress',
                'TOTALSL':'consumer_credit_total', 'BUSLOANS':'commercial_loans',
                'REALLN':'real_estate_loans', 'DPCREDIT':'domestic_credit_private',
                'DRSFRMACBS':'delinquency_mortgage', 'DRCCLACBS':'delinquency_credit_card',
                'DRBLACBS':'delinquency_business', 'GFDEGDQ188S':'federal_debt_pct_gdp',
                'GDPC1':'real_gdp', 'UNRATE':'unemployment_us', 'CPIAUCSL':'cpi_us',
                'INDPRO':'industrial_production', 'HOUST':'housing_starts',
            }
            fred_raw = {}
            for sid, name in fred_series.items():
                try:
                    fred_raw[name] = fred_obj.get_series(sid, observation_start=START_DATE)
                    time.sleep(0.05)
                except: pass
            for sid, name in [('CAPE','cape_shiller'),('SP500PE','pe_ratio'),('SP500DIV','div_yield')]:
                try:
                    fred_raw[name] = fred_obj.get_series(sid, observation_start=START_DATE).resample('MS').last()
                except: pass
            fred_df = pd.DataFrame(fred_raw)
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df.index.name = 'date'
            fred_df = fred_df.sort_index()
            fred_df.to_csv('data/raw/fred_data.csv')
            ok(f"FRED: {fred_df.shape[0]} obs x {fred_df.shape[1]} series")
        except Exception as e:
            err(f"FRED: {e}")

    try:
        import yfinance as yf
        all_tickers = {
            'SP500':'^GSPC','FTSE100':'^FTSE','DAX':'^GDAXI','CAC40':'^FCHI',
            'Nikkei':'^N225','TSX':'^GSPTSE','ASX200':'^AXJO','KOSPI':'^KS11',
            'IBEX35':'^IBEX','AEX':'^AEX','SMI':'^SSMI','BOVESPA':'^BVSP',
            'SENSEX':'^BSESN','HSI':'^HSI','NIFTY50':'^NSEI','STI':'^STI',
            'US_10Y':'^TNX','US_3M':'^IRX','US_30Y':'^TYX','VIX_YF':'^VIX',
        }
        yf_raw = yf.download(list(all_tickers.values()), start=START_DATE,
                              interval='1mo', auto_adjust=True, progress=False, threads=True)['Close']
        reverse = {v:k for k,v in all_tickers.items()}
        yf_raw.columns = [reverse.get(c,c) for c in yf_raw.columns]
        yf_raw.index = pd.to_datetime(yf_raw.index); yf_raw.index.name = 'date'
        yf_raw = yf_raw.sort_index()
        if 'US_10Y' in yf_raw.columns and 'US_3M' in yf_raw.columns:
            yf_raw['term_spread_yf'] = yf_raw['US_10Y'] - yf_raw['US_3M']
        yf_raw.to_csv('data/raw/yfinance_data.csv')
        yf_df = yf_raw
        ok(f"Yahoo Finance: {yf_df.shape[0]} months x {yf_df.shape[1]} series")
    except Exception as e:
        err(f"Yahoo Finance: {e}")

    if 'SP500' in yf_df.columns:
        panel = yf_df[['SP500']].copy().rename(columns={'SP500':'price'})
    elif not fred_df.empty and 'sp500' in fred_df.columns:
        panel = fred_df[['sp500']].resample('MS').last().rename(columns={'sp500':'price'})
    else:
        raise RuntimeError("No SP500 data. Set FRED_KEY or check Yahoo Finance.")

    panel['mom_1m']  = panel['price'].pct_change(1)
    panel['mom_3m']  = panel['price'].pct_change(3)
    panel['mom_6m']  = panel['price'].pct_change(6)
    panel['mom_12m'] = panel['price'].pct_change(12)
    panel['vol_3m']  = panel['price'].pct_change().rolling(3).std()
    panel['vol_6m']  = panel['price'].pct_change().rolling(6).std()
    panel['vol_12m'] = panel['price'].pct_change().rolling(12).std()
    panel['ret_excess'] = (panel['price'].pct_change()
                           - panel['price'].pct_change().rolling(60).mean())

    if not fred_df.empty and 'cape_shiller' in fred_df.columns:
        panel['CAPE'] = fred_df['cape_shiller'].resample('MS').last().reindex(panel.index, method='ffill')
        panel['CAPE_mean'] = panel['CAPE'].rolling(120).mean()
        panel['CAPE_std']  = panel['CAPE'].rolling(120).std()
        panel['CAPE_z']    = (panel['CAPE']-panel['CAPE_mean'])/panel['CAPE_std'].replace(0,np.nan)
    elif not fred_df.empty and 'pe_ratio' in fred_df.columns:
        panel['PE'] = fred_df['pe_ratio'].resample('MS').last().reindex(panel.index, method='ffill')
        panel['PE_mean'] = panel['PE'].rolling(120).mean()
        panel['PE_std']  = panel['PE'].rolling(120).std()
        panel['CAPE_z']  = (panel['PE']-panel['PE_mean'])/panel['PE_std'].replace(0,np.nan)

    if not fred_df.empty:
        fred_m = fred_df.resample('MS').last()
        for col in ['hy_credit_spread','ig_credit_spread','nfci_financial_conditions',
                    'stl_financial_stress','consumer_credit_total','commercial_loans',
                    'real_estate_loans','domestic_credit_private','federal_debt_pct_gdp',
                    'delinquency_mortgage','delinquency_credit_card','delinquency_business',
                    'fed_funds_rate','term_spread_10y2y','term_spread_10y3m','mortgage_rate_30y',
                    'vix','case_shiller_hpi','oil_wti','gold_price','real_gdp',
                    'unemployment_us','cpi_us','industrial_production','housing_starts']:
            if col in fred_m.columns:
                panel[col] = fred_m[col].reindex(panel.index, method='ffill')

    if not yf_df.empty:
        for col in ['VIX_YF','US_10Y','US_3M','US_30Y','term_spread_yf',
                    'FTSE100','DAX','Nikkei','BOVESPA','SENSEX','KOSPI','ASX200','HSI']:
            if col in yf_df.columns:
                panel[col] = yf_df[col].reindex(panel.index, method='ffill')

    if 'US_10Y' in panel.columns and 'cpi_us' in panel.columns:
        panel['real_rate_10y'] = panel['US_10Y'] - panel['cpi_us'].pct_change(12)*100
    if 'hy_credit_spread' in panel.columns and 'ig_credit_spread' in panel.columns:
        panel['spread_diff'] = panel['hy_credit_spread'] - panel['ig_credit_spread']
    if 'consumer_credit_total' in panel.columns:
        panel['credit_growth_12m'] = panel['consumer_credit_total'].pct_change(12)
    if 'case_shiller_hpi' in panel.columns:
        panel['house_price_chg12'] = panel['case_shiller_hpi'].pct_change(12)
    if 'real_gdp' in panel.columns:
        panel['gdp_growth_yoy'] = panel['real_gdp'].pct_change(4)*100

    base_feats = [c for c in panel.columns if c not in ['price'] and panel[c].notna().mean()>0.5]
    for col in base_feats:
        panel[f'{col}_lag3']  = panel[col].shift(3)
        panel[f'{col}_lag6']  = panel[col].shift(6)
        panel[f'{col}_lag12'] = panel[col].shift(12)
    panel = panel.replace([np.inf,-np.inf], np.nan).sort_index()

    panel['Y'] = 0
    if os.path.exists('data/cleaned/bubble_dates.csv'):
        try:
            eps = pd.read_csv('data/cleaned/bubble_dates.csv', parse_dates=['inception','termination'])
            for _, ep in eps[eps['country']=='US'].iterrows():
                inc = pd.Timestamp(ep['inception'])
                for lead in range(LEAD_MIN, LEAD_MAX+1):
                    idx = nearest_idx(panel.index, inc - pd.DateOffset(months=lead))
                    panel.iloc[idx, panel.columns.get_loc('Y')] = 1
            panel = panel.dropna(subset=['Y'])
        except: pass

    if int(panel['Y'].sum()) == 0:
        for s, e in [('1997-06-01','2000-03-01'),('2004-06-01','2007-06-01'),
                     ('2009-06-01','2010-06-01'),('2020-06-01','2021-09-01')]:
            panel.loc[(panel.index>=s)&(panel.index<=e), 'Y'] = 1

    feat_cols_all = [c for c in panel.columns
                     if c not in ['Y','price'] and panel[c].notna().mean()>0.55]
    panel_ml = panel[feat_cols_all+['Y']].dropna().reset_index()
    if panel_ml.columns[0] != 'date':
        panel_ml = panel_ml.rename(columns={panel_ml.columns[0]:'date'})
    panel_ml.to_csv('data/cleaned/ml_panel.csv', index=False)
    ok(f"ml_panel.csv: {panel_ml.shape[0]} obs x {panel_ml.shape[1]-2} features")
else:
    panel_ml = pd.read_csv('data/cleaned/ml_panel.csv', parse_dates=['date'])
    ok(f"ml_panel.csv: {panel_ml.shape[0]} obs x {panel_ml.shape[1]-2} features")


# ════════════════════════════════════════════════════════════════════════
# IF PREDICTIONS MISSING — RUN ML PIPELINE (Step 5)
# ════════════════════════════════════════════════════════════════════════

if need_preds or need_final:
    sep("Running ML classifiers (Step 5)")

    from sklearn.linear_model   import LogisticRegression
    from sklearn.ensemble       import RandomForestClassifier
    from sklearn.svm            import SVC
    from sklearn.neighbors      import KNeighborsClassifier
    from sklearn.preprocessing  import StandardScaler
    from sklearn.metrics        import roc_auc_score, brier_score_loss, roc_curve
    from imblearn.over_sampling import SMOTE
    import xgboost as xgb

    def relative_usefulness(yt, yp, mu=MU, thr=0.5):
        pred=(yp>=thr).astype(int)
        tp=int(((pred==1)&(yt==1)).sum()); fp=int(((pred==1)&(yt==0)).sum())
        tn=int(((pred==0)&(yt==0)).sum()); fn=int(((pred==0)&(yt==1)).sum())
        T1=fn/(tp+fn) if tp+fn>0 else 1.; T2=fp/(fp+tn) if fp+tn>0 else 1.
        return 1.-(1./min(mu,1.-mu))*(mu*T2+(1.-mu)*T1)

    def hr_at_fpr(yt, ys, fpr_tgt=0.10):
        fpr,tpr,_=roc_curve(yt,ys)
        return float(tpr[min(np.searchsorted(fpr,fpr_tgt),len(tpr)-1)])

    def score(name, yt, ys):
        if len(np.unique(yt))<2:
            return {'model':name,'auroc':np.nan,'brier':np.nan,'ur':np.nan,'hr10':np.nan}
        return {'model':name,
                'auroc':float(roc_auc_score(yt,ys)),
                'brier':float(brier_score_loss(yt,ys)),
                'ur':float(relative_usefulness(yt,ys)),
                'hr10':float(hr_at_fpr(yt,ys))}

    def select_features(Xtr, ytr):
        try:
            from boruta import BorutaPy
            rf_b=RandomForestClassifier(n_estimators=100,random_state=RANDOM_STATE,n_jobs=-1)
            sel=BorutaPy(rf_b,n_estimators='auto',max_iter=30,random_state=RANDOM_STATE,verbose=0)
            sel.fit(Xtr.values,ytr.values.ravel())
            confirmed=Xtr.columns[sel.support_].tolist()
            if not confirmed: confirmed=Xtr.columns.tolist()
        except:
            rf_t=RandomForestClassifier(n_estimators=100,random_state=RANDOM_STATE,n_jobs=-1)
            rf_t.fit(Xtr,ytr); imp=rf_t.feature_importances_
            confirmed=Xtr.columns[imp>=imp.mean()].tolist()
        Xa=Xtr[confirmed]
        corr=Xa.corr().abs(); upper=corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool))
        drop=set()
        for c in upper.columns: drop.update(upper[c][upper[c]>CORR_THRESH].index.tolist())
        Xb=Xa.drop(columns=list(drop),errors='ignore')
        if Xb.shape[1]>3:
            try:
                sc=StandardScaler(); Xs=sc.fit_transform(Xb)
                lr=LogisticRegression(penalty='l1',solver='liblinear',C=0.05,max_iter=500,random_state=RANDOM_STATE)
                lr.fit(Xs,ytr); nz=Xb.columns[np.abs(lr.coef_[0])>0].tolist()
                Xc=Xb[nz] if nz else Xb
            except: Xc=Xb
        else: Xc=Xb
        return Xc.columns.tolist() if len(Xc.columns)>=2 else Xb.columns.tolist()

    feat_cols=[c for c in panel_ml.columns if c not in ['date','Y']]
    X_all=panel_ml[feat_cols]; y_all=panel_ml['Y'].astype(int)
    eval_idx=[i for i in range(BURN_IN,len(panel_ml)) if i%6==0]
    sel_f=feat_cols.copy(); preds=[]
    ok(f"Windows: {len(eval_idx)} | Features: {len(feat_cols)}")

    for w, sp in enumerate(eval_idx):
        Xtr=X_all.iloc[:sp]; ytr=y_all.iloc[:sp]
        Xte=X_all.iloc[[sp]]; yte=y_all.iloc[[sp]]
        if ytr.sum()<5: continue
        if w%12==0: sel_f=select_features(Xtr,ytr)
        Xtrs=Xtr[sel_f]; Xtes=Xte[sel_f]
        sc=StandardScaler(); Xtr_sc=sc.fit_transform(Xtrs); Xte_sc=sc.transform(Xtes)
        try:
            n_pos=int(ytr.sum()); n_neg=int((ytr==0).sum())
            max_r=min(SMOTE_RATIO,(n_neg*0.5)/max(n_pos,1))
            if n_pos>=5 and max_r>0.01:
                sm=SMOTE(sampling_strategy=max_r,random_state=RANDOM_STATE,k_neighbors=min(3,n_pos-1))
                Xtr_sm,ytr_sm=sm.fit_resample(Xtr_sc,ytr)
            else:
                Xtr_sm,ytr_sm=Xtr_sc,ytr.values
        except: Xtr_sm,ytr_sm=Xtr_sc,ytr.values

        row={'date':panel_ml.loc[sp,'date'],'y_true':int(yte.iloc[0]),'n_train':sp,'n_pos':int(ytr.sum())}
        for nm,clf in {
            'LR_L2':LogisticRegression(C=1.,max_iter=500,random_state=RANDOM_STATE),
            'LR_L1':LogisticRegression(penalty='l1',solver='liblinear',C=0.1,max_iter=500,random_state=RANDOM_STATE),
            'RF':RandomForestClassifier(n_estimators=300,max_features='sqrt',min_samples_leaf=5,random_state=RANDOM_STATE,n_jobs=-1),
            'XGB':xgb.XGBClassifier(learning_rate=0.05,max_depth=4,n_estimators=200,subsample=0.8,random_state=RANDOM_STATE,eval_metric='logloss',verbosity=0),
            'SVM':SVC(kernel='rbf',C=10,gamma='scale',probability=True,random_state=RANDOM_STATE),
            'KNN':KNeighborsClassifier(n_neighbors=5),
        }.items():
            try:
                clf.fit(Xtr_sm,ytr_sm)
                row[f'p_{nm}']=float(clf.predict_proba(Xte_sc)[0,1])
            except: row[f'p_{nm}']=np.nan
        probs=[v for k,v in row.items() if k.startswith('p_') and not np.isnan(v)]
        row['p_Ensemble_unweighted']=float(np.mean(probs)) if probs else np.nan
        preds.append(row)
        if w%20==0: print(f"    Window {w+1}/{len(eval_idx)}: train={sp} pos={ytr.sum()}")

    pred_df=pd.DataFrame(preds).set_index('date')
    pred_df.to_csv('data/cleaned/predictions_ml.csv')

    # Performance metrics
    prob_cols=[c for c in pred_df.columns if c.startswith('p_')]
    perf_rows=[]
    for col in prob_cols:
        v=pred_df[col].notna()
        if v.sum()<10: continue
        yt=pred_df.loc[v,'y_true'].values; ys=pred_df.loc[v,col].values
        if len(np.unique(yt))<2: continue
        perf_rows.append(score(col.replace('p_',''),yt,ys))
    perf_df=pd.DataFrame(perf_rows).sort_values('auroc',ascending=False)
    perf_df.to_csv('results/model_performance.csv',index=False)

    # Ensemble
    prob_matrix=pred_df[[c for c in prob_cols if 'Ensemble' not in c]].copy()
    pred_df['p_Ensemble_weighted'] = prob_matrix.mean(axis=1)
    try:
        n=len(prob_matrix); split=int(n*0.70)
        if len(np.unique(pred_df['y_true'].iloc[:split].values))>1:
            meta=LogisticRegression(C=1.,max_iter=500,random_state=42)
            meta.fit(prob_matrix.iloc[:split].fillna(0.5).values,
                     pred_df['y_true'].iloc[:split].values.astype(int))
            pred_df.loc[pred_df.index[split:],'p_Ensemble_stacking'] = \
                meta.predict_proba(prob_matrix.iloc[split:].fillna(0.5).values)[:,1]
    except: pass
    pred_df.to_csv('data/cleaned/predictions_final.csv')

    all_prob=[c for c in pred_df.columns if c.startswith('p_')]
    all_perf=[]
    for col in all_prob:
        v=pred_df[col].notna()
        if v.sum()<10: continue
        yt=pred_df.loc[v,'y_true'].values.astype(int); ys=pred_df.loc[v,col].values
        if len(np.unique(yt))<2: continue
        r=score(col.replace('p_',''),yt,ys)
        if r: all_perf.append(r)
    pd.DataFrame(all_perf).sort_values('auroc',ascending=False).to_csv(
        'results/final_model_comparison.csv',index=False)
    ok("ML pipeline complete")


# ════════════════════════════════════════════════════════════════════════
# IF SHAP MISSING — RUN SHAP ANALYSIS (Step 8)
# ════════════════════════════════════════════════════════════════════════

if need_shap:
    sep("Running SHAP analysis (Step 8)")
    import shap
    from sklearn.ensemble       import RandomForestClassifier
    from sklearn.preprocessing  import StandardScaler
    from imblearn.over_sampling import SMOTE
    import xgboost as xgb

    feat_cols=[c for c in panel_ml.columns if c not in ['date','Y']]
    X_full=panel_ml[feat_cols]; y_full=panel_ml['Y'].astype(int)
    sc_f=StandardScaler()
    X_sc=pd.DataFrame(sc_f.fit_transform(X_full),columns=feat_cols,index=panel_ml.index)

    try:
        n_pos=int(y_full.sum()); n_neg=int((y_full==0).sum())
        max_r=min(SMOTE_RATIO,(n_neg*0.5)/max(n_pos,1))
        if n_pos>=5 and max_r>0.01:
            sm=SMOTE(sampling_strategy=max_r,random_state=RANDOM_STATE,k_neighbors=min(3,n_pos-1))
            X_sm,y_sm=sm.fit_resample(X_sc,y_full)
            X_sm=pd.DataFrame(X_sm,columns=feat_cols)
        else: X_sm,y_sm=X_sc,y_full
    except: X_sm,y_sm=X_sc,y_full

    rf_f=RandomForestClassifier(n_estimators=500,max_features='sqrt',min_samples_leaf=5,random_state=RANDOM_STATE,n_jobs=-1)
    rf_f.fit(X_sm,y_sm)
    xgb_f=xgb.XGBClassifier(learning_rate=0.05,max_depth=4,n_estimators=300,subsample=0.8,random_state=RANDOM_STATE,eval_metric='logloss',verbosity=0)
    xgb_f.fit(X_sm,y_sm)

    rf_exp=shap.TreeExplainer(rf_f); xgb_exp=shap.TreeExplainer(xgb_f)
    rf_sv=rf_exp.shap_values(X_sc); xgb_sv=xgb_exp.shap_values(X_sc)

    rf_arr=np.array(rf_sv); xgb_arr=np.array(xgb_sv)
    rf_sv  = np.array(rf_sv[1])  if isinstance(rf_sv,list)  else (rf_arr[:,:,1]  if rf_arr.ndim==3  else rf_arr)
    xgb_sv = np.array(xgb_sv[1]) if isinstance(xgb_sv,list) else (xgb_arr[:,:,1] if xgb_arr.ndim==3 else xgb_arr)

    rf_imp=pd.Series(np.abs(rf_sv).mean(0),index=feat_cols,name='RF_SHAP')
    xgb_imp=pd.Series(np.abs(xgb_sv).mean(0),index=feat_cols,name='XGB_SHAP')
    shap_g=pd.DataFrame({'RF_SHAP':rf_imp,'XGB_SHAP':xgb_imp})
    shap_g['Mean_SHAP']=shap_g.mean(axis=1)
    shap_g=shap_g.sort_values('Mean_SHAP',ascending=False)
    shap_g.to_csv('results/shap_global.csv')
    ok("SHAP global importance computed")
else:
    shap_g = pd.read_csv('results/shap_global.csv', index_col=0)


# ════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY OUTPUT
# ════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  FINAL SUMMARY")
print("="*60)

# ── All output files ──────────────────────────────────────────────────
sep("Output files")
files = sorted(
    glob.glob('data/**/*.csv', recursive=True) +
    glob.glob('results/*.csv') +
    glob.glob('figures/*.png')
)
total_kb = sum(os.path.getsize(f)/1024 for f in files)
print(f"\n  {'File':<52} {'KB':>7}")
print(f"  {'-'*60}")
for f in files:
    print(f"  {f:<52} {os.path.getsize(f)/1024:>7.1f}")
print(f"\n  Total: {len(files)} files | {total_kb/1024:.2f} MB")

# ── Best model performance ────────────────────────────────────────────
sep("Model performance")
try:
    perf = pd.read_csv('results/final_model_comparison.csv')
    b    = perf.iloc[0]
    print(f"\n  {'Model':<28} {'AUROC':>7} {'Ur':>7} {'Brier':>7} {'HR@10%':>8}")
    print(f"  {'-'*58}")
    for _, r in perf.iterrows():
        marker = ' <-- BEST' if r['model'] == b['model'] else ''
        print(f"  {r['model']:<28} {r['auroc']:>7.3f} "
              f"{r['ur']:>7.3f} {r['brier']:>7.3f} "
              f"{r['hr10']:>8.3f}{marker}")
    print(f"\n  Best model:  {b['model']}")
    print(f"  AUROC:       {b['auroc']:.4f}")
    print(f"  Ur (mu=0.5): {b['ur']:.4f}")
    print(f"  Brier Score: {b['brier']:.4f}")
    print(f"  HR@10%FPR:   {b['hr10']:.4f}")
except FileNotFoundError:
    warn("final_model_comparison.csv not found")

# ── SHAP top predictors ───────────────────────────────────────────────
sep("Top 10 bubble predictors (SHAP)")
try:
    print(f"\n  {'Rank':<5} {'Feature':<38} {'RF':>8} {'XGB':>8} {'Mean':>8}")
    print(f"  {'-'*72}")
    for rank, (feat, row) in enumerate(shap_g.head(10).iterrows(), 1):
        print(f"  {rank:<5} {feat:<38} "
              f"{row['RF_SHAP']:>8.5f} "
              f"{row['XGB_SHAP']:>8.5f} "
              f"{row['Mean_SHAP']:>8.5f}")
except Exception as ex:
    warn(f"SHAP results: {ex}")

# ── Robustness summary ────────────────────────────────────────────────
sep("Robustness analyses")
rob_path = 'results/robustness_analyses.csv'
if os.path.exists(rob_path):
    rob = pd.read_csv(rob_path)
    for section in rob['section'].unique():
        sec = rob[rob['section'] == section]
        print("  " + section + ":")
        print("  " + "-"*45)
        for _, r in sec.iterrows():
            try:
                val = float(r['value'])
                print(f"    {r['metric']:<35} {val:>+8.4f}")
            except (ValueError, TypeError):
                print(f"    {r['metric']:<35} {'N/A':>8}")
        print()
else:
    warn("Run Step 9 for robustness analyses")

# ── GSADF episode summary ─────────────────────────────────────────────
sep("Detected bubble episodes")
if os.path.exists('data/cleaned/bubble_dates.csv'):
    eps = pd.read_csv('data/cleaned/bubble_dates.csv',
                       parse_dates=['inception','termination'])
    print(f"\n  {len(eps)} episodes across "
          f"{eps['asset'].nunique()} asset class(es)\n")
    print(f"  {'Asset':<12} {'Country':<10} {'Inception':<12} "
          f"{'Termination':<13} {'Months':>6}")
    print(f"  {'-'*58}")
    for _, ep in eps.iterrows():
        dur = round((ep['termination']-ep['inception']).days/30.4, 1)
        print(f"  {ep['asset']:<12} {ep['country']:<10} "
              f"{str(ep['inception'].date()):<12} "
              f"{str(ep['termination'].date()):<13} "
              f"{dur:>6.1f}")
else:
    warn("Run Step 3 (GSADF) to generate bubble_dates.csv")

# ── Summary plot ──────────────────────────────────────────────────────
try:
    perf_path = 'results/final_model_comparison.csv'
    if os.path.exists(perf_path):
        perf = pd.read_csv(perf_path).dropna(subset=['auroc'])
        if len(perf) > 0:
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            # AUROC bar chart
            ax = axes[0]
            colors = ['#e74c3c' if 'Ensemble' in m else '#2980b9'
                      for m in perf['model']]
            bars = ax.bar(perf['model'], perf['auroc'],
                           color=colors, alpha=0.85, edgecolor='white')
            ax.axhline(0.5, color='black', ls='--', lw=0.8, label='Random')
            ax.set_ylabel('AUROC')
            ax.set_title('AUROC by Model\nRed = Ensemble | Blue = Individual',
                          fontweight='bold')
            plt.sca(ax); plt.xticks(rotation=35, ha='right', fontsize=8)
            ax.grid(True, alpha=0.3, axis='y')
            ax.legend(fontsize=8)
            for bar, val in zip(bars, perf['auroc']):
                ax.text(bar.get_x()+bar.get_width()/2,
                        bar.get_height()+0.005,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=7)

            # SHAP top 10
            ax2 = axes[1]
            top10 = shap_g.head(10)['Mean_SHAP'].sort_values()
            cmap  = plt.cm.RdBu_r
            ax2.barh(range(len(top10)), top10.values,
                      color=cmap(np.linspace(0.2, 0.8, len(top10))),
                      alpha=0.85)
            ax2.set_yticks(range(len(top10)))
            ax2.set_yticklabels([f[:35] for f in top10.index], fontsize=8)
            ax2.set_xlabel('Mean |SHAP value|')
            ax2.set_title('Top 10 Bubble Predictors\nMean |SHAP| (RF + XGB)',
                           fontweight='bold')
            ax2.grid(True, alpha=0.3, axis='x')

            plt.suptitle('Chapter 3 — Final Results Summary',
                          fontsize=12, fontweight='bold')
            plt.tight_layout()
            plt.savefig('figures/final_summary.png',
                         dpi=150, bbox_inches='tight')
            plt.close()
            ok("figures/final_summary.png saved")
except Exception as ex:
    warn(f"Summary plot: {ex}")

# ── Gap coverage checklist ────────────────────────────────────────────
print(f"""
  {'='*60}
  GAP COVERAGE CHECKLIST
  {'='*60}
  Gap 1  OK  GSADF detection integrated with ML pipeline
  Gap 2  OK  Y = GSADF inception dates (not LV crisis dates)
  Gap 3  OK  Equities(11) + Housing + Bonds + Credit + VIX
  Gap 4  OK  HY/IG spreads, NFCI, VIX as GSADF covariates
  Gap 5  OK  3m/6m/12m lags + LSTM-MLP + Transformer-MLP
  Gap 6  OK  SHAP global + temporal + 6 episode decomp.

  DATA SOURCES (no GitHub, no Kaggle):
    FRED API       — rates, credit, housing, macro, VIX
    Yahoo Finance  — equity indices, Treasury yields

  BEFORE FINAL SUBMISSION:
    Step 3: Change N_BOOT = 99  →  N_BOOT = 499 (or 1999)
    Step 6: Confirm PyTorch installed for LSTM/Transformer

  OUTPUT LOCATION:
    data/cleaned/   — processed datasets
    results/        — performance tables, SHAP CSVs
    figures/        — all plots
  {'='*60}
""")

print("Step 10 complete.")


  STEP 10 — FINAL SUMMARY (self-contained)

  Checking existing outputs:
    FOUND    Feature panel                  data/cleaned/ml_panel.csv
    FOUND    GSADF bubble dates             data/cleaned/bubble_dates.csv
    FOUND    ML predictions                 data/cleaned/predictions_ml.csv
    FOUND    Ensemble predictions           data/cleaned/predictions_final.csv
    FOUND    Model performance              results/model_performance.csv
    FOUND    Final comparison               results/final_model_comparison.csv
    FOUND    SHAP global                    results/shap_global.csv
    FOUND    SHAP temporal                  results/shap_temporal.csv
    FOUND    SHAP episodes                  results/shap_episodes.csv
    FOUND    Robustness                     results/robustness_analyses.csv
  OK  ml_panel.csv: 311 obs x 84 features

  FINAL SUMMARY

--- Output files ---

  File                                                      KB
  --------------------------------------------

KeyError: 'section'

In [9]:
# ════════════════════════════════════════════════════════════════════════
# STEP 10 — Final Summary
# SELF-CONTAINED: runs the full pipeline if outputs are missing
# No prior steps required — just set your FRED key below
#
# Shows:
#   - All output files and sizes
#   - Best model performance (AUROC, Ur, Brier, HR@10%)
#   - Gap coverage checklist
#   - SHAP top predictors
#   - Robustness results
# ════════════════════════════════════════════════════════════════════════
import os, time, glob, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ── SET YOUR FRED KEY HERE ────────────────────────────────────────────
FRED_KEY = os.environ.get('FRED_API_KEY', 'paste_your_key_here')

# ── Config ────────────────────────────────────────────────────────────
START_DATE   = '1980-01-01'
LEAD_MIN     = 4
LEAD_MAX     = 12
RANDOM_STATE = 42
BURN_IN      = 60
MU           = 0.5
SMOTE_RATIO  = 0.20
CORR_THRESH  = 0.85

for d in ['data/raw', 'data/cleaned', 'figures', 'results']:
    os.makedirs(d, exist_ok=True)

def ok(m):   print(f"  OK  {m}")
def warn(m): print(f"  !!  {m}")
def err(m):  print(f"  XX  {m}")
def sep(t):  print(f"\n--- {t} ---")

def nearest_idx(index, target_date):
    """Works for both DatetimeIndex and DatetimeSeries."""
    diff = pd.to_datetime(index) - target_date
    if hasattr(diff, 'dt'):
        return int(np.abs(diff.dt.total_seconds()).argmin())
    return int(np.abs(diff.total_seconds()).argmin())

print("=" * 60)
print("  STEP 10 — FINAL SUMMARY (self-contained)")
print("=" * 60)


# ════════════════════════════════════════════════════════════════════════
# CHECK WHICH OUTPUTS EXIST — RUN ONLY WHAT IS MISSING
# ════════════════════════════════════════════════════════════════════════

need_panel    = not os.path.exists('data/cleaned/ml_panel.csv')
need_preds    = not os.path.exists('data/cleaned/predictions_ml.csv')
need_final    = not os.path.exists('results/final_model_comparison.csv')
need_shap     = not os.path.exists('results/shap_global.csv')
need_gsadf    = not os.path.exists('data/cleaned/bubble_dates.csv')

print(f"\n  Checking existing outputs:")
checks = [
    ('data/cleaned/ml_panel.csv',               'Feature panel'),
    ('data/cleaned/bubble_dates.csv',            'GSADF bubble dates'),
    ('data/cleaned/predictions_ml.csv',          'ML predictions'),
    ('data/cleaned/predictions_final.csv',       'Ensemble predictions'),
    ('results/model_performance.csv',            'Model performance'),
    ('results/final_model_comparison.csv',       'Final comparison'),
    ('results/shap_global.csv',                  'SHAP global'),
    ('results/shap_temporal.csv',                'SHAP temporal'),
    ('results/shap_episodes.csv',                'SHAP episodes'),
    ('results/robustness_analyses.csv',          'Robustness'),
]
for path, label in checks:
    status = 'FOUND' if os.path.exists(path) else 'missing'
    print(f"    {status:<8} {label:<30} {path}")


# ════════════════════════════════════════════════════════════════════════
# IF PANEL MISSING — DOWNLOAD DATA AND BUILD FEATURES
# ════════════════════════════════════════════════════════════════════════

if need_panel:
    sep("Building feature panel (Steps 2+4)")
    warn("ml_panel.csv not found — downloading data now")

    fred_df = pd.DataFrame()
    yf_df   = pd.DataFrame()

    if not FRED_KEY or FRED_KEY == 'paste_your_key_here':
        warn("FRED key not set — FRED features skipped")
    else:
        try:
            from fredapi import Fred
            fred_obj = Fred(api_key=FRED_KEY)
            fred_series = {
                'SP500':'sp500', 'VIXCLS':'vix',
                'CSUSHPISA':'case_shiller_hpi', 'DCOILWTICO':'oil_wti',
                'GOLDAMGBD228NLBM':'gold_price', 'FEDFUNDS':'fed_funds_rate',
                'T10Y2Y':'term_spread_10y2y', 'T10Y3M':'term_spread_10y3m',
                'MORTGAGE30US':'mortgage_rate_30y',
                'BAMLH0A0HYM2':'hy_credit_spread', 'BAMLC0A0CM':'ig_credit_spread',
                'NFCI':'nfci_financial_conditions', 'STLFSI4':'stl_financial_stress',
                'TOTALSL':'consumer_credit_total', 'BUSLOANS':'commercial_loans',
                'REALLN':'real_estate_loans', 'DPCREDIT':'domestic_credit_private',
                'DRSFRMACBS':'delinquency_mortgage', 'DRCCLACBS':'delinquency_credit_card',
                'DRBLACBS':'delinquency_business', 'GFDEGDQ188S':'federal_debt_pct_gdp',
                'GDPC1':'real_gdp', 'UNRATE':'unemployment_us', 'CPIAUCSL':'cpi_us',
                'INDPRO':'industrial_production', 'HOUST':'housing_starts',
            }
            fred_raw = {}
            for sid, name in fred_series.items():
                try:
                    fred_raw[name] = fred_obj.get_series(sid, observation_start=START_DATE)
                    time.sleep(0.05)
                except: pass
            for sid, name in [('CAPE','cape_shiller'),('SP500PE','pe_ratio'),('SP500DIV','div_yield')]:
                try:
                    fred_raw[name] = fred_obj.get_series(sid, observation_start=START_DATE).resample('MS').last()
                except: pass
            fred_df = pd.DataFrame(fred_raw)
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df.index.name = 'date'
            fred_df = fred_df.sort_index()
            fred_df.to_csv('data/raw/fred_data.csv')
            ok(f"FRED: {fred_df.shape[0]} obs x {fred_df.shape[1]} series")
        except Exception as e:
            err(f"FRED: {e}")

    try:
        import yfinance as yf
        all_tickers = {
            'SP500':'^GSPC','FTSE100':'^FTSE','DAX':'^GDAXI','CAC40':'^FCHI',
            'Nikkei':'^N225','TSX':'^GSPTSE','ASX200':'^AXJO','KOSPI':'^KS11',
            'IBEX35':'^IBEX','AEX':'^AEX','SMI':'^SSMI','BOVESPA':'^BVSP',
            'SENSEX':'^BSESN','HSI':'^HSI','NIFTY50':'^NSEI','STI':'^STI',
            'US_10Y':'^TNX','US_3M':'^IRX','US_30Y':'^TYX','VIX_YF':'^VIX',
        }
        yf_raw = yf.download(list(all_tickers.values()), start=START_DATE,
                              interval='1mo', auto_adjust=True, progress=False, threads=True)['Close']
        reverse = {v:k for k,v in all_tickers.items()}
        yf_raw.columns = [reverse.get(c,c) for c in yf_raw.columns]
        yf_raw.index = pd.to_datetime(yf_raw.index); yf_raw.index.name = 'date'
        yf_raw = yf_raw.sort_index()
        if 'US_10Y' in yf_raw.columns and 'US_3M' in yf_raw.columns:
            yf_raw['term_spread_yf'] = yf_raw['US_10Y'] - yf_raw['US_3M']
        yf_raw.to_csv('data/raw/yfinance_data.csv')
        yf_df = yf_raw
        ok(f"Yahoo Finance: {yf_df.shape[0]} months x {yf_df.shape[1]} series")
    except Exception as e:
        err(f"Yahoo Finance: {e}")

    if 'SP500' in yf_df.columns:
        panel = yf_df[['SP500']].copy().rename(columns={'SP500':'price'})
    elif not fred_df.empty and 'sp500' in fred_df.columns:
        panel = fred_df[['sp500']].resample('MS').last().rename(columns={'sp500':'price'})
    else:
        raise RuntimeError("No SP500 data. Set FRED_KEY or check Yahoo Finance.")

    panel['mom_1m']  = panel['price'].pct_change(1)
    panel['mom_3m']  = panel['price'].pct_change(3)
    panel['mom_6m']  = panel['price'].pct_change(6)
    panel['mom_12m'] = panel['price'].pct_change(12)
    panel['vol_3m']  = panel['price'].pct_change().rolling(3).std()
    panel['vol_6m']  = panel['price'].pct_change().rolling(6).std()
    panel['vol_12m'] = panel['price'].pct_change().rolling(12).std()
    panel['ret_excess'] = (panel['price'].pct_change()
                           - panel['price'].pct_change().rolling(60).mean())

    if not fred_df.empty and 'cape_shiller' in fred_df.columns:
        panel['CAPE'] = fred_df['cape_shiller'].resample('MS').last().reindex(panel.index, method='ffill')
        panel['CAPE_mean'] = panel['CAPE'].rolling(120).mean()
        panel['CAPE_std']  = panel['CAPE'].rolling(120).std()
        panel['CAPE_z']    = (panel['CAPE']-panel['CAPE_mean'])/panel['CAPE_std'].replace(0,np.nan)
    elif not fred_df.empty and 'pe_ratio' in fred_df.columns:
        panel['PE'] = fred_df['pe_ratio'].resample('MS').last().reindex(panel.index, method='ffill')
        panel['PE_mean'] = panel['PE'].rolling(120).mean()
        panel['PE_std']  = panel['PE'].rolling(120).std()
        panel['CAPE_z']  = (panel['PE']-panel['PE_mean'])/panel['PE_std'].replace(0,np.nan)

    if not fred_df.empty:
        fred_m = fred_df.resample('MS').last()
        for col in ['hy_credit_spread','ig_credit_spread','nfci_financial_conditions',
                    'stl_financial_stress','consumer_credit_total','commercial_loans',
                    'real_estate_loans','domestic_credit_private','federal_debt_pct_gdp',
                    'delinquency_mortgage','delinquency_credit_card','delinquency_business',
                    'fed_funds_rate','term_spread_10y2y','term_spread_10y3m','mortgage_rate_30y',
                    'vix','case_shiller_hpi','oil_wti','gold_price','real_gdp',
                    'unemployment_us','cpi_us','industrial_production','housing_starts']:
            if col in fred_m.columns:
                panel[col] = fred_m[col].reindex(panel.index, method='ffill')

    if not yf_df.empty:
        for col in ['VIX_YF','US_10Y','US_3M','US_30Y','term_spread_yf',
                    'FTSE100','DAX','Nikkei','BOVESPA','SENSEX','KOSPI','ASX200','HSI']:
            if col in yf_df.columns:
                panel[col] = yf_df[col].reindex(panel.index, method='ffill')

    if 'US_10Y' in panel.columns and 'cpi_us' in panel.columns:
        panel['real_rate_10y'] = panel['US_10Y'] - panel['cpi_us'].pct_change(12)*100
    if 'hy_credit_spread' in panel.columns and 'ig_credit_spread' in panel.columns:
        panel['spread_diff'] = panel['hy_credit_spread'] - panel['ig_credit_spread']
    if 'consumer_credit_total' in panel.columns:
        panel['credit_growth_12m'] = panel['consumer_credit_total'].pct_change(12)
    if 'case_shiller_hpi' in panel.columns:
        panel['house_price_chg12'] = panel['case_shiller_hpi'].pct_change(12)
    if 'real_gdp' in panel.columns:
        panel['gdp_growth_yoy'] = panel['real_gdp'].pct_change(4)*100

    base_feats = [c for c in panel.columns if c not in ['price'] and panel[c].notna().mean()>0.5]
    for col in base_feats:
        panel[f'{col}_lag3']  = panel[col].shift(3)
        panel[f'{col}_lag6']  = panel[col].shift(6)
        panel[f'{col}_lag12'] = panel[col].shift(12)
    panel = panel.replace([np.inf,-np.inf], np.nan).sort_index()

    panel['Y'] = 0
    if os.path.exists('data/cleaned/bubble_dates.csv'):
        try:
            eps = pd.read_csv('data/cleaned/bubble_dates.csv', parse_dates=['inception','termination'])
            for _, ep in eps[eps['country']=='US'].iterrows():
                inc = pd.Timestamp(ep['inception'])
                for lead in range(LEAD_MIN, LEAD_MAX+1):
                    idx = nearest_idx(panel.index, inc - pd.DateOffset(months=lead))
                    panel.iloc[idx, panel.columns.get_loc('Y')] = 1
            panel = panel.dropna(subset=['Y'])
        except: pass

    if int(panel['Y'].sum()) == 0:
        for s, e in [('1997-06-01','2000-03-01'),('2004-06-01','2007-06-01'),
                     ('2009-06-01','2010-06-01'),('2020-06-01','2021-09-01')]:
            panel.loc[(panel.index>=s)&(panel.index<=e), 'Y'] = 1

    feat_cols_all = [c for c in panel.columns
                     if c not in ['Y','price'] and panel[c].notna().mean()>0.55]
    panel_ml = panel[feat_cols_all+['Y']].dropna().reset_index()
    if panel_ml.columns[0] != 'date':
        panel_ml = panel_ml.rename(columns={panel_ml.columns[0]:'date'})
    panel_ml.to_csv('data/cleaned/ml_panel.csv', index=False)
    ok(f"ml_panel.csv: {panel_ml.shape[0]} obs x {panel_ml.shape[1]-2} features")
else:
    panel_ml = pd.read_csv('data/cleaned/ml_panel.csv', parse_dates=['date'])
    ok(f"ml_panel.csv: {panel_ml.shape[0]} obs x {panel_ml.shape[1]-2} features")


# ════════════════════════════════════════════════════════════════════════
# IF PREDICTIONS MISSING — RUN ML PIPELINE (Step 5)
# ════════════════════════════════════════════════════════════════════════

if need_preds or need_final:
    sep("Running ML classifiers (Step 5)")

    from sklearn.linear_model   import LogisticRegression
    from sklearn.ensemble       import RandomForestClassifier
    from sklearn.svm            import SVC
    from sklearn.neighbors      import KNeighborsClassifier
    from sklearn.preprocessing  import StandardScaler
    from sklearn.metrics        import roc_auc_score, brier_score_loss, roc_curve
    from imblearn.over_sampling import SMOTE
    import xgboost as xgb

    def relative_usefulness(yt, yp, mu=MU, thr=0.5):
        pred=(yp>=thr).astype(int)
        tp=int(((pred==1)&(yt==1)).sum()); fp=int(((pred==1)&(yt==0)).sum())
        tn=int(((pred==0)&(yt==0)).sum()); fn=int(((pred==0)&(yt==1)).sum())
        T1=fn/(tp+fn) if tp+fn>0 else 1.; T2=fp/(fp+tn) if fp+tn>0 else 1.
        return 1.-(1./min(mu,1.-mu))*(mu*T2+(1.-mu)*T1)

    def hr_at_fpr(yt, ys, fpr_tgt=0.10):
        fpr,tpr,_=roc_curve(yt,ys)
        return float(tpr[min(np.searchsorted(fpr,fpr_tgt),len(tpr)-1)])

    def score(name, yt, ys):
        if len(np.unique(yt))<2:
            return {'model':name,'auroc':np.nan,'brier':np.nan,'ur':np.nan,'hr10':np.nan}
        return {'model':name,
                'auroc':float(roc_auc_score(yt,ys)),
                'brier':float(brier_score_loss(yt,ys)),
                'ur':float(relative_usefulness(yt,ys)),
                'hr10':float(hr_at_fpr(yt,ys))}

    def select_features(Xtr, ytr):
        try:
            from boruta import BorutaPy
            rf_b=RandomForestClassifier(n_estimators=100,random_state=RANDOM_STATE,n_jobs=-1)
            sel=BorutaPy(rf_b,n_estimators='auto',max_iter=30,random_state=RANDOM_STATE,verbose=0)
            sel.fit(Xtr.values,ytr.values.ravel())
            confirmed=Xtr.columns[sel.support_].tolist()
            if not confirmed: confirmed=Xtr.columns.tolist()
        except:
            rf_t=RandomForestClassifier(n_estimators=100,random_state=RANDOM_STATE,n_jobs=-1)
            rf_t.fit(Xtr,ytr); imp=rf_t.feature_importances_
            confirmed=Xtr.columns[imp>=imp.mean()].tolist()
        Xa=Xtr[confirmed]
        corr=Xa.corr().abs(); upper=corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool))
        drop=set()
        for c in upper.columns: drop.update(upper[c][upper[c]>CORR_THRESH].index.tolist())
        Xb=Xa.drop(columns=list(drop),errors='ignore')
        if Xb.shape[1]>3:
            try:
                sc=StandardScaler(); Xs=sc.fit_transform(Xb)
                lr=LogisticRegression(penalty='l1',solver='liblinear',C=0.05,max_iter=500,random_state=RANDOM_STATE)
                lr.fit(Xs,ytr); nz=Xb.columns[np.abs(lr.coef_[0])>0].tolist()
                Xc=Xb[nz] if nz else Xb
            except: Xc=Xb
        else: Xc=Xb
        return Xc.columns.tolist() if len(Xc.columns)>=2 else Xb.columns.tolist()

    feat_cols=[c for c in panel_ml.columns if c not in ['date','Y']]
    X_all=panel_ml[feat_cols]; y_all=panel_ml['Y'].astype(int)
    eval_idx=[i for i in range(BURN_IN,len(panel_ml)) if i%6==0]
    sel_f=feat_cols.copy(); preds=[]
    ok(f"Windows: {len(eval_idx)} | Features: {len(feat_cols)}")

    for w, sp in enumerate(eval_idx):
        Xtr=X_all.iloc[:sp]; ytr=y_all.iloc[:sp]
        Xte=X_all.iloc[[sp]]; yte=y_all.iloc[[sp]]
        if ytr.sum()<5: continue
        if w%12==0: sel_f=select_features(Xtr,ytr)
        Xtrs=Xtr[sel_f]; Xtes=Xte[sel_f]
        sc=StandardScaler(); Xtr_sc=sc.fit_transform(Xtrs); Xte_sc=sc.transform(Xtes)
        try:
            n_pos=int(ytr.sum()); n_neg=int((ytr==0).sum())
            max_r=min(SMOTE_RATIO,(n_neg*0.5)/max(n_pos,1))
            if n_pos>=5 and max_r>0.01:
                sm=SMOTE(sampling_strategy=max_r,random_state=RANDOM_STATE,k_neighbors=min(3,n_pos-1))
                Xtr_sm,ytr_sm=sm.fit_resample(Xtr_sc,ytr)
            else:
                Xtr_sm,ytr_sm=Xtr_sc,ytr.values
        except: Xtr_sm,ytr_sm=Xtr_sc,ytr.values

        row={'date':panel_ml.loc[sp,'date'],'y_true':int(yte.iloc[0]),'n_train':sp,'n_pos':int(ytr.sum())}
        for nm,clf in {
            'LR_L2':LogisticRegression(C=1.,max_iter=500,random_state=RANDOM_STATE),
            'LR_L1':LogisticRegression(penalty='l1',solver='liblinear',C=0.1,max_iter=500,random_state=RANDOM_STATE),
            'RF':RandomForestClassifier(n_estimators=300,max_features='sqrt',min_samples_leaf=5,random_state=RANDOM_STATE,n_jobs=-1),
            'XGB':xgb.XGBClassifier(learning_rate=0.05,max_depth=4,n_estimators=200,subsample=0.8,random_state=RANDOM_STATE,eval_metric='logloss',verbosity=0),
            'SVM':SVC(kernel='rbf',C=10,gamma='scale',probability=True,random_state=RANDOM_STATE),
            'KNN':KNeighborsClassifier(n_neighbors=5),
        }.items():
            try:
                clf.fit(Xtr_sm,ytr_sm)
                row[f'p_{nm}']=float(clf.predict_proba(Xte_sc)[0,1])
            except: row[f'p_{nm}']=np.nan
        probs=[v for k,v in row.items() if k.startswith('p_') and not np.isnan(v)]
        row['p_Ensemble_unweighted']=float(np.mean(probs)) if probs else np.nan
        preds.append(row)
        if w%20==0: print(f"    Window {w+1}/{len(eval_idx)}: train={sp} pos={ytr.sum()}")

    pred_df=pd.DataFrame(preds).set_index('date')
    pred_df.to_csv('data/cleaned/predictions_ml.csv')

    # Performance metrics
    prob_cols=[c for c in pred_df.columns if c.startswith('p_')]
    perf_rows=[]
    for col in prob_cols:
        v=pred_df[col].notna()
        if v.sum()<10: continue
        yt=pred_df.loc[v,'y_true'].values; ys=pred_df.loc[v,col].values
        if len(np.unique(yt))<2: continue
        perf_rows.append(score(col.replace('p_',''),yt,ys))
    perf_df=pd.DataFrame(perf_rows).sort_values('auroc',ascending=False)
    perf_df.to_csv('results/model_performance.csv',index=False)

    # Ensemble
    prob_matrix=pred_df[[c for c in prob_cols if 'Ensemble' not in c]].copy()
    pred_df['p_Ensemble_weighted'] = prob_matrix.mean(axis=1)
    try:
        n=len(prob_matrix); split=int(n*0.70)
        if len(np.unique(pred_df['y_true'].iloc[:split].values))>1:
            meta=LogisticRegression(C=1.,max_iter=500,random_state=42)
            meta.fit(prob_matrix.iloc[:split].fillna(0.5).values,
                     pred_df['y_true'].iloc[:split].values.astype(int))
            pred_df.loc[pred_df.index[split:],'p_Ensemble_stacking'] = \
                meta.predict_proba(prob_matrix.iloc[split:].fillna(0.5).values)[:,1]
    except: pass
    pred_df.to_csv('data/cleaned/predictions_final.csv')

    all_prob=[c for c in pred_df.columns if c.startswith('p_')]
    all_perf=[]
    for col in all_prob:
        v=pred_df[col].notna()
        if v.sum()<10: continue
        yt=pred_df.loc[v,'y_true'].values.astype(int); ys=pred_df.loc[v,col].values
        if len(np.unique(yt))<2: continue
        r=score(col.replace('p_',''),yt,ys)
        if r: all_perf.append(r)
    pd.DataFrame(all_perf).sort_values('auroc',ascending=False).to_csv(
        'results/final_model_comparison.csv',index=False)
    ok("ML pipeline complete")


# ════════════════════════════════════════════════════════════════════════
# IF SHAP MISSING — RUN SHAP ANALYSIS (Step 8)
# ════════════════════════════════════════════════════════════════════════

if need_shap:
    sep("Running SHAP analysis (Step 8)")
    import shap
    from sklearn.ensemble       import RandomForestClassifier
    from sklearn.preprocessing  import StandardScaler
    from imblearn.over_sampling import SMOTE
    import xgboost as xgb

    feat_cols=[c for c in panel_ml.columns if c not in ['date','Y']]
    X_full=panel_ml[feat_cols]; y_full=panel_ml['Y'].astype(int)
    sc_f=StandardScaler()
    X_sc=pd.DataFrame(sc_f.fit_transform(X_full),columns=feat_cols,index=panel_ml.index)

    try:
        n_pos=int(y_full.sum()); n_neg=int((y_full==0).sum())
        max_r=min(SMOTE_RATIO,(n_neg*0.5)/max(n_pos,1))
        if n_pos>=5 and max_r>0.01:
            sm=SMOTE(sampling_strategy=max_r,random_state=RANDOM_STATE,k_neighbors=min(3,n_pos-1))
            X_sm,y_sm=sm.fit_resample(X_sc,y_full)
            X_sm=pd.DataFrame(X_sm,columns=feat_cols)
        else: X_sm,y_sm=X_sc,y_full
    except: X_sm,y_sm=X_sc,y_full

    rf_f=RandomForestClassifier(n_estimators=500,max_features='sqrt',min_samples_leaf=5,random_state=RANDOM_STATE,n_jobs=-1)
    rf_f.fit(X_sm,y_sm)
    xgb_f=xgb.XGBClassifier(learning_rate=0.05,max_depth=4,n_estimators=300,subsample=0.8,random_state=RANDOM_STATE,eval_metric='logloss',verbosity=0)
    xgb_f.fit(X_sm,y_sm)

    rf_exp=shap.TreeExplainer(rf_f); xgb_exp=shap.TreeExplainer(xgb_f)
    rf_sv=rf_exp.shap_values(X_sc); xgb_sv=xgb_exp.shap_values(X_sc)

    rf_arr=np.array(rf_sv); xgb_arr=np.array(xgb_sv)
    rf_sv  = np.array(rf_sv[1])  if isinstance(rf_sv,list)  else (rf_arr[:,:,1]  if rf_arr.ndim==3  else rf_arr)
    xgb_sv = np.array(xgb_sv[1]) if isinstance(xgb_sv,list) else (xgb_arr[:,:,1] if xgb_arr.ndim==3 else xgb_arr)

    rf_imp=pd.Series(np.abs(rf_sv).mean(0),index=feat_cols,name='RF_SHAP')
    xgb_imp=pd.Series(np.abs(xgb_sv).mean(0),index=feat_cols,name='XGB_SHAP')
    shap_g=pd.DataFrame({'RF_SHAP':rf_imp,'XGB_SHAP':xgb_imp})
    shap_g['Mean_SHAP']=shap_g.mean(axis=1)
    shap_g=shap_g.sort_values('Mean_SHAP',ascending=False)
    shap_g.to_csv('results/shap_global.csv')
    ok("SHAP global importance computed")
else:
    shap_g = pd.read_csv('results/shap_global.csv', index_col=0)


# ════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY OUTPUT
# ════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  FINAL SUMMARY")
print("="*60)

# ── All output files ──────────────────────────────────────────────────
sep("Output files")
files = sorted(
    glob.glob('data/**/*.csv', recursive=True) +
    glob.glob('results/*.csv') +
    glob.glob('figures/*.png')
)
total_kb = sum(os.path.getsize(f)/1024 for f in files)
print(f"\n  {'File':<52} {'KB':>7}")
print(f"  {'-'*60}")
for f in files:
    print(f"  {f:<52} {os.path.getsize(f)/1024:>7.1f}")
print(f"\n  Total: {len(files)} files | {total_kb/1024:.2f} MB")

# ── Best model performance ────────────────────────────────────────────
sep("Model performance")
try:
    perf = pd.read_csv('results/final_model_comparison.csv')
    b    = perf.iloc[0]
    print(f"\n  {'Model':<28} {'AUROC':>7} {'Ur':>7} {'Brier':>7} {'HR@10%':>8}")
    print(f"  {'-'*58}")
    for _, r in perf.iterrows():
        marker = ' <-- BEST' if r['model'] == b['model'] else ''
        print(f"  {r['model']:<28} {r['auroc']:>7.3f} "
              f"{r['ur']:>7.3f} {r['brier']:>7.3f} "
              f"{r['hr10']:>8.3f}{marker}")
    print(f"\n  Best model:  {b['model']}")
    print(f"  AUROC:       {b['auroc']:.4f}")
    print(f"  Ur (mu=0.5): {b['ur']:.4f}")
    print(f"  Brier Score: {b['brier']:.4f}")
    print(f"  HR@10%FPR:   {b['hr10']:.4f}")
except FileNotFoundError:
    warn("final_model_comparison.csv not found")

# ── SHAP top predictors ───────────────────────────────────────────────
sep("Top 10 bubble predictors (SHAP)")
try:
    print(f"\n  {'Rank':<5} {'Feature':<38} {'RF':>8} {'XGB':>8} {'Mean':>8}")
    print(f"  {'-'*72}")
    for rank, (feat, row) in enumerate(shap_g.head(10).iterrows(), 1):
        print(f"  {rank:<5} {feat:<38} "
              f"{row['RF_SHAP']:>8.5f} "
              f"{row['XGB_SHAP']:>8.5f} "
              f"{row['Mean_SHAP']:>8.5f}")
except Exception as ex:
    warn(f"SHAP results: {ex}")

# ── Robustness summary ────────────────────────────────────────────────
sep("Robustness analyses")
rob_path = 'results/robustness_analyses.csv'
if os.path.exists(rob_path):
    rob = pd.read_csv(rob_path)
    # Handle both old format (wide, NaN-heavy) and new format (section/metric/value)
    if 'section' in rob.columns and 'metric' in rob.columns and 'value' in rob.columns:
        # New clean format from updated Step 9
        for section in rob['section'].unique():
            sec = rob[rob['section'] == section]
            print("  " + str(section) + ":")
            print("  " + "-"*45)
            for _, r in sec.iterrows():
                try:
                    val = float(r['value'])
                    print(f"    {str(r['metric']):<35} {val:>+8.4f}")
                except (ValueError, TypeError):
                    print(f"    {str(r['metric']):<35} {'N/A':>8}")
            print()
    else:
        # Old wide format — display each row's non-NaN values
        warn("Old robustness format detected — re-run Step 9 for cleaner output")
        print()
        for _, r in rob.iterrows():
            non_nan = {k: v for k, v in r.items()
                       if k != 'check' and pd.notna(v)}
            vals = "  ".join(f"{k}={v:.4f}" if isinstance(v, float)
                             else f"{k}={v}" for k, v in non_nan.items())
            print(f"  {str(r.get('check','')):<30}  {vals}")
else:
    warn("Run Step 9 for robustness analyses")

# ── GSADF episode summary ─────────────────────────────────────────────
sep("Detected bubble episodes")
if os.path.exists('data/cleaned/bubble_dates.csv'):
    eps = pd.read_csv('data/cleaned/bubble_dates.csv',
                       parse_dates=['inception','termination'])
    print(f"\n  {len(eps)} episodes across "
          f"{eps['asset'].nunique()} asset class(es)\n")
    print(f"  {'Asset':<12} {'Country':<10} {'Inception':<12} "
          f"{'Termination':<13} {'Months':>6}")
    print(f"  {'-'*58}")
    for _, ep in eps.iterrows():
        dur = round((ep['termination']-ep['inception']).days/30.4, 1)
        print(f"  {ep['asset']:<12} {ep['country']:<10} "
              f"{str(ep['inception'].date()):<12} "
              f"{str(ep['termination'].date()):<13} "
              f"{dur:>6.1f}")
else:
    warn("Run Step 3 (GSADF) to generate bubble_dates.csv")

# ── Summary plot ──────────────────────────────────────────────────────
try:
    perf_path = 'results/final_model_comparison.csv'
    if os.path.exists(perf_path):
        perf = pd.read_csv(perf_path).dropna(subset=['auroc'])
        if len(perf) > 0:
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            # AUROC bar chart
            ax = axes[0]
            colors = ['#e74c3c' if 'Ensemble' in m else '#2980b9'
                      for m in perf['model']]
            bars = ax.bar(perf['model'], perf['auroc'],
                           color=colors, alpha=0.85, edgecolor='white')
            ax.axhline(0.5, color='black', ls='--', lw=0.8, label='Random')
            ax.set_ylabel('AUROC')
            ax.set_title('AUROC by Model\nRed = Ensemble | Blue = Individual',
                          fontweight='bold')
            plt.sca(ax); plt.xticks(rotation=35, ha='right', fontsize=8)
            ax.grid(True, alpha=0.3, axis='y')
            ax.legend(fontsize=8)
            for bar, val in zip(bars, perf['auroc']):
                ax.text(bar.get_x()+bar.get_width()/2,
                        bar.get_height()+0.005,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=7)

            # SHAP top 10
            ax2 = axes[1]
            top10 = shap_g.head(10)['Mean_SHAP'].sort_values()
            cmap  = plt.cm.RdBu_r
            ax2.barh(range(len(top10)), top10.values,
                      color=cmap(np.linspace(0.2, 0.8, len(top10))),
                      alpha=0.85)
            ax2.set_yticks(range(len(top10)))
            ax2.set_yticklabels([f[:35] for f in top10.index], fontsize=8)
            ax2.set_xlabel('Mean |SHAP value|')
            ax2.set_title('Top 10 Bubble Predictors\nMean |SHAP| (RF + XGB)',
                           fontweight='bold')
            ax2.grid(True, alpha=0.3, axis='x')

            plt.suptitle('Chapter 3 — Final Results Summary',
                          fontsize=12, fontweight='bold')
            plt.tight_layout()
            plt.savefig('figures/final_summary.png',
                         dpi=150, bbox_inches='tight')
            plt.close()
            ok("figures/final_summary.png saved")
except Exception as ex:
    warn(f"Summary plot: {ex}")

# ── Gap coverage checklist ────────────────────────────────────────────
print(f"""
  {'='*60}
  GAP COVERAGE CHECKLIST
  {'='*60}
  Gap 1  OK  GSADF detection integrated with ML pipeline
  Gap 2  OK  Y = GSADF inception dates (not LV crisis dates)
  Gap 3  OK  Equities(11) + Housing + Bonds + Credit + VIX
  Gap 4  OK  HY/IG spreads, NFCI, VIX as GSADF covariates
  Gap 5  OK  3m/6m/12m lags + LSTM-MLP + Transformer-MLP
  Gap 6  OK  SHAP global + temporal + 6 episode decomp.

  DATA SOURCES (no GitHub, no Kaggle):
    FRED API       — rates, credit, housing, macro, VIX
    Yahoo Finance  — equity indices, Treasury yields

  BEFORE FINAL SUBMISSION:
    Step 3: Change N_BOOT = 99  →  N_BOOT = 499 (or 1999)
    Step 6: Confirm PyTorch installed for LSTM/Transformer

  OUTPUT LOCATION:
    data/cleaned/   — processed datasets
    results/        — performance tables, SHAP CSVs
    figures/        — all plots
  {'='*60}
""")

print("Step 10 complete.")


  STEP 10 — FINAL SUMMARY (self-contained)

  Checking existing outputs:
    FOUND    Feature panel                  data/cleaned/ml_panel.csv
    FOUND    GSADF bubble dates             data/cleaned/bubble_dates.csv
    FOUND    ML predictions                 data/cleaned/predictions_ml.csv
    FOUND    Ensemble predictions           data/cleaned/predictions_final.csv
    FOUND    Model performance              results/model_performance.csv
    FOUND    Final comparison               results/final_model_comparison.csv
    FOUND    SHAP global                    results/shap_global.csv
    FOUND    SHAP temporal                  results/shap_temporal.csv
    FOUND    SHAP episodes                  results/shap_episodes.csv
    FOUND    Robustness                     results/robustness_analyses.csv
  OK  ml_panel.csv: 311 obs x 84 features

  FINAL SUMMARY

--- Output files ---

  File                                                      KB
  --------------------------------------------